<a href="https://colab.research.google.com/github/uniqueeee777/stability-aware-multiview-food-security/blob/main/06_Simulation_and_Statistical_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kunci protokol simulasi

In [ ]:

# ============================================================
# CELL 1 — MONTE CARLO SIMULATION PROTOCOL
#
# Research:
# Stability-aware multi-view clustering
#
# Phase A: Validate synthetic data generation.
# No clustering or model selection in this cell.
# ============================================================

from google.colab import drive
from pathlib import Path
from importlib.metadata import version
from IPython.display import display
from scipy.spatial.distance import pdist, squareform

import numpy as np
import pandas as pd
import scipy
import sklearn
import hashlib
import json

drive.mount("/content/drive", force_remount=False)

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

SIM_DIR = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v1"
)

SIM_DIR.mkdir(parents=True, exist_ok=True)

N = 85
BASE_SEED = 20261019

# Generator validation only.
# Not enough repetitions for final inference.
N_PILOT = 10

VIEWS_SIM = {
    "Food": [
        "F1_ikp_ketersediaan",
        "F2_ikp_keterjangkauan",
        "F3_ikp_pemanfaatan"
    ],
    "Climate": [
        "C1_anomali_hujan_mm",
        "C2_cdd_hari",
        "C3_hari_hujan_ekstrem"
    ],
    "Accessibility": [
        "A1_persen_tanpa_angkutan",
        "A2_persen_jalan_tanah_lainnya",
        "A3_persen_tanpa_pertokoan_pasar"
    ]
}

FEATURES_SIM = sum(VIEWS_SIM.values(), [])

# Scenario parameters are explicit and fixed here.
# Signal values are simulation-design choices,
# NOT estimates from the empirical dataset.

SCENARIOS = {
    "aligned_K2": {
        "K_true": 2,
        "signal": 1.20,
        "type": "aligned"
    },
    "aligned_K3": {
        "K_true": 3,
        "signal": 1.20,
        "type": "aligned"
    },
    "aligned_K4": {
        "K_true": 4,
        "signal": 1.20,
        "type": "aligned"
    },
    "weak_K3": {
        "K_true": 3,
        "signal": 0.45,
        "type": "weak"
    },
    "imbalanced_K3": {
        "K_true": 3,
        "signal": 1.20,
        "type": "imbalanced"
    },
    "conflict_K3": {
        "K_true": 3,
        "signal": 1.20,
        "type": "conflict"
    },
    "null_gaussian": {
        "K_true": None,
        "signal": 0.0,
        "type": "null"
    }
}

# Common Gaussian noise factors:
# global factor, view-specific factor, idiosyncratic noise
NOISE_PARAMETERS = {
    "global_sd": 0.35,
    "view_sd": 0.50,
    "idio_sd": 0.90
}

PROTOCOL = {
    "name": "Multiview Monte Carlo generator pilot",
    "version": "1.0",
    "n": N,
    "n_features": len(FEATURES_SIM),
    "n_views": len(VIEWS_SIM),
    "n_pilot_replicates": N_PILOT,
    "base_seed": BASE_SEED,
    "scenarios": SCENARIOS,
    "noise_parameters": NOISE_PARAMETERS,
    "ground_truth_policy": (
        "Stored separately; never used in "
        "model fitting or hyperparameter selection"
    ),
    "null_policy": (
        "Correlated unimodal Gaussian, "
        "no discrete ground truth"
    ),
    "simulation_status": "GENERATOR_PILOT"
}

protocol_file = SIM_DIR / "generator_protocol.json"

if protocol_file.exists():
    existing = json.loads(
        protocol_file.read_text(encoding="utf-8")
    )
    assert existing == PROTOCOL, (
        "Existing protocol differs. Use a new directory."
    )
else:
    protocol_file.write_text(
        json.dumps(PROTOCOL, indent=2),
        encoding="utf-8"
    )

print("SIMULATION GENERATOR PROTOCOL LOCKED")
print("Scenarios:", len(SCENARIOS))
print("Pilot datasets:", len(SCENARIOS) * N_PILOT)
print("Observations per dataset:", N)
print("Features:", len(FEATURES_SIM))
print("Output:", SIM_DIR)


Mounted at /content/drive
SIMULATION GENERATOR PROTOCOL LOCKED
Scenarios: 7
Pilot datasets: 70
Observations per dataset: 85
Features: 9
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/generator_pilot_v1


# Generator multi-view dengan ground truth

In [ ]:

# ============================================================
# CELL 2 — SYNTHETIC MULTI-VIEW DATA GENERATOR
#
# Explicit latent groups, correlated Gaussian noise,
# imbalance, conflicting views and a unimodal null.
# ============================================================

def make_class_labels(n, K, rng, proportions=None):
    """Generate exactly n labels with controlled class sizes."""

    if proportions is None:
        proportions = np.ones(K) / K

    proportions = np.asarray(proportions, dtype=float)

    assert len(proportions) == K
    assert np.all(proportions > 0)
    assert np.isclose(proportions.sum(), 1.0)

    expected = n * proportions
    counts = np.floor(expected).astype(int)

    leftover = n - counts.sum()
    priority = np.argsort(
        -(expected - counts),
        kind="stable"
    )

    counts[priority[:leftover]] += 1

    labels = np.repeat(
        np.arange(K, dtype=int),
        counts
    )

    rng.shuffle(labels)

    assert len(labels) == n
    assert len(np.unique(labels)) == K

    return labels


def generate_dataset(scenario, rng):

    cfg = SCENARIOS[scenario]
    scenario_type = cfg["type"]
    K = cfg["K_true"]
    signal = cfg["signal"]

    n = N
    p = len(FEATURES_SIM)

    # --------------------------------------------------------
    # 1. TRUE CLUSTER STRUCTURE
    # --------------------------------------------------------

    if K is None:
        truth = np.full(n, -1, dtype=int)

    elif scenario_type == "imbalanced":
        truth = make_class_labels(
            n, K, rng,
            proportions=[0.60, 0.28, 0.12]
        )

    else:
        truth = make_class_labels(n, K, rng)

    nuisance = np.full(n, -1, dtype=int)

    # --------------------------------------------------------
    # 2. CORRELATED MULTIVARIATE GAUSSIAN NOISE
    # --------------------------------------------------------

    global_factor = rng.standard_normal((n, 1))
    view_factors = rng.standard_normal((n, 3))
    residual = rng.standard_normal((n, p))

    # Signs reflect plausible positive/negative
    # correlations within and across views.
    loadings = np.array([
        1.0,  1.0,  1.0,    # Food
        1.0, -1.0,  1.0,    # Climate
       -1.0, -1.0, -1.0    # Accessibility
    ])

    global_component = (
        NOISE_PARAMETERS["global_sd"]
        * global_factor * loadings[None, :]
    )

    view_component = np.zeros((n, p))

    for v in range(3):
        start = 3 * v
        stop = start + 3

        view_component[:, start:stop] = (
            NOISE_PARAMETERS["view_sd"]
            * view_factors[:, [v]]
            * loadings[None, start:stop]
        )

    idiosyncratic = (
        NOISE_PARAMETERS["idio_sd"] * residual
    )

    X = (
        global_component
        + view_component
        + idiosyncratic
    )

    # --------------------------------------------------------
    # 3. STRUCTURED CLASS EFFECTS
    # --------------------------------------------------------

    if K is not None:

        levels = np.linspace(-1.0, 1.0, K)

        # Non-identical relationships between views.
        food_levels = levels
        climate_levels = np.roll(levels, 1)
        access_levels = -levels

        # Within-view feature loadings.
        shapes = {
            "Food": np.array([0.70, 1.00, 1.00]),
            "Climate": np.array([1.00, -0.80, 1.00]),
            "Accessibility": np.array([1.00, 1.00, 0.65])
        }

        # Conflict: climate reflects independent
        # nuisance groups instead of the target groups.
        if scenario_type == "conflict":
            nuisance = make_class_labels(
                n, K, rng
            )
            climate_labels = nuisance
            climate_strength = 1.60
        else:
            climate_labels = truth
            climate_strength = signal

        effects = {
            "Food": (
                signal
                * food_levels[truth, None]
                * shapes["Food"][None, :]
            ),
            "Climate": (
                climate_strength
                * climate_levels[climate_labels, None]
                * shapes["Climate"][None, :]
            ),
            "Accessibility": (
                signal
                * access_levels[truth, None]
                * shapes["Accessibility"][None, :]
            )
        }

        X += np.column_stack([
            effects["Food"],
            effects["Climate"],
            effects["Accessibility"]
        ])

    # --------------------------------------------------------
    # 4. QUALITY CONTROL
    # --------------------------------------------------------

    assert X.shape == (n, p)
    assert np.isfinite(X).all()

    if K is None:
        assert np.all(truth == -1)
    else:
        assert len(np.unique(truth)) == K

    return X, truth, nuisance


# ------------------------------------------------------------
# 5. GENERATE INDEPENDENT MONTE CARLO DATASETS
# ------------------------------------------------------------

DATA_DIR = SIM_DIR / "datasets"
TRUTH_DIR = SIM_DIR / "ground_truth"

DATA_DIR.mkdir(exist_ok=True)
TRUTH_DIR.mkdir(exist_ok=True)

scenario_names = list(SCENARIOS)

master_seed = np.random.SeedSequence(BASE_SEED)

children = master_seed.spawn(
    len(scenario_names) * N_PILOT
)

manifest_rows = []
counter = 0

for scenario in scenario_names:

    for rep in range(1, N_PILOT + 1):

        child = children[counter]
        counter += 1

        rng = np.random.default_rng(child)

        X, truth, nuisance = generate_dataset(
            scenario, rng
        )

        dataset_id = f"{scenario}_r{rep:03d}"

        data_file = DATA_DIR / f"{dataset_id}.csv"
        truth_file = TRUTH_DIR / f"{dataset_id}.csv"

        # Features and truth are stored separately.
        data_df = pd.DataFrame(
            X, columns=FEATURES_SIM
        )

        data_df.insert(
            0,
            "observation_id",
            np.arange(N, dtype=int)
        )

        truth_df = pd.DataFrame({
            "observation_id": np.arange(N),
            "true_cluster": truth,
            "nuisance_climate_group": nuisance
        })

        if data_file.exists():
            previous = pd.read_csv(data_file)
            assert np.allclose(
                previous[FEATURES_SIM].to_numpy(),
                X,
                atol=1e-12
            )
        else:
            data_df.to_csv(
                data_file, index=False,
                float_format="%.17g"
            )

        if truth_file.exists():
            previous_truth = pd.read_csv(truth_file)
            assert np.array_equal(
                previous_truth["true_cluster"],
                truth
            )
            assert np.array_equal(
                previous_truth["nuisance_climate_group"],
                nuisance
            )
        else:
            truth_df.to_csv(
                truth_file,
                index=False
            )

        manifest_rows.append({
            "dataset_id": dataset_id,
            "scenario": scenario,
            "replicate": rep,
            "n": N,
            "K_true": (
                SCENARIOS[scenario]["K_true"]
                if SCENARIOS[scenario]["K_true"] is not None
                else -1
            ),
            "n_truth_clusters": (
                len(np.unique(truth))
                if SCENARIOS[scenario]["K_true"] is not None
                else 0
            ),
            "child_seed_state": json.dumps(
                child.generate_state(4).tolist()
            ),
            "data_sha256": hashlib.sha256(
                data_file.read_bytes()
            ).hexdigest(),
            "truth_sha256": hashlib.sha256(
                truth_file.read_bytes()
            ).hexdigest()
        })


simulation_manifest = pd.DataFrame(manifest_rows)

assert len(simulation_manifest) == (
    len(SCENARIOS) * N_PILOT
)

simulation_manifest.to_csv(
    SIM_DIR / "simulation_dataset_manifest.csv",
    index=False
)

print("SYNTHETIC DATA GENERATION COMPLETED")
print("Datasets:", len(simulation_manifest))
print("Features per dataset:", len(FEATURES_SIM))
print("Ground truth files:", len(list(TRUTH_DIR.glob("*.csv"))))

display(
    simulation_manifest.groupby(
        ["scenario", "K_true"]
    ).size().reset_index(name="n_datasets")
)


SYNTHETIC DATA GENERATION COMPLETED
Datasets: 70
Features per dataset: 9
Ground truth files: 70


,scenario,K_true,n_datasets
0,aligned_K2,2,10
1,aligned_K3,3,10
2,aligned_K4,4,10
3,conflict_K3,3,10
4,imbalanced_K3,3,10
5,null_gaussian,-1,10
6,weak_K3,3,10


# Audit struktur dan geometri simulasi

In [ ]:

# ============================================================
# CELL 3 — SIMULATION GENERATOR VALIDATION
# No clustering algorithms are fitted.
# ============================================================

def representation_R0(X):
    """
    Same geometry as empirical R0:
    - sample z-score
    - within-view Euclidean distance / sqrt(p_v)
    - view RMS normalized to sqrt(2)
    - equal view weighting
    """

    X = np.asarray(X, dtype=float)

    assert X.ndim == 2
    assert X.shape[1] == 9
    assert np.isfinite(X).all()

    mean = X.mean(axis=0)
    sd = X.std(axis=0, ddof=1)

    if np.any(sd <= 0):
        raise ValueError("Constant simulated feature.")

    Z = (X - mean) / sd

    blocks = []
    view_distances = {}

    for i, view in enumerate(VIEWS_SIM):

        A = Z[:, 3*i:3*i+3]

        D = squareform(pdist(A)) / np.sqrt(3)

        upper = D[np.triu_indices(len(D), 1)]
        rms = np.sqrt(np.mean(upper ** 2))

        if not np.isfinite(rms) or rms <= 0:
            raise ValueError("Invalid view RMS.")

        factor = np.sqrt(2.0) / rms

        view_distances[view] = D * factor

        blocks.append(
            A * factor / 3.0
        )

    Phi = np.column_stack(blocks)
    fused = squareform(pdist(Phi))

    reconstruction = np.sqrt(
        sum(
            D ** 2 / 3.0
            for D in view_distances.values()
        )
    )

    assert np.allclose(
        fused, reconstruction, atol=1e-10
    )

    assert np.allclose(
        fused, fused.T, atol=1e-12
    )

    assert np.allclose(
        np.diag(fused), 0, atol=1e-12
    )

    return Phi, fused, view_distances


audit_rows = []

for row in simulation_manifest.itertuples(index=False):

    data = pd.read_csv(
        DATA_DIR / f"{row.dataset_id}.csv"
    )

    truth = pd.read_csv(
        TRUTH_DIR / f"{row.dataset_id}.csv"
    )

    X = data[FEATURES_SIM].to_numpy(
        dtype=float
    )

    assert X.shape == (85, 9)

    assert np.array_equal(
        data["observation_id"],
        truth["observation_id"]
    )

    assert not data.isna().any().any()
    assert np.isfinite(X).all()

    if row.K_true == -1:
        assert truth["true_cluster"].eq(-1).all()
        cluster_counts = []
    else:
        cluster_counts = (
            truth["true_cluster"]
            .value_counts()
            .sort_index()
            .tolist()
        )
        assert len(cluster_counts) == row.K_true

    # Positive-definite sample covariance QC
    covariance = np.cov(X, rowvar=False)

    min_eigenvalue = np.linalg.eigvalsh(
        covariance
    ).min()

    assert min_eigenvalue > 1e-10

    Phi, D, view_D = representation_R0(X)

    audit_rows.append({
        "dataset_id": row.dataset_id,
        "scenario": row.scenario,
        "K_true": row.K_true,
        "n": len(X),
        "p": X.shape[1],
        "minimum_cov_eigenvalue": float(
            min_eigenvalue
        ),
        "n_classes": len(cluster_counts),
        "smallest_true_class": (
            min(cluster_counts)
            if cluster_counts else np.nan
        ),
        "largest_true_class": (
            max(cluster_counts)
            if cluster_counts else np.nan
        ),
        "mean_fused_distance": float(
            D[np.triu_indices(len(D), 1)].mean()
        ),
        "geometry_pass": True
    })


generator_audit = pd.DataFrame(audit_rows)

assert len(generator_audit) == len(simulation_manifest)
assert generator_audit["geometry_pass"].all()

summary = (
    generator_audit.groupby(
        ["scenario", "K_true"],
        as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "count"),
        min_class_size=(
            "smallest_true_class", "min"
        ),
        max_class_size=(
            "largest_true_class", "max"
        ),
        minimum_cov_eigenvalue=(
            "minimum_cov_eigenvalue", "min"
        ),
        mean_fused_distance=(
            "mean_fused_distance", "mean"
        )
    )
)

print("A. GENERATOR STRUCTURE AND GEOMETRY")
display(summary.round(4))

print("\nB. QUALITY CONTROL")
print("All datasets generated:", len(generator_audit))
print("Missing values: 0")
print("Invalid covariance matrices: 0")
print("R0 geometry checks passed:",
      int(generator_audit["geometry_pass"].sum()))

generator_audit.to_csv(
    SIM_DIR / "generator_validation_all.csv",
    index=False
)

summary.to_csv(
    SIM_DIR / "generator_validation_summary.csv",
    index=False
)

print("\nGENERATOR PILOT VALIDATION COMPLETED")
print("Saved:", SIM_DIR)


A. GENERATOR STRUCTURE AND GEOMETRY


,scenario,K_true,n_datasets,min_class_size,max_class_size,minimum_cov_eigenvalue,mean_fused_distance
0,aligned_K2,2,10,42.0,43.0,0.3973,1.3348
1,aligned_K3,3,10,28.0,29.0,0.4167,1.3541
2,aligned_K4,4,10,21.0,22.0,0.3972,1.3551
3,conflict_K3,3,10,28.0,29.0,0.3528,1.3512
4,imbalanced_K3,3,10,10.0,51.0,0.4003,1.3540
5,null_gaussian,-1,10,NaN,NaN,0.4179,1.3706
6,weak_K3,3,10,28.0,29.0,0.3777,1.3681



B. QUALITY CONTROL
All datasets generated: 70
Missing values: 0
Invalid covariance matrices: 0
R0 geometry checks passed: 70

GENERATOR PILOT VALIDATION COMPLETED
Saved: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/generator_pilot_v1


# Audit keterpisahan kelas dan konflik antar-view

In [ ]:

# ============================================================
# CELL 4 — GENERATOR SIGNAL & VIEW-CONFLICT AUDIT
#
# Validate whether synthetic class structure is recoverable
# in the observed R0 geometry.
#
# No clustering models fitted.
# No hyperparameter selection.
# ============================================================

from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score
)
from IPython.display import display

import numpy as np
import pandas as pd
import hashlib

assert len(simulation_manifest) == 70
assert len(generator_audit) == 70


def distance_separation_ratio(D, labels):
    """
    Mean between-class distance divided by
    mean within-class distance.

    Descriptive measure, not a significance test.
    """

    D = np.asarray(D, dtype=float)
    labels = np.asarray(labels)

    ii, jj = np.triu_indices(len(labels), k=1)

    values = D[ii, jj]
    same = labels[ii] == labels[jj]

    if not same.any() or not (~same).any():
        return np.nan

    within = values[same].mean()
    between = values[~same].mean()

    return float(between / within)


signal_records = []

for record in simulation_manifest.itertuples(index=False):

    data_file = DATA_DIR / f"{record.dataset_id}.csv"
    truth_file = TRUTH_DIR / f"{record.dataset_id}.csv"

    # Verify integrity of the generated files.
    assert hashlib.sha256(
        data_file.read_bytes()
    ).hexdigest() == record.data_sha256

    assert hashlib.sha256(
        truth_file.read_bytes()
    ).hexdigest() == record.truth_sha256

    data = pd.read_csv(data_file)
    targets = pd.read_csv(truth_file)

    assert np.array_equal(
        data["observation_id"].to_numpy(),
        targets["observation_id"].to_numpy()
    )

    X = data[FEATURES_SIM].to_numpy(dtype=float)

    truth = targets[
        "true_cluster"
    ].to_numpy(dtype=int)

    nuisance = targets[
        "nuisance_climate_group"
    ].to_numpy(dtype=int)

    Phi, D_fused, D_views = representation_R0(X)

    row = {
        "dataset_id": record.dataset_id,
        "scenario": record.scenario,
        "K_true": record.K_true,
        "replicate": record.replicate,
        "sil_truth_fused": np.nan,
        "sil_truth_food": np.nan,
        "sil_truth_climate": np.nan,
        "sil_truth_access": np.nan,
        "negative_silhouette_pct": np.nan,
        "between_within_fused": np.nan,
        "sil_climate_nuisance": np.nan,
        "ARI_truth_nuisance": np.nan,
        "climate_conflict_gap": np.nan
    }

    # Null: no ground-truth partition exists.
    if record.K_true == -1:
        assert np.all(truth == -1)
        signal_records.append(row)
        continue

    assert len(np.unique(truth)) == record.K_true

    # --------------------------------------------------------
    # A. Ground-truth separation, overall and by view
    # --------------------------------------------------------

    row["sil_truth_fused"] = silhouette_score(
        D_fused,
        truth,
        metric="precomputed"
    )

    row["sil_truth_food"] = silhouette_score(
        D_views["Food"],
        truth,
        metric="precomputed"
    )

    row["sil_truth_climate"] = silhouette_score(
        D_views["Climate"],
        truth,
        metric="precomputed"
    )

    row["sil_truth_access"] = silhouette_score(
        D_views["Accessibility"],
        truth,
        metric="precomputed"
    )

    individual_sil = silhouette_samples(
        D_fused,
        truth,
        metric="precomputed"
    )

    row["negative_silhouette_pct"] = (
        100.0 * np.mean(individual_sil < 0)
    )

    row["between_within_fused"] = (
        distance_separation_ratio(
            D_fused, truth
        )
    )

    # --------------------------------------------------------
    # B. Conflict: Climate follows a nuisance partition
    # --------------------------------------------------------

    if record.scenario == "conflict_K3":

        assert len(np.unique(nuisance)) == 3

        row["ARI_truth_nuisance"] = (
            adjusted_rand_score(
                truth, nuisance
            )
        )

        row["sil_climate_nuisance"] = (
            silhouette_score(
                D_views["Climate"],
                nuisance,
                metric="precomputed"
            )
        )

        row["climate_conflict_gap"] = (
            row["sil_climate_nuisance"]
            - row["sil_truth_climate"]
        )

    signal_records.append(row)


signal_audit = pd.DataFrame(signal_records)

# ------------------------------------------------------------
# C. Scenario-level summary
# ------------------------------------------------------------

scenario_signal = (
    signal_audit.groupby(
        "scenario", as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "count"),
        mean_sil_fused=("sil_truth_fused", "mean"),
        sd_sil_fused=("sil_truth_fused", "std"),
        min_sil_fused=("sil_truth_fused", "min"),
        mean_sil_food=("sil_truth_food", "mean"),
        mean_sil_climate=("sil_truth_climate", "mean"),
        mean_sil_access=("sil_truth_access", "mean"),
        mean_negative_pct=(
            "negative_silhouette_pct", "mean"
        ),
        mean_between_within=(
            "between_within_fused", "mean"
        ),
        mean_conflict_gap=(
            "climate_conflict_gap", "mean"
        )
    )
)

print("A. GROUND-TRUTH SEPARATION BY SCENARIO")
display(scenario_signal.round(4))

# ------------------------------------------------------------
# D. Conflict-specific diagnostics
# ------------------------------------------------------------

conflict = signal_audit.loc[
    signal_audit["scenario"] == "conflict_K3"
]

assert len(conflict) == N_PILOT

print("\nB. CONFLICTING VIEW DIAGNOSTICS")

display(
    conflict[
        [
            "dataset_id",
            "ARI_truth_nuisance",
            "sil_truth_climate",
            "sil_climate_nuisance",
            "climate_conflict_gap"
        ]
    ].round(4)
)

print(
    "\nConflict replicates with positive gap:",
    int((conflict["climate_conflict_gap"] > 0).sum()),
    "/", len(conflict)
)

# ------------------------------------------------------------
# E. Save the diagnostics
# ------------------------------------------------------------

signal_audit.to_csv(
    SIM_DIR / "generator_signal_audit_all.csv",
    index=False
)

scenario_signal.to_csv(
    SIM_DIR / "generator_signal_audit_summary.csv",
    index=False
)

print("\nGENERATOR SIGNAL AUDIT COMPLETED")
print("Output:", SIM_DIR)


A. GROUND-TRUTH SEPARATION BY SCENARIO


,scenario,n_datasets,mean_sil_fused,sd_sil_fused,min_sil_fused,mean_sil_food,mean_sil_climate,mean_sil_access,mean_negative_pct,mean_between_within,mean_conflict_gap
0,aligned_K2,10,0.4050,0.0223,0.3694,0.3816,0.3995,0.3591,0.4706,1.7099,NaN
1,aligned_K3,10,0.1876,0.0219,0.1555,0.0930,0.0916,0.0769,13.1765,1.3893,NaN
2,aligned_K4,10,0.0883,0.0257,0.0475,-0.0181,-0.0023,-0.0139,30.0000,1.2968,NaN
3,conflict_K3,10,0.0738,0.0172,0.0500,0.0898,-0.0430,0.0831,26.3529,1.2290,0.1981
4,imbalanced_K3,10,0.1966,0.0163,0.1678,0.0927,0.1279,0.0774,13.6471,1.4000,NaN
5,null_gaussian,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,weak_K3,10,0.0133,0.0133,-0.0140,-0.0225,-0.0240,-0.0159,43.0588,1.0678,NaN



B. CONFLICTING VIEW DIAGNOSTICS


,dataset_id,ARI_truth_nuisance,sil_truth_climate,sil_climate_nuisance,climate_conflict_gap
50,conflict_K3_r001,-0.0118,-0.0471,0.1640,0.2110
51,conflict_K3_r002,-0.0156,-0.0492,0.1478,0.1970
52,conflict_K3_r003,0.0023,-0.0475,0.1697,0.2172
53,conflict_K3_r004,0.0163,-0.0372,0.1363,0.1735
54,conflict_K3_r005,0.0061,-0.0332,0.0833,0.1165
55,conflict_K3_r006,-0.0054,-0.0415,0.1796,0.2211
56,conflict_K3_r007,-0.0079,-0.0397,0.1612,0.2009
57,conflict_K3_r008,-0.0130,-0.0455,0.1680,0.2135
58,conflict_K3_r009,0.0201,-0.0519,0.1932,0.2452
59,conflict_K3_r010,0.0444,-0.0369,0.1480,0.1849



Conflict replicates with positive gap: 10 / 10

GENERATOR SIGNAL AUDIT COMPLETED
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/generator_pilot_v1


# Definisi generator V2

In [ ]:

# ============================================================
# CELL 5 — REGULAR-SIMPLEX MULTI-VIEW GENERATOR V2
#
# Mathematical design:
# - K = 2, 3, 4
# - Equal pairwise Euclidean centroid distances within views
# - Correlated Gaussian noise
# - Independent nuisance partition for conflicting climate view
#
# No clustering model is fitted here.
# ============================================================

from scipy.linalg import helmert
from scipy.spatial.transform import Rotation
from scipy.spatial.distance import pdist
from pathlib import Path
import numpy as np
import pandas as pd
import json
import hashlib

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

SIM_V2 = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v2"
)

SIM_V2.mkdir(parents=True, exist_ok=True)

N_V2 = 85
N_REP_V2 = 10
SEED_WORDS_V2 = [20261019, 2]

FEATURES_V2 = [
    "F1_ikp_ketersediaan",
    "F2_ikp_keterjangkauan",
    "F3_ikp_pemanfaatan",
    "C1_anomali_hujan_mm",
    "C2_cdd_hari",
    "C3_hari_hujan_ekstrem",
    "A1_persen_tanpa_angkutan",
    "A2_persen_jalan_tanah_lainnya",
    "A3_persen_tanpa_pertokoan_pasar"
]

VIEW_ORDER_V2 = [
    "Food", "Climate", "Accessibility"
]

SCENARIOS_V2 = {
    "aligned_K2": {
        "K": 2, "signal": 1.50, "type": "aligned"
    },
    "aligned_K3": {
        "K": 3, "signal": 1.50, "type": "aligned"
    },
    "aligned_K4": {
        "K": 4, "signal": 1.50, "type": "aligned"
    },
    "weak_K3": {
        "K": 3, "signal": 0.55, "type": "weak"
    },
    "imbalanced_K3": {
        "K": 3, "signal": 1.50, "type": "imbalanced"
    },
    "conflict_K3": {
        "K": 3, "signal": 1.50, "type": "conflict"
    },
    "null_gaussian": {
        "K": None, "signal": 0.0, "type": "null"
    }
}

NOISE_V2 = {
    "global_sd": 0.35,
    "view_sd": 0.50,
    "idio_sd": 0.90
}

# Fixed orthogonal rotations change indicator directions
# without changing pairwise centroid distances.
ANGLES_V2 = {
    "Food": [18, 28, 37],
    "Climate": [31, -19, 44],
    "Accessibility": [-24, 39, -33]
}

ROTATIONS_V2 = {
    view: Rotation.from_euler(
        "xyz", angles, degrees=True
    ).as_matrix()
    for view, angles in ANGLES_V2.items()
}

for R in ROTATIONS_V2.values():
    assert np.allclose(
        R @ R.T, np.eye(3), atol=1e-12
    )


def simplex_centroids(K):
    """
    K vertices in R^3, with:
        center of vertices = 0
        all pairwise distances = sqrt(2)

    K must be 2, 3, or 4.
    """

    if K not in (2, 3, 4):
        raise ValueError("Simplex requires K in {2,3,4}.")

    M = helmert(K, full=False).T

    M = np.pad(
        M,
        ((0, 0), (0, 3 - (K - 1))),
        mode="constant"
    )

    assert M.shape == (K, 3)
    assert np.allclose(M.mean(axis=0), 0, atol=1e-12)
    assert np.allclose(
        pdist(M), np.sqrt(2), atol=1e-12
    )

    return M


def controlled_labels(n, K, rng, proportions=None):
    """
    Balanced or explicitly imbalanced class counts.
    Largest-remainder allocation, then random permutation.
    """

    if proportions is None:
        proportions = np.ones(K) / K

    proportions = np.asarray(
        proportions, dtype=float
    )

    assert len(proportions) == K
    assert np.all(proportions > 0)
    assert np.isclose(proportions.sum(), 1)

    expected = n * proportions
    counts = np.floor(expected).astype(int)

    remainder = n - counts.sum()

    priority = np.argsort(
        -(expected - counts),
        kind="stable"
    )

    counts[priority[:remainder]] += 1

    assert counts.sum() == n
    assert (counts > 0).all()

    y = np.repeat(
        np.arange(K, dtype=int), counts
    )

    rng.shuffle(y)
    return y


def generate_multiview_v2(scenario, rng):

    cfg = SCENARIOS_V2[scenario]

    K = cfg["K"]
    signal = cfg["signal"]
    scenario_type = cfg["type"]

    n = N_V2

    # --------------------------------------------------------
    # A. TRUE CLASS LABELS
    # --------------------------------------------------------

    if K is None:
        truth = np.full(n, -1, dtype=int)

    elif scenario_type == "imbalanced":
        truth = controlled_labels(
            n, K, rng,
            proportions=[0.60, 0.28, 0.12]
        )

    else:
        truth = controlled_labels(n, K, rng)

    if scenario_type == "conflict":
        nuisance = controlled_labels(n, K, rng)
    else:
        nuisance = np.full(n, -1, dtype=int)

    # --------------------------------------------------------
    # B. CORRELATED GAUSSIAN NOISE
    # --------------------------------------------------------

    loading = np.array([
         1,  1,  1,
         1, -1,  1,
        -1, -1, -1
    ], dtype=float)

    global_factor = rng.normal(size=(n, 1))
    view_factors = rng.normal(size=(n, 3))
    idio_noise = rng.normal(size=(n, 9))

    X = (
        NOISE_V2["global_sd"]
        * global_factor * loading[None, :]
        + NOISE_V2["idio_sd"] * idio_noise
    )

    for j in range(3):

        start = 3 * j
        stop = start + 3

        X[:, start:stop] += (
            NOISE_V2["view_sd"]
            * view_factors[:, [j]]
            * loading[None, start:stop]
        )

    # --------------------------------------------------------
    # C. CLASS-CENTROID EFFECTS
    # --------------------------------------------------------

    if K is not None:

        simplex = simplex_centroids(K)

        for j, view in enumerate(VIEW_ORDER_V2):

            start = 3 * j
            stop = start + 3

            labels_view = truth
            amplitude = signal

            # Climate is governed by a separate nuisance
            # partition in the conflict scenario.
            if (
                scenario_type == "conflict"
                and view == "Climate"
            ):
                labels_view = nuisance
                amplitude = 1.20 * signal

            centers = (
                amplitude
                * simplex @ ROTATIONS_V2[view]
            )

            # Analytical centroid-distance check
            assert np.allclose(
                pdist(centers),
                np.sqrt(2) * amplitude,
                atol=1e-12
            )

            X[:, start:stop] += centers[
                labels_view
            ]

    assert X.shape == (85, 9)
    assert np.isfinite(X).all()

    return X, truth, nuisance


PROTOCOL_V2 = {
    "generator": "regular_simplex_multiview_v2",
    "stage": "generator_pilot",
    "n": N_V2,
    "p": len(FEATURES_V2),
    "n_replicates": N_REP_V2,
    "seed_words": SEED_WORDS_V2,
    "features": FEATURES_V2,
    "scenarios": SCENARIOS_V2,
    "noise": NOISE_V2,
    "rotation_angles_xyz_degrees": ANGLES_V2,
    "centroid_rule": (
        "Helmert simplex; equal Euclidean "
        "pairwise centroid distances within each view"
    ),
    "ground_truth_usage": "evaluation_only",
    "status": "PILOT_NOT_FINAL_INFERENCE"
}

protocol_path_v2 = SIM_V2 / "generator_protocol.json"

if protocol_path_v2.exists():

    previous = json.loads(
        protocol_path_v2.read_text(encoding="utf-8")
    )

    assert previous == PROTOCOL_V2, (
        "Generator V2 protocol mismatch."
    )

else:
    protocol_path_v2.write_text(
        json.dumps(
            PROTOCOL_V2, indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

print("REGULAR SIMPLEX GEOMETRY VERIFIED")

for K in [2, 3, 4]:
    M = simplex_centroids(K)
    print(
        f"K={K}: shape={M.shape}, "
        f"pairwise_distance={pdist(M)[0]:.6f}"
    )

print("Generator V2 protocol locked.")
print("Output:", SIM_V2)


REGULAR SIMPLEX GEOMETRY VERIFIED
K=2: shape=(2, 3), pairwise_distance=1.414214
K=3: shape=(3, 3), pairwise_distance=1.414214
K=4: shape=(4, 3), pairwise_distance=1.414214
Generator V2 protocol locked.
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/generator_pilot_v2


# Bangkitkan 70 dataset V2 dengan ground truth terpisah

In [ ]:

# ============================================================
# CELL 6 — GENERATE & PRESERVE V2 PILOT DATASETS
# ============================================================

DATA_V2 = SIM_V2 / "datasets"
TRUTH_V2 = SIM_V2 / "ground_truth"

DATA_V2.mkdir(exist_ok=True)
TRUTH_V2.mkdir(exist_ok=True)

scenario_names_v2 = list(SCENARIOS_V2)

children_v2 = np.random.SeedSequence(
    SEED_WORDS_V2
).spawn(
    len(scenario_names_v2) * N_REP_V2
)

manifest_records_v2 = []
counter = 0


def verify_or_save_frame(path, frame):
    """
    Never overwrite an existing simulated dataset.
    Verify values against deterministic regeneration.
    """

    if path.exists():

        previous = pd.read_csv(path)

        assert list(previous.columns) == list(
            frame.columns
        )

        assert previous.shape == frame.shape

        for column in frame.columns:

            a = previous[column].to_numpy()
            b = frame[column].to_numpy()

            assert np.allclose(
                a, b,
                atol=1e-12,
                rtol=1e-12
            ), f"Mismatch: {path}, {column}"

    else:
        frame.to_csv(
            path,
            index=False,
            float_format="%.17g"
        )


for scenario in scenario_names_v2:

    for rep in range(1, N_REP_V2 + 1):

        child = children_v2[counter]
        counter += 1

        rng = np.random.default_rng(child)

        X, truth, nuisance = generate_multiview_v2(
            scenario, rng
        )

        dataset_id = f"{scenario}_r{rep:03d}"

        data_file = (
            DATA_V2 / f"{dataset_id}.csv"
        )

        truth_file = (
            TRUTH_V2 / f"{dataset_id}.csv"
        )

        data_frame = pd.DataFrame(
            X, columns=FEATURES_V2
        )

        data_frame.insert(
            0, "observation_id",
            np.arange(N_V2, dtype=int)
        )

        truth_frame = pd.DataFrame({
            "observation_id": np.arange(
                N_V2, dtype=int
            ),
            "true_cluster": truth,
            "nuisance_climate_group": nuisance
        })

        verify_or_save_frame(
            data_file, data_frame
        )

        verify_or_save_frame(
            truth_file, truth_frame
        )

        manifest_records_v2.append({
            "dataset_id": dataset_id,
            "scenario": scenario,
            "replicate": rep,
            "K_true": (
                SCENARIOS_V2[scenario]["K"]
                if SCENARIOS_V2[scenario]["K"] is not None
                else -1
            ),
            "data_sha256": hashlib.sha256(
                data_file.read_bytes()
            ).hexdigest(),
            "truth_sha256": hashlib.sha256(
                truth_file.read_bytes()
            ).hexdigest()
        })


manifest_v2 = pd.DataFrame(
    manifest_records_v2
)

assert len(manifest_v2) == 70
assert manifest_v2["dataset_id"].is_unique
assert len(list(DATA_V2.glob("*.csv"))) == 70
assert len(list(TRUTH_V2.glob("*.csv"))) == 70

manifest_path = (
    SIM_V2 / "simulation_dataset_manifest.csv"
)

if manifest_path.exists():
    previous_manifest = pd.read_csv(manifest_path)
    assert previous_manifest.equals(manifest_v2)
else:
    manifest_v2.to_csv(
        manifest_path, index=False
    )

print("V2 DATA GENERATION COMPLETED")
print("Total datasets:", len(manifest_v2))
print("Ground truth stored separately.")
print("Output:", SIM_V2)

display(
    manifest_v2.groupby(
        ["scenario", "K_true"],
        as_index=False
    ).size().rename(
        columns={"size": "n_datasets"}
    )
)


V2 DATA GENERATION COMPLETED
Total datasets: 70
Ground truth stored separately.
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/generator_pilot_v2


,scenario,K_true,n_datasets
0,aligned_K2,2,10
1,aligned_K3,3,10
2,aligned_K4,4,10
3,conflict_K3,3,10
4,imbalanced_K3,3,10
5,null_gaussian,-1,10
6,weak_K3,3,10


# Audit akhir generator V2 dan perbandingan V1

In [ ]:

# ============================================================
# CELL 7 — V2 GENERATOR VALIDATION
#
# Ground truth is used ONLY to audit the generator.
# No clustering model is fitted.
# ============================================================

from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score
)
from IPython.display import display
import numpy as np
import pandas as pd
import hashlib

assert len(manifest_v2) == 70
assert callable(representation_R0)

audit_v2_records = []

for r in manifest_v2.itertuples(index=False):

    data_path = DATA_V2 / f"{r.dataset_id}.csv"
    truth_path = TRUTH_V2 / f"{r.dataset_id}.csv"

    assert hashlib.sha256(
        data_path.read_bytes()
    ).hexdigest() == r.data_sha256

    assert hashlib.sha256(
        truth_path.read_bytes()
    ).hexdigest() == r.truth_sha256

    data = pd.read_csv(data_path)
    target = pd.read_csv(truth_path)

    assert np.array_equal(
        data["observation_id"],
        target["observation_id"]
    )

    X = data[FEATURES_V2].to_numpy(dtype=float)
    y = target["true_cluster"].to_numpy(dtype=int)
    nuisance = target[
        "nuisance_climate_group"
    ].to_numpy(dtype=int)

    assert X.shape == (85, 9)
    assert np.isfinite(X).all()

    covariance = np.cov(X, rowvar=False)

    min_eigenvalue = float(
        np.linalg.eigvalsh(covariance).min()
    )

    assert min_eigenvalue > 1e-10

    Phi, D, D_view = representation_R0(X)

    assert D.shape == (85, 85)
    assert np.allclose(D, D.T, atol=1e-12)
    assert np.allclose(np.diag(D), 0, atol=1e-12)

    record = {
        "dataset_id": r.dataset_id,
        "scenario": r.scenario,
        "K_true": r.K_true,
        "min_cov_eigenvalue": min_eigenvalue,
        "sil_truth_fused": np.nan,
        "sil_truth_food": np.nan,
        "sil_truth_climate": np.nan,
        "sil_truth_access": np.nan,
        "negative_sil_pct": np.nan,
        "ARI_truth_nuisance": np.nan,
        "climate_conflict_gap": np.nan
    }

    if r.K_true == -1:
        assert np.all(y == -1)
        assert np.all(nuisance == -1)

    else:
        assert len(np.unique(y)) == r.K_true

        record["sil_truth_fused"] = silhouette_score(
            D, y, metric="precomputed"
        )

        for view, key in [
            ("Food", "sil_truth_food"),
            ("Climate", "sil_truth_climate"),
            ("Accessibility", "sil_truth_access")
        ]:
            record[key] = silhouette_score(
                D_view[view],
                y,
                metric="precomputed"
            )

        samples = silhouette_samples(
            D, y, metric="precomputed"
        )

        record["negative_sil_pct"] = float(
            100 * np.mean(samples < 0)
        )

        if r.scenario == "conflict_K3":

            record["ARI_truth_nuisance"] = (
                adjusted_rand_score(
                    y, nuisance
                )
            )

            nuisance_sil = silhouette_score(
                D_view["Climate"],
                nuisance,
                metric="precomputed"
            )

            record["climate_conflict_gap"] = (
                nuisance_sil
                - record["sil_truth_climate"]
            )

    audit_v2_records.append(record)


audit_v2 = pd.DataFrame(
    audit_v2_records
)

summary_v2 = (
    audit_v2.groupby(
        ["scenario", "K_true"],
        as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "count"),
        mean_sil_fused=("sil_truth_fused", "mean"),
        sd_sil_fused=("sil_truth_fused", "std"),
        mean_sil_food=("sil_truth_food", "mean"),
        mean_sil_climate=("sil_truth_climate", "mean"),
        mean_sil_access=("sil_truth_access", "mean"),
        mean_negative_pct=("negative_sil_pct", "mean"),
        mean_conflict_gap=("climate_conflict_gap", "mean"),
        min_cov_eigenvalue=("min_cov_eigenvalue", "min")
    )
)

print("A. V2 GROUND-TRUTH GEOMETRY")
display(summary_v2.round(4))

# ------------------------------------------------------------
# V1 vs V2 comparison: descriptive only
# ------------------------------------------------------------

V1_DIR = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v1"
)

v1_summary_file = (
    V1_DIR / "generator_signal_audit_summary.csv"
)

assert v1_summary_file.exists()

v1_summary = pd.read_csv(
    v1_summary_file
)

comparison = v1_summary[
    ["scenario", "mean_sil_fused"]
].rename(columns={
    "mean_sil_fused": "V1_mean_sil"
}).merge(
    summary_v2[
        ["scenario", "mean_sil_fused"]
    ].rename(columns={
        "mean_sil_fused": "V2_mean_sil"
    }),
    on="scenario",
    validate="one_to_one"
)

comparison["V2_minus_V1"] = (
    comparison["V2_mean_sil"]
    - comparison["V1_mean_sil"]
)

print("\nB. GENERATOR V1 VS V2")
display(comparison.round(4))

# ------------------------------------------------------------
# Conflict audit
# ------------------------------------------------------------

conflict_v2 = audit_v2.loc[
    audit_v2["scenario"] == "conflict_K3"
]

assert len(conflict_v2) == 10

print("\nC. CONFLICT VIEW CHECK")
print(
    "Positive climate conflict gap:",
    int(
        (conflict_v2["climate_conflict_gap"] > 0)
        .sum()
    ),
    "/10"
)

print(
    "Mean ARI between truth and nuisance:",
    round(
        conflict_v2["ARI_truth_nuisance"].mean(), 4
    )
)

# ------------------------------------------------------------
# Fixed outputs
# ------------------------------------------------------------

audit_v2.to_csv(
    SIM_V2 / "generator_validation_all.csv",
    index=False
)

summary_v2.to_csv(
    SIM_V2 / "generator_validation_summary.csv",
    index=False
)

comparison.to_csv(
    SIM_V2 / "generator_V1_V2_comparison.csv",
    index=False
)

print("\nV2 GENERATOR AUDIT COMPLETED")
print("Datasets verified:", len(audit_v2))
print("Output:", SIM_V2)


A. V2 GROUND-TRUTH GEOMETRY


,scenario,K_true,n_datasets,mean_sil_fused,sd_sil_fused,mean_sil_food,mean_sil_climate,mean_sil_access,mean_negative_pct,mean_conflict_gap,min_cov_eigenvalue
0,aligned_K2,2,10,0.1992,0.0200,0.2066,0.1778,0.1704,6.5882,NaN,0.3765
1,aligned_K3,3,10,0.1600,0.0146,0.1034,0.1182,0.1054,9.1765,NaN,0.4141
2,aligned_K4,4,10,0.1588,0.0293,0.0892,0.0904,0.0885,10.7059,NaN,0.4201
3,conflict_K3,3,10,0.0925,0.0151,0.1086,-0.0369,0.1010,18.1176,0.1985,0.4796
4,imbalanced_K3,3,10,0.1693,0.0229,0.1173,0.1390,0.1128,12.7059,NaN,0.4072
5,null_gaussian,-1,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.4132
6,weak_K3,3,10,0.0060,0.0110,-0.0193,-0.0210,-0.0217,46.2353,NaN,0.3571



B. GENERATOR V1 VS V2


,scenario,V1_mean_sil,V2_mean_sil,V2_minus_V1
0,aligned_K2,0.4050,0.1992,-0.2059
1,aligned_K3,0.1876,0.1600,-0.0276
2,aligned_K4,0.0883,0.1588,0.0705
3,conflict_K3,0.0738,0.0925,0.0188
4,imbalanced_K3,0.1966,0.1693,-0.0273
5,null_gaussian,NaN,NaN,NaN
6,weak_K3,0.0133,0.0060,-0.0073



C. CONFLICT VIEW CHECK
Positive climate conflict gap: 10 /10
Mean ARI between truth and nuisance: -0.0018

V2 GENERATOR AUDIT COMPLETED
Datasets verified: 70
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/generator_pilot_v2


# kunci ruang kandidat dan protokol recovery pilot

In [ ]:

# ============================================================
# CELL 8 — GROUND-TRUTH RECOVERY PILOT PROTOCOL
#
# 70 synthetic datasets × 70 candidate configurations.
# No truth labels used during fitting or selection.
# ============================================================

!pip -q install "kmedoids==0.5.5"

from pathlib import Path
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import SpectralClustering
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    adjusted_rand_score
)
from importlib.metadata import version
from IPython.display import display

import numpy as np
import pandas as pd
import scipy
import sklearn
import kmedoids
import hashlib
import json
import os

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

SIM_V2 = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v2"
)

RECOVERY_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "recovery_pilot_v1"
)

RECOVERY_OUT.mkdir(parents=True, exist_ok=True)

DATA_V2 = SIM_V2 / "datasets"
TRUTH_V2 = SIM_V2 / "ground_truth"

MANIFEST = pd.read_csv(
    SIM_V2 / "simulation_dataset_manifest.csv"
)

FEATURES = [
    "F1_ikp_ketersediaan",
    "F2_ikp_keterjangkauan",
    "F3_ikp_pemanfaatan",
    "C1_anomali_hujan_mm",
    "C2_cdd_hari",
    "C3_hari_hujan_ekstrem",
    "A1_persen_tanpa_angkutan",
    "A2_persen_jalan_tanah_lainnya",
    "A3_persen_tanpa_pertokoan_pasar"
]

VIEW_NAMES = ["Food", "Climate", "Accessibility"]

WEIGHTS = {
    "equal": (1/3, 1/3, 1/3),
    "food50": (0.50, 0.25, 0.25),
    "climate50": (0.25, 0.50, 0.25),
    "access50": (0.25, 0.25, 0.50)
}

MODEL_SEED = 20261018
K_GRID = [2, 3, 4, 5, 6]
KG_GRID = [8, 16]
KS_FIXED = 7

assert len(MANIFEST) == 70
assert MANIFEST["dataset_id"].is_unique
assert len(FEATURES) == 9

for record in MANIFEST.itertuples(index=False):
    path = DATA_V2 / f"{record.dataset_id}.csv"
    assert path.exists()
    assert hashlib.sha256(
        path.read_bytes()
    ).hexdigest() == record.data_sha256

PLAN = []

for method in [
    "KMedoids_PAM",
    "Hierarchical_Ward",
    "Hierarchical_Average",
    "Hierarchical_Complete"
]:
    for K in K_GRID:
        PLAN.append({
            "model_id": f"{method}_K{K}",
            "method": method,
            "K": K,
            "k_graph": np.nan,
            "k_scale": np.nan,
            "weight_id": "na"
        })

for kg in KG_GRID:
    for K in K_GRID:
        PLAN.append({
            "model_id": f"Early_K{K}_G{kg}_S7",
            "method": "Spectral_Early",
            "K": K,
            "k_graph": kg,
            "k_scale": KS_FIXED,
            "weight_id": "na"
        })

for weight_id in WEIGHTS:
    for kg in KG_GRID:
        for K in K_GRID:
            PLAN.append({
                "model_id": (
                    f"Multi_K{K}_G{kg}_S7_{weight_id}"
                ),
                "method": "Spectral_MultiView",
                "K": K,
                "k_graph": kg,
                "k_scale": KS_FIXED,
                "weight_id": weight_id
            })

GRID_SIM = pd.DataFrame(PLAN)

assert len(GRID_SIM) == 70
assert GRID_SIM["model_id"].is_unique

def file_sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

PROTOCOL = {
    "stage": "ground_truth_recovery_pilot",
    "generator_version": "regular_simplex_multiview_v2",
    "generator_protocol_sha256": file_sha256(
        SIM_V2 / "generator_protocol.json"
    ),
    "generator_manifest_sha256": file_sha256(
        SIM_V2 / "simulation_dataset_manifest.csv"
    ),
    "n_datasets": 70,
    "n_configurations": 70,
    "K_grid": K_GRID,
    "k_graph_grid": KG_GRID,
    "k_scale": KS_FIXED,
    "weights": WEIGHTS,
    "model_seed": MODEL_SEED,
    "representation": "R0",
    "quality_metric": "silhouette_R0",
    "ground_truth_policy": (
        "for evaluation only; "
        "not available to model fitting"
    ),
    "status": "PILOT_NOT_CONFIRMATORY",
    "versions": {
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "sklearn": sklearn.__version__,
        "kmedoids": version("kmedoids")
    }
}

protocol_path = RECOVERY_OUT / "protocol.json"

if protocol_path.exists():
    assert json.loads(
        protocol_path.read_text(encoding="utf-8")
    ) == PROTOCOL
else:
    protocol_path.write_text(
        json.dumps(PROTOCOL, indent=2),
        encoding="utf-8"
    )

grid_file = RECOVERY_OUT / "candidate_grid.csv"

if grid_file.exists():
    previous = pd.read_csv(grid_file)
    pd.testing.assert_frame_equal(
        previous,
        GRID_SIM,
        check_dtype=False
    )
else:
    GRID_SIM.to_csv(grid_file, index=False)

print("RECOVERY PILOT PROTOCOL LOCKED")
print("Datasets:", len(MANIFEST))
print("Candidates per dataset:", len(GRID_SIM))
print("Expected full-data fits:", 70 * 70)

display(
    GRID_SIM.groupby("method")
    .size()
    .reset_index(name="n_configurations")
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 kB 2.5 MB/s eta 0:00:00
RECOVERY PILOT PROTOCOL LOCKED
Datasets: 70
Candidates per dataset: 70
Expected full-data fits: 4900


,method,n_configurations
0,Hierarchical_Average,5
1,Hierarchical_Complete,5
2,Hierarchical_Ward,5
3,KMedoids_PAM,5
4,Spectral_Early,10
5,Spectral_MultiView,40


# Jalankan 70 konfigurasi pada 70 dataset sintetis

In [ ]:

# ============================================================
# CELL 9 — SYNTHETIC RECOVERY PILOT FITTING
#
# Full dataset clustering only.
# Truth files are NEVER opened in this cell.
# ============================================================

QUALITY_DIR = RECOVERY_OUT / "quality"
LABEL_DIR = RECOVERY_OUT / "labels"
ERROR_DIR = RECOVERY_OUT / "errors"

for directory in [QUALITY_DIR, LABEL_DIR, ERROR_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def make_graph_sim(D, kg, ks):
    D = np.asarray(D, dtype=float)
    n = D.shape[0]

    if D.shape != (n, n):
        raise ValueError("Invalid distance dimensions.")

    if not np.allclose(D, D.T, atol=1e-12):
        raise ValueError("Distance not symmetric.")

    if not np.isfinite(D).all() or (D < 0).any():
        raise ValueError("Invalid distance values.")

    if not 1 <= max(kg, ks) < n:
        raise ValueError("Invalid k.")

    A = D.copy()
    np.fill_diagonal(A, np.inf)

    order = np.argsort(
        A, axis=1, kind="stable"
    )

    sigma = A[
        np.arange(n),
        order[:, ks - 1]
    ]

    if (sigma <= 0).any():
        raise ValueError("Invalid bandwidth.")

    adjacency = np.zeros(
        (n, n), dtype=bool
    )

    adjacency[
        np.arange(n)[:, None],
        order[:, :kg]
    ] = True

    adjacency |= adjacency.T

    W = np.exp(
        -(D ** 2) /
        (sigma[:, None] * sigma[None, :])
    )

    W = np.where(adjacency, W, 0.0)
    np.fill_diagonal(W, 0.0)

    n_components = connected_components(
        (W > 0).astype(np.uint8),
        directed=False
    )[0]

    if n_components != 1:
        raise ValueError(
            f"Disconnected graph: {n_components}"
        )

    return W


def make_sim_context(X):

    # Reuse the audited V2 representation from Cell 3.
    Phi, D, D_views = representation_R0(X)

    trees = {
        "Hierarchical_Ward": linkage(
            Phi,
            method="ward",
            metric="euclidean"
        ),
        "Hierarchical_Average": linkage(
            squareform(D, checks=True),
            method="average"
        ),
        "Hierarchical_Complete": linkage(
            squareform(D, checks=True),
            method="complete"
        )
    }

    graph_cache = {}

    for kg in KG_GRID:

        graph_cache[("Early", kg)] = (
            make_graph_sim(D, kg, KS_FIXED)
        )

        for view in VIEW_NAMES:

            Wv = make_graph_sim(
                D_views[view], kg, KS_FIXED
            )

            graph_cache[(view, kg)] = (
                Wv / np.linalg.norm(Wv, "fro")
            )

    return {
        "Phi": Phi,
        "D": D,
        "D_views": D_views,
        "trees": trees,
        "graphs": graph_cache
    }


def fit_sim_candidate(row, ctx):

    method = row["method"]
    K = int(row["K"])
    density = np.nan

    if method == "KMedoids_PAM":

        labels = kmedoids.pam(
            ctx["D"],
            K,
            init="build",
            max_iter=300,
            random_state=MODEL_SEED
        ).labels

    elif method in ctx["trees"]:

        labels = cut_tree(
            ctx["trees"][method],
            n_clusters=K
        ).ravel()

    elif method.startswith("Spectral_"):

        kg = int(row["k_graph"])

        if method == "Spectral_Early":
            W = ctx["graphs"][("Early", kg)]

        elif method == "Spectral_MultiView":

            weights = WEIGHTS[row["weight_id"]]

            W = sum(
                weights[i]
                * ctx["graphs"][(view, kg)]
                for i, view in enumerate(VIEW_NAMES)
            )

        else:
            raise ValueError("Unknown spectral method.")

        fitted = SpectralClustering(
            n_clusters=K,
            affinity="precomputed",
            assign_labels="kmeans",
            eigen_solver="arpack",
            n_init=20,
            random_state=MODEL_SEED
        )

        labels = fitted.fit_predict(W)

        n = len(W)
        density = (
            np.count_nonzero(
                np.triu(W, k=1)
            )
            / (n * (n - 1) / 2)
        )

    else:
        raise ValueError(f"Unknown method: {method}")

    labels = np.asarray(labels, dtype=int)

    counts = np.bincount(labels)

    if len(np.unique(labels)) != K:
        raise ValueError("Wrong cluster count.")

    return labels, counts, density


for record in MANIFEST.itertuples(index=False):

    dataset_id = record.dataset_id

    quality_file = QUALITY_DIR / f"{dataset_id}.csv"
    labels_file = LABEL_DIR / f"{dataset_id}.csv"

    if quality_file.exists() and labels_file.exists():

        old_q = pd.read_csv(quality_file)
        old_l = pd.read_csv(labels_file)

        assert len(old_q) == len(GRID_SIM)
        assert len(old_l) == 85
        assert old_q["model_id"].is_unique
        assert set(old_q["model_id"]) == set(
            GRID_SIM["model_id"]
        )
        assert set(old_l.columns) == (
            set(GRID_SIM["model_id"])
            | {"observation_id"}
        )
        print("Verified:", dataset_id)
        continue

    if quality_file.exists() != labels_file.exists():
        raise RuntimeError(
            f"Incomplete checkpoint: {dataset_id}"
        )

    data = pd.read_csv(
        DATA_V2 / f"{dataset_id}.csv"
    )

    X = data[FEATURES].to_numpy(dtype=float)

    assert X.shape == (85, 9)

    ctx = make_sim_context(X)

    quality_rows = []
    failed_rows = []

    label_columns = {
        "observation_id": data[
            "observation_id"
        ].to_numpy()
    }

    for row in GRID_SIM.to_dict("records"):

        try:

            labels, sizes, density = (
                fit_sim_candidate(row, ctx)
            )

            sample_sil = silhouette_samples(
                ctx["D"],
                labels,
                metric="precomputed"
            )

            quality_rows.append({
                **row,
                "dataset_id": dataset_id,
                "silhouette_R0": float(
                    sample_sil.mean()
                ),
                "negative_silhouette_pct": float(
                    100 * np.mean(sample_sil < 0)
                ),
                "smallest_cluster": int(
                    sizes.min()
                ),
                "largest_cluster": int(
                    sizes.max()
                ),
                "graph_density": density
            })

            label_columns[row["model_id"]] = labels

        except Exception as exc:

            failed_rows.append({
                "dataset_id": dataset_id,
                "model_id": row["model_id"],
                "error_type": type(exc).__name__,
                "error": str(exc)
            })

    # Do not silently drop failed configurations.
    if failed_rows:

        pd.DataFrame(failed_rows).to_csv(
            ERROR_DIR / f"{dataset_id}.csv",
            index=False
        )

        raise RuntimeError(
            f"{dataset_id}: "
            f"{len(failed_rows)} failed configurations. "
            "Inspect error file before continuing."
        )

    quality_df = pd.DataFrame(quality_rows)
    labels_df = pd.DataFrame(label_columns)

    assert len(quality_df) == 70
    assert len(labels_df.columns) == 71

    # Commit checkpoints only after all fits succeed.
    q_tmp = quality_file.with_suffix(".tmp")
    l_tmp = labels_file.with_suffix(".tmp")

    quality_df.to_csv(q_tmp, index=False)
    labels_df.to_csv(l_tmp, index=False)

    os.replace(l_tmp, labels_file)
    os.replace(q_tmp, quality_file)

    print(
        f"Completed {dataset_id}: "
        f"{len(quality_df)} configurations"
    )

print("\nRECOVERY PILOT FITTING COMPLETED")
print("Quality files:", len(list(QUALITY_DIR.glob("*.csv"))))
print("Label files:", len(list(LABEL_DIR.glob("*.csv"))))
print("Output:", RECOVERY_OUT)


Completed aligned_K2_r001: 70 configurations
Completed aligned_K2_r002: 70 configurations
Completed aligned_K2_r003: 70 configurations
Completed aligned_K2_r004: 70 configurations
Completed aligned_K2_r005: 70 configurations
Completed aligned_K2_r006: 70 configurations
Completed aligned_K2_r007: 70 configurations
Completed aligned_K2_r008: 70 configurations
Completed aligned_K2_r009: 70 configurations
Completed aligned_K2_r010: 70 configurations
Completed aligned_K3_r001: 70 configurations
Completed aligned_K3_r002: 70 configurations
Completed aligned_K3_r003: 70 configurations
Completed aligned_K3_r004: 70 configurations
Completed aligned_K3_r005: 70 configurations
Completed aligned_K3_r006: 70 configurations
Completed aligned_K3_r007: 70 configurations
Completed aligned_K3_r008: 70 configurations
Completed aligned_K3_r009: 70 configurations
Completed aligned_K3_r010: 70 configurations
Completed aligned_K4_r001: 70 configurations
Completed aligned_K4_r002: 70 configurations
Completed 

# Evaluasi pemulihan ground truth dan batas kemampuan kandidat

In [ ]:

# ============================================================
# CELL 10 — GROUND-TRUTH RECOVERY DIAGNOSTICS
#
# Ground truth accessed only AFTER model fitting.
#
# Quality-only: choose maximum full-data silhouette.
# Oracle: maximum true ARI among all candidate partitions.
#
# Oracle is a diagnostic upper bound, NOT an unsupervised
# selection method.
# ============================================================

quality_files = sorted(
    QUALITY_DIR.glob("*.csv")
)

label_files = sorted(
    LABEL_DIR.glob("*.csv")
)

assert len(quality_files) == 70
assert len(label_files) == 70

recovery_records = []
candidate_evaluation = []

for record in MANIFEST.itertuples(index=False):

    dataset_id = record.dataset_id

    quality = pd.read_csv(
        QUALITY_DIR / f"{dataset_id}.csv"
    )

    labels = pd.read_csv(
        LABEL_DIR / f"{dataset_id}.csv"
    )

    truth_file = (
        TRUTH_V2 / f"{dataset_id}.csv"
    )

    assert file_sha256(
        truth_file
    ) == record.truth_sha256

    ground_truth = pd.read_csv(truth_file)

    assert np.array_equal(
        labels["observation_id"].to_numpy(),
        ground_truth["observation_id"].to_numpy()
    )

    y_true = ground_truth[
        "true_cluster"
    ].to_numpy(dtype=int)

    is_null = (record.K_true == -1)

    # --------------------------------------------------------
    # A. Add recovery scores AFTER fitting
    # --------------------------------------------------------

    evaluation_rows = []

    for row in quality.to_dict("records"):

        model_id = row["model_id"]

        fitted_labels = labels[
            model_id
        ].to_numpy(dtype=int)

        true_ari = (
            np.nan
            if is_null
            else adjusted_rand_score(
                y_true, fitted_labels
            )
        )

        evaluation_rows.append({
            **row,
            "true_ARI": true_ari
        })

    evaluated = pd.DataFrame(
        evaluation_rows
    )

    assert len(evaluated) == 70

    # --------------------------------------------------------
    # B. Quality-only selection
    # Deterministic tie-break:
    # 1. Higher silhouette
    # 2. Smaller K
    # 3. Lexicographically smaller model ID
    # --------------------------------------------------------

    quality_ranked = evaluated.sort_values(
        ["silhouette_R0", "K", "model_id"],
        ascending=[False, True, True],
        kind="stable"
    )

    quality_selected = quality_ranked.iloc[0]

    result = {
        "dataset_id": dataset_id,
        "scenario": record.scenario,
        "replicate": record.replicate,
        "K_true": int(record.K_true),
        "quality_model": quality_selected["model_id"],
        "quality_K": int(quality_selected["K"]),
        "quality_silhouette": float(
            quality_selected["silhouette_R0"]
        ),
        "quality_true_ARI": np.nan,
        "quality_correct_K": np.nan,
        "oracle_model": "",
        "oracle_K": np.nan,
        "oracle_true_ARI": np.nan,
        "oracle_gap": np.nan
    }

    if not is_null:

        assert len(np.unique(y_true)) == record.K_true

        # Oracle sees true labels; diagnostic only.
        oracle_ranked = evaluated.sort_values(
            ["true_ARI", "silhouette_R0", "model_id"],
            ascending=[False, False, True],
            kind="stable"
        )

        oracle_selected = oracle_ranked.iloc[0]

        result.update({
            "quality_true_ARI": float(
                quality_selected["true_ARI"]
            ),
            "quality_correct_K": int(
                quality_selected["K"] == record.K_true
            ),
            "oracle_model": oracle_selected["model_id"],
            "oracle_K": int(oracle_selected["K"]),
            "oracle_true_ARI": float(
                oracle_selected["true_ARI"]
            ),
            "oracle_gap": float(
                oracle_selected["true_ARI"]
                - quality_selected["true_ARI"]
            )
        })

        assert result["oracle_gap"] >= -1e-12

    recovery_records.append(result)

    evaluated["scenario"] = record.scenario
    evaluated["replicate"] = record.replicate
    candidate_evaluation.append(evaluated)


recovery = pd.DataFrame(
    recovery_records
)

all_candidate_eval = pd.concat(
    candidate_evaluation,
    ignore_index=True
)

assert len(recovery) == 70
assert len(all_candidate_eval) == 4900
assert not recovery["dataset_id"].duplicated().any()

# ------------------------------------------------------------
# C. Scenario summary
# Unit: independently generated dataset
# ------------------------------------------------------------

summary_recovery = (
    recovery.groupby(
        ["scenario", "K_true"],
        as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "count"),
        mean_quality_ARI=("quality_true_ARI", "mean"),
        sd_quality_ARI=("quality_true_ARI", "std"),
        mean_oracle_ARI=("oracle_true_ARI", "mean"),
        mean_oracle_gap=("oracle_gap", "mean"),
        quality_K_accuracy=("quality_correct_K", "mean"),
        mean_quality_silhouette=(
            "quality_silhouette", "mean"
        )
    )
)

print("A. GROUND-TRUTH RECOVERY BY SCENARIO")
display(summary_recovery.round(4))

print("\nB. QUALITY-ONLY SELECTED K DISTRIBUTION")

K_distribution = (
    recovery.groupby(
        ["scenario", "quality_K"]
    )
    .size()
    .reset_index(name="n_datasets")
)

display(K_distribution)

print("\nC. LARGEST QUALITY-ONLY VS ORACLE GAPS")

non_null = recovery.loc[
    recovery["K_true"] > 0
]

display(
    non_null.sort_values(
        "oracle_gap",
        ascending=False
    )[
        [
            "dataset_id",
            "K_true",
            "quality_model",
            "quality_K",
            "quality_true_ARI",
            "oracle_model",
            "oracle_K",
            "oracle_true_ARI",
            "oracle_gap"
        ]
    ].head(15).round(4)
)

print("\nD. NULL GAUSSIAN — DIAGNOSTIC ONLY")

null_results = recovery.loc[
    recovery["K_true"] == -1
]

assert len(null_results) == 10
assert null_results["quality_true_ARI"].isna().all()

display(
    null_results[
        [
            "dataset_id",
            "quality_model",
            "quality_K",
            "quality_silhouette"
        ]
    ].round(4)
)

print(
    "\nNote: all null datasets are forced into "
    "K=2...6 by the candidate grid. "
    "No cluster-rejection mechanism exists yet."
)

# ------------------------------------------------------------
# E. Save complete diagnostics
# ------------------------------------------------------------

recovery.to_csv(
    RECOVERY_OUT / "quality_only_recovery.csv",
    index=False
)

summary_recovery.to_csv(
    RECOVERY_OUT / "quality_only_recovery_summary.csv",
    index=False
)

all_candidate_eval.to_csv(
    RECOVERY_OUT / "all_candidate_truth_evaluation.csv",
    index=False
)

K_distribution.to_csv(
    RECOVERY_OUT / "selected_K_distribution.csv",
    index=False
)

print("\nGROUND-TRUTH RECOVERY PILOT COMPLETED")
print("No stability-aware selection performed yet.")
print("Output:", RECOVERY_OUT)


A. GROUND-TRUTH RECOVERY BY SCENARIO


,scenario,K_true,n_datasets,mean_quality_ARI,sd_quality_ARI,mean_oracle_ARI,mean_oracle_gap,quality_K_accuracy,mean_quality_silhouette
0,aligned_K2,2,10,0.3724,0.3976,0.7659,0.3935,1.0,0.2272
1,aligned_K3,3,10,0.2053,0.3286,0.7256,0.5204,0.3,0.2245
2,aligned_K4,4,10,0.2686,0.3466,0.7150,0.4464,0.4,0.2070
3,conflict_K3,3,10,0.0458,0.1434,0.4041,0.3582,0.1,0.2004
4,imbalanced_K3,3,10,0.1745,0.2262,0.6438,0.4693,0.1,0.2216
5,null_gaussian,-1,10,NaN,NaN,NaN,NaN,NaN,0.2575
6,weak_K3,3,10,0.0004,0.0019,0.0945,0.0941,0.0,0.2363



B. QUALITY-ONLY SELECTED K DISTRIBUTION


,scenario,quality_K,n_datasets
0,aligned_K2,2,10
1,aligned_K3,2,7
2,aligned_K3,3,3
3,aligned_K4,2,6
4,aligned_K4,4,4
5,conflict_K3,2,9
6,conflict_K3,3,1
7,imbalanced_K3,2,9
8,imbalanced_K3,3,1
9,null_gaussian,2,9



C. LARGEST QUALITY-ONLY VS ORACLE GAPS


,dataset_id,K_true,quality_model,quality_K,quality_true_ARI,oracle_model,oracle_K,oracle_true_ARI,oracle_gap
21,aligned_K4_r002,4,Hierarchical_Average_K2,2,-0.0006,Early_K4_G16_S7,4.0,0.8460,0.8466
26,aligned_K4_r007,4,Hierarchical_Average_K2,2,0.0002,Early_K4_G16_S7,4.0,0.8454,0.8452
12,aligned_K3_r003,3,Hierarchical_Average_K2,2,0.0017,Early_K3_G16_S7,3.0,0.8284,0.8267
28,aligned_K4_r009,4,Hierarchical_Average_K2,2,-0.0005,Early_K4_G16_S7,4.0,0.8163,0.8168
48,imbalanced_K3_r009,3,Hierarchical_Average_K2,2,0.0291,Multi_K2_G16_S7_food50,2.0,0.8094,0.7803
8,aligned_K2_r009,2,Hierarchical_Average_K2,2,-0.0005,Early_K2_G8_S7,2.0,0.7759,0.7764
0,aligned_K2_r001,2,Hierarchical_Average_K2,2,-0.0005,Early_K2_G8_S7,2.0,0.7759,0.7764
10,aligned_K3_r001,3,Hierarchical_Average_K2,2,-0.0006,Early_K3_G8_S7,3.0,0.7645,0.7650
22,aligned_K4_r003,4,Hierarchical_Average_K2,2,0.0016,Early_K4_G8_S7,4.0,0.7565,0.7550
2,aligned_K2_r003,2,Hierarchical_Average_K2,2,0.0006,Multi_K2_G16_S7_equal,2.0,0.7344,0.7339



D. NULL GAUSSIAN — DIAGNOSTIC ONLY


,dataset_id,quality_model,quality_K,quality_silhouette
60,null_gaussian_r001,Hierarchical_Average_K2,2,0.3098
61,null_gaussian_r002,Hierarchical_Average_K2,2,0.2911
62,null_gaussian_r003,Early_K2_G16_S7,2,0.1860
63,null_gaussian_r004,Hierarchical_Average_K2,2,0.3342
64,null_gaussian_r005,Early_K3_G16_S7,3,0.1265
65,null_gaussian_r006,Hierarchical_Average_K2,2,0.3075
66,null_gaussian_r007,Hierarchical_Complete_K2,2,0.2204
67,null_gaussian_r008,Hierarchical_Average_K2,2,0.3494
68,null_gaussian_r009,Hierarchical_Average_K2,2,0.2595
69,null_gaussian_r010,Hierarchical_Average_K2,2,0.1908



Note: all null datasets are forced into K=2...6 by the candidate grid. No cluster-rejection mechanism exists yet.

GROUND-TRUTH RECOVERY PILOT COMPLETED
No stability-aware selection performed yet.
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/recovery_pilot_v1


# Protokol dan fungsi refitting simulasi

In [ ]:

# ============================================================
# CELL 11 — PAIRED STABILITY-SELECTION PILOT
#
# 7 scenarios × 2 datasets × 70 models × 20 subsamples
#
# The same subsamples are used for every candidate.
# Ground truth is NOT accessed during fitting.
# ============================================================

from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.spatial.distance import squareform
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import kmedoids
import hashlib
import json
import os

assert callable(representation_R0)
assert callable(make_graph_sim)

MODEL_SEED = 20261018
SUBSAMPLE_SEED = 20261023

B_PILOT = 20
PILOT_REPLICATES = 2
SUBSET_N = 68

PILOT_INPUTS = (
    MANIFEST.loc[
        MANIFEST["replicate"] <= PILOT_REPLICATES
    ]
    .sort_values(["scenario", "replicate"])
    .reset_index(drop=True)
)

assert len(PILOT_INPUTS) == 14
assert len(GRID_SIM) == 70
assert GRID_SIM["model_id"].is_unique

STAB_SIM_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "stability_recovery_pilot_v1"
)

STAB_SIM_OUT.mkdir(
    parents=True, exist_ok=True
)

SCORES_DIR = STAB_SIM_OUT / "subsampling_scores"
INDICES_DIR = STAB_SIM_OUT / "subsample_indices"

SCORES_DIR.mkdir(exist_ok=True)
INDICES_DIR.mkdir(exist_ok=True)


def sha256_sim(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


SIM_STAB_PROTOCOL = {
    "stage": "stability_recovery_pilot_v1",
    "dataset_ids": PILOT_INPUTS[
        "dataset_id"
    ].tolist(),
    "n_candidates": 70,
    "n_subsamples": B_PILOT,
    "subsample_size": SUBSET_N,
    "subsampling": "simple_random_without_replacement",
    "stratification": "none",
    "model_seed": MODEL_SEED,
    "subsample_seed": SUBSAMPLE_SEED,
    "preprocessing": "R0_refit_per_subset",
    "selection_truth_access": False,
    "candidate_grid_sha256": sha256_sim(
        RECOVERY_OUT / "candidate_grid.csv"
    ),
    "dataset_manifest_sha256": sha256_sim(
        SIM_V2 / "simulation_dataset_manifest.csv"
    ),
    "status": "EXPLORATORY_PILOT"
}

protocol_file = STAB_SIM_OUT / "protocol.json"

if protocol_file.exists():
    assert json.loads(
        protocol_file.read_text(encoding="utf-8")
    ) == SIM_STAB_PROTOCOL
else:
    protocol_file.write_text(
        json.dumps(
            SIM_STAB_PROTOCOL,
            indent=2
        ),
        encoding="utf-8"
    )


def make_subset_context_sim(X):
    """
    Refit R0 preprocessing and distance geometry.

    Spectral graphs are constructed lazily so failure
    of one graph does not invalidate unrelated methods.
    """

    Phi, D, D_views = representation_R0(X)

    trees = {
        "Hierarchical_Ward": linkage(
            Phi, method="ward",
            metric="euclidean"
        ),
        "Hierarchical_Average": linkage(
            squareform(D, checks=True),
            method="average"
        ),
        "Hierarchical_Complete": linkage(
            squareform(D, checks=True),
            method="complete"
        )
    }

    return {
        "D": D,
        "D_views": D_views,
        "trees": trees,
        "cache": {}
    }


def get_subset_affinity(ctx, row):

    method = row["method"]
    kg = int(row["k_graph"])
    ks = int(row["k_scale"])

    cache = ctx["cache"]

    if method == "Spectral_Early":

        key = ("early", kg, ks)

        if key not in cache:
            cache[key] = make_graph_sim(
                ctx["D"], kg, ks
            )

        return cache[key]

    if method == "Spectral_MultiView":

        weight_id = row["weight_id"]
        fused_key = ("fused", kg, ks, weight_id)

        if fused_key not in cache:

            weights = WEIGHTS[weight_id]
            W = np.zeros_like(ctx["D"])

            for i, view in enumerate(VIEW_NAMES):

                key = ("view", view, kg, ks)

                if key not in cache:

                    Wv = make_graph_sim(
                        ctx["D_views"][view],
                        kg, ks
                    )

                    norm = np.linalg.norm(
                        Wv, ord="fro"
                    )

                    if norm <= 0:
                        raise ValueError(
                            "Invalid graph normalization."
                        )

                    cache[key] = Wv / norm

                W += weights[i] * cache[key]

            cache[fused_key] = W

        return cache[fused_key]

    raise ValueError("Unknown spectral method.")


def fit_subset_sim(row, ctx):

    method = row["method"]
    K = int(row["K"])

    if method == "KMedoids_PAM":

        labels = kmedoids.pam(
            ctx["D"],
            K,
            init="build",
            max_iter=300,
            random_state=MODEL_SEED
        ).labels

    elif method in ctx["trees"]:

        labels = cut_tree(
            ctx["trees"][method],
            n_clusters=K
        ).ravel()

    elif method.startswith("Spectral_"):

        W = get_subset_affinity(ctx, row)

        labels = SpectralClustering(
            n_clusters=K,
            affinity="precomputed",
            assign_labels="kmeans",
            eigen_solver="arpack",
            n_init=20,
            random_state=MODEL_SEED
        ).fit_predict(W)

    else:
        raise ValueError(
            f"Unknown method: {method}"
        )

    labels = np.asarray(labels, dtype=int)

    if len(np.unique(labels)) != K:
        raise ValueError(
            f"Expected {K} clusters."
        )

    return labels


# Reproduction check using one complete synthetic dataset
test_id = PILOT_INPUTS.iloc[0]["dataset_id"]

test_data = pd.read_csv(
    DATA_V2 / f"{test_id}.csv"
)

test_labels = pd.read_csv(
    LABEL_DIR / f"{test_id}.csv"
)

test_ctx = make_subset_context_sim(
    test_data[FEATURES].to_numpy(dtype=float)
)

reproduced = []

for row in GRID_SIM.to_dict("records"):

    fitted = fit_subset_sim(row, test_ctx)

    reference = test_labels[
        row["model_id"]
    ].to_numpy(dtype=int)

    ari = adjusted_rand_score(
        reference, fitted
    )

    reproduced.append(
        np.isclose(ari, 1.0, atol=1e-12)
    )

assert all(reproduced), (
    "Full-data reproduction failed. "
    "Inspect refitting implementation."
)

print("SIMULATION REFITTING VERIFIED")
print("Reference partitions reproduced:", sum(reproduced))
print("Pilot datasets:", len(PILOT_INPUTS))
print("Subsamples per dataset:", B_PILOT)
print(
    "Expected subset fits:",
    len(PILOT_INPUTS) * B_PILOT * len(GRID_SIM)
)
print("Output:", STAB_SIM_OUT)


SIMULATION REFITTING VERIFIED
Reference partitions reproduced: 70
Pilot datasets: 14
Subsamples per dataset: 20
Expected subset fits: 19600
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/stability_recovery_pilot_v1


# Jalankan stability refitting berpasangan

In [ ]:

# ============================================================
# CELL 12 — PAIRED SUBSAMPLING REFITTING
#
# Reference: full-data clustering from recovery pilot.
# Comparison: refitted clustering on the same observations.
#
# No ground truth read here.
# ============================================================

def get_or_create_subsamples(dataset_id, n):
    """
    Deterministic and dataset-specific subsampling.
    Reuses saved indices on notebook restart.
    """

    seed_id = int.from_bytes(
        hashlib.sha256(
            dataset_id.encode("utf-8")
        ).digest()[:4],
        byteorder="little"
    )

    rng = np.random.default_rng(
        np.random.SeedSequence([
            SUBSAMPLE_SEED, seed_id
        ])
    )

    generated = np.vstack([
        np.sort(
            rng.choice(
                n,
                size=SUBSET_N,
                replace=False
            )
        )
        for _ in range(B_PILOT)
    ])

    path = INDICES_DIR / f"{dataset_id}.csv"

    if path.exists():

        saved = pd.read_csv(
            path
        ).to_numpy(dtype=int)

        assert np.array_equal(
            saved, generated
        )

    else:

        subset_frame = pd.DataFrame(generated)

        temp = path.with_suffix(".tmp")
        subset_frame.to_csv(
            temp, index=False
        )
        os.replace(temp, path)

    assert generated.shape == (
        B_PILOT, SUBSET_N
    )

    return generated


def subset_partition_metrics(
    full_reference, subset_reference, fitted
):
    """
    ARI and reference-cluster Jaccard.

    Missing reference clusters are counted separately,
    rather than assigned an arbitrary Jaccard score.
    """

    ari = adjusted_rand_score(
        subset_reference, fitted
    )

    cluster_j = []

    for c in np.unique(subset_reference):

        a = subset_reference == c

        best_j = max(
            np.sum(a & (fitted == d))
            / np.sum(a | (fitted == d))
            for d in np.unique(fitted)
        )

        cluster_j.append(float(best_j))

    missing = (
        len(np.unique(full_reference))
        - len(np.unique(subset_reference))
    )

    return {
        "ARI": float(ari),
        "jaccard_macro": float(
            np.mean(cluster_j)
        ),
        "jaccard_min": float(
            np.min(cluster_j)
        ),
        "missing_reference_clusters": int(missing)
    }


candidate_rows = GRID_SIM.to_dict("records")

for record in PILOT_INPUTS.itertuples(index=False):

    dataset_id = record.dataset_id
    result_file = SCORES_DIR / f"{dataset_id}.csv"

    expected_rows = (
        len(candidate_rows) * B_PILOT
    )

    if result_file.exists():

        old = pd.read_csv(result_file)

        assert len(old) == expected_rows

        assert not old.duplicated(
            ["repeat", "model_id"]
        ).any()

        assert old["repeat"].nunique() == B_PILOT

        assert set(old["model_id"]) == set(
            GRID_SIM["model_id"]
        )

        print("Verified checkpoint:", dataset_id)
        continue

    input_file = DATA_V2 / f"{dataset_id}.csv"

    assert sha256_sim(
        input_file
    ) == record.data_sha256

    data = pd.read_csv(input_file)

    X = data[FEATURES].to_numpy(dtype=float)

    reference_labels = pd.read_csv(
        LABEL_DIR / f"{dataset_id}.csv"
    )

    assert np.array_equal(
        data["observation_id"].to_numpy(),
        reference_labels["observation_id"].to_numpy()
    )

    indices_matrix = get_or_create_subsamples(
        dataset_id, len(X)
    )

    rows = []

    for b, indices in enumerate(
        indices_matrix, start=1
    ):

        ctx = make_subset_context_sim(
            X[indices]
        )

        for model_row in candidate_rows:

            model_id = model_row["model_id"]

            ref_full = reference_labels[
                model_id
            ].to_numpy(dtype=int)

            ref_subset = ref_full[indices]

            try:

                fitted = fit_subset_sim(
                    model_row, ctx
                )

                metrics = subset_partition_metrics(
                    ref_full,
                    ref_subset,
                    fitted
                )

                rows.append({
                    "dataset_id": dataset_id,
                    "repeat": b,
                    "model_id": model_id,
                    "status": "SUCCESS",
                    **metrics,
                    "error": ""
                })

            except Exception as exc:

                rows.append({
                    "dataset_id": dataset_id,
                    "repeat": b,
                    "model_id": model_id,
                    "status": "FAILED",
                    "ARI": np.nan,
                    "jaccard_macro": np.nan,
                    "jaccard_min": np.nan,
                    "missing_reference_clusters": np.nan,
                    "error": (
                        f"{type(exc).__name__}: {exc}"
                    )
                })

    scores = pd.DataFrame(rows)

    assert len(scores) == expected_rows
    assert not scores.duplicated(
        ["repeat", "model_id"]
    ).any()

    temp = result_file.with_suffix(".tmp")
    scores.to_csv(temp, index=False)
    os.replace(temp, result_file)

    print(
        f"{dataset_id}: "
        f"{scores['status'].eq('SUCCESS').sum()}"
        f"/{expected_rows} successful fits"
    )

print("\nPAIRED STABILITY PILOT COMPLETED")
print("Expected datasets:", len(PILOT_INPUTS))
print("Saved score files:",
      len(list(SCORES_DIR.glob("*.csv"))))
print("Output:", STAB_SIM_OUT)


aligned_K2_r001: 1400/1400 successful fits
aligned_K2_r002: 1400/1400 successful fits
aligned_K3_r001: 1400/1400 successful fits
aligned_K3_r002: 1400/1400 successful fits
aligned_K4_r001: 1400/1400 successful fits
aligned_K4_r002: 1400/1400 successful fits
conflict_K3_r001: 1400/1400 successful fits
conflict_K3_r002: 1400/1400 successful fits
imbalanced_K3_r001: 1400/1400 successful fits
imbalanced_K3_r002: 1400/1400 successful fits
null_gaussian_r001: 1400/1400 successful fits
null_gaussian_r002: 1400/1400 successful fits
weak_K3_r001: 1400/1400 successful fits
weak_K3_r002: 1400/1400 successful fits

PAIRED STABILITY PILOT COMPLETED
Expected datasets: 14
Saved score files: 14
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/stability_recovery_pilot_v1


# Bandingkan strategi pemilihan model terhadap ground truth

In [ ]:

# ============================================================
# CELL 13 — PAIRED SELECTION-RECOVERY EVALUATION
#
# Ground truth is loaded ONLY AFTER all candidate
# quality and stability metrics have been calculated.
#
# Exploratory development pilot; not final inference.
# ============================================================

from sklearn.metrics import adjusted_rand_score
from IPython.display import display

import numpy as np
import pandas as pd

score_files = [
    SCORES_DIR / f"{d}.csv"
    for d in PILOT_INPUTS["dataset_id"]
]

assert all(p.exists() for p in score_files)

all_scores = pd.concat(
    [pd.read_csv(p) for p in score_files],
    ignore_index=True
)

expected = (
    len(PILOT_INPUTS)
    * len(GRID_SIM)
    * B_PILOT
)

assert len(all_scores) == expected

assert not all_scores.duplicated(
    ["dataset_id", "repeat", "model_id"]
).any()

print("A. COMPUTATIONAL COVERAGE")
print("Expected:", expected)
print("Successful:",
      int(all_scores["status"].eq("SUCCESS").sum()))
print("Failed:",
      int(all_scores["status"].eq("FAILED").sum()))

# ------------------------------------------------------------
# 1. STABILITY SUMMARIES
# ------------------------------------------------------------

stability_summary = (
    all_scores.groupby(
        ["dataset_id", "model_id"],
        as_index=False
    )
    .agg(
        n_success=(
            "status",
            lambda x: int(
                (x == "SUCCESS").sum()
            )
        ),
        mean_ARI=("ARI", "mean"),
        p10_ARI=(
            "ARI",
            lambda x: x.quantile(0.10)
        ),
        mean_jaccard=(
            "jaccard_macro", "mean"
        ),
        worst_mean_jaccard=(
            "jaccard_min", "mean"
        )
    )
)

assert len(stability_summary) == (
    len(PILOT_INPUTS) * len(GRID_SIM)
)

# Only models with complete refitting coverage
# are eligible for comparisons.
stability_summary["complete_refitting"] = (
    stability_summary["n_success"] == B_PILOT
)

DELTA_GRID = [0.02, 0.03, 0.05, 0.07, 0.10]
MIN_CLUSTER_GRID = [1, 5, 8]

evaluation_records = []
paired_records = []

for record in PILOT_INPUTS.itertuples(index=False):

    dataset_id = record.dataset_id

    quality = pd.read_csv(
        QUALITY_DIR / f"{dataset_id}.csv"
    )

    fitted_labels = pd.read_csv(
        LABEL_DIR / f"{dataset_id}.csv"
    )

    scores = stability_summary.loc[
        stability_summary["dataset_id"]
        == dataset_id
    ]

    candidate = quality.merge(
        scores.drop(columns="dataset_id"),
        on="model_id",
        validate="one_to_one"
    )

    assert len(candidate) == 70

    # --------------------------------------------------------
    # GROUND TRUTH ACCESSED ONLY DURING EVALUATION
    # --------------------------------------------------------

    true_data = pd.read_csv(
        TRUTH_V2 / f"{dataset_id}.csv"
    )

    assert np.array_equal(
        fitted_labels["observation_id"],
        true_data["observation_id"]
    )

    truth = true_data[
        "true_cluster"
    ].to_numpy(dtype=int)

    is_null = (record.K_true == -1)

    if is_null:
        true_ari_lookup = {}
    else:
        true_ari_lookup = {
            model_id: adjusted_rand_score(
                truth,
                fitted_labels[model_id].to_numpy(
                    dtype=int
                )
            )
            for model_id in candidate["model_id"]
        }

    # --------------------------------------------------------
    # COMPARE ALL POLICIES ON SAME ELIGIBLE CANDIDATES
    # --------------------------------------------------------

    for min_size in MIN_CLUSTER_GRID:

        eligible = candidate.loc[
            (
                candidate["smallest_cluster"]
                >= min_size
            )
            & candidate["complete_refitting"]
        ].copy()

        if eligible.empty:
            raise RuntimeError(
                f"No eligible models for {dataset_id}, "
                f"minimum size {min_size}"
            )

        qmax = eligible["silhouette_R0"].max()

        # Quality-only
        quality_selected = eligible.sort_values(
            [
                "silhouette_R0",
                "K",
                "model_id"
            ],
            ascending=[False, True, True],
            kind="stable"
        ).iloc[0]

        # Stability-only
        stability_selected = eligible.sort_values(
            [
                "mean_ARI",
                "p10_ARI",
                "silhouette_R0",
                "K",
                "model_id"
            ],
            ascending=[
                False, False, False,
                True, True
            ],
            kind="stable"
        ).iloc[0]

        choices = {
            "quality_only": (
                quality_selected,
                len(eligible),
                np.nan
            ),
            "stability_only": (
                stability_selected,
                len(eligible),
                np.nan
            )
        }

        for delta in DELTA_GRID:

            within_quality = eligible.loc[
                eligible["silhouette_R0"]
                >= qmax - delta - 1e-12
            ]

            selected = within_quality.sort_values(
                [
                    "mean_ARI",
                    "p10_ARI",
                    "silhouette_R0",
                    "K",
                    "model_id"
                ],
                ascending=[
                    False, False, False,
                    True, True
                ],
                kind="stable"
            ).iloc[0]

            choices[f"SA_delta_{delta:.2f}"] = (
                selected,
                len(within_quality),
                delta
            )

        # Oracle is evaluated but not used in any choice.
        if not is_null:
            oracle_ari = max(
                true_ari_lookup[m]
                for m in eligible["model_id"]
            )
        else:
            oracle_ari = np.nan

        for policy, (
            selected, n_allowed, delta
        ) in choices.items():

            model_id = selected["model_id"]

            evaluation_records.append({
                "dataset_id": dataset_id,
                "scenario": record.scenario,
                "replicate": record.replicate,
                "K_true": record.K_true,
                "min_cluster_rule": min_size,
                "policy": policy,
                "delta": delta,
                "n_eligible": len(eligible),
                "n_quality_allowed": n_allowed,
                "selected_model": model_id,
                "selected_K": int(selected["K"]),
                "selected_silhouette": float(
                    selected["silhouette_R0"]
                ),
                "selected_mean_ARI": float(
                    selected["mean_ARI"]
                ),
                "selected_p10_ARI": float(
                    selected["p10_ARI"]
                ),
                "selected_min_cluster": int(
                    selected["smallest_cluster"]
                ),
                "true_ARI": (
                    np.nan
                    if is_null
                    else float(
                        true_ari_lookup[model_id]
                    )
                ),
                "correct_K": (
                    np.nan
                    if is_null
                    else int(
                        selected["K"]
                        == record.K_true
                    )
                ),
                "eligible_oracle_ARI": oracle_ari
            })


selection_eval = pd.DataFrame(
    evaluation_records
)

expected_selection_rows = (
    len(PILOT_INPUTS)
    * len(MIN_CLUSTER_GRID)
    * (2 + len(DELTA_GRID))
)

assert len(selection_eval) == expected_selection_rows

assert not selection_eval.duplicated(
    [
        "dataset_id",
        "min_cluster_rule",
        "policy"
    ]
).any()

# ------------------------------------------------------------
# 2. PAIR EACH STRATEGY WITH QUALITY-ONLY
# ------------------------------------------------------------

baseline = (
    selection_eval.loc[
        selection_eval["policy"] == "quality_only",
        [
            "dataset_id",
            "min_cluster_rule",
            "true_ARI",
            "correct_K"
        ]
    ]
    .rename(columns={
        "true_ARI": "quality_only_true_ARI",
        "correct_K": "quality_only_correct_K"
    })
)

paired = selection_eval.merge(
    baseline,
    on=["dataset_id", "min_cluster_rule"],
    validate="many_to_one"
)

paired["delta_true_ARI"] = (
    paired["true_ARI"]
    - paired["quality_only_true_ARI"]
)

# ------------------------------------------------------------
# 3. DESCRIPTIVE PILOT RESULTS
# ------------------------------------------------------------

print("\nB. PAIRED RECOVERY — MINIMUM CLUSTER SIZE 1")

display(
    paired.loc[
        (paired["min_cluster_rule"] == 1)
        & (
            paired["policy"].isin([
                "quality_only",
                "stability_only",
                "SA_delta_0.03",
                "SA_delta_0.05",
                "SA_delta_0.07"
            ])
        )
        & (paired["K_true"] > 0)
    ]
    .groupby("policy")
    .agg(
        n_datasets=("dataset_id", "nunique"),
        mean_true_ARI=("true_ARI", "mean"),
        mean_delta_vs_quality=(
            "delta_true_ARI", "mean"
        ),
        correct_K_rate=("correct_K", "mean")
    )
    .sort_values(
        "mean_true_ARI", ascending=False
    )
    .round(4)
)

print("\nC. EFFECT OF MINIMUM CLUSTER SIZE")

display(
    paired.loc[
        (paired["K_true"] > 0)
        & (
            paired["policy"].isin([
                "quality_only",
                "SA_delta_0.05"
            ])
        )
    ]
    .groupby(
        ["min_cluster_rule", "policy"]
    )
    .agg(
        n_datasets=("dataset_id", "nunique"),
        mean_true_ARI=("true_ARI", "mean"),
        mean_delta_vs_quality=(
            "delta_true_ARI", "mean"
        ),
        correct_K_rate=("correct_K", "mean")
    )
    .round(4)
)

print("\nD. SCENARIO-SPECIFIC PAIRED DIFFERENCES")

display(
    paired.loc[
        (paired["min_cluster_rule"] == 5)
        & (paired["policy"] == "SA_delta_0.05")
        & (paired["K_true"] > 0),
        [
            "dataset_id",
            "scenario",
            "selected_model",
            "selected_K",
            "true_ARI",
            "quality_only_true_ARI",
            "delta_true_ARI",
            "eligible_oracle_ARI"
        ]
    ].round(4)
)

print("\nE. NULL DATASETS — NO REJECTION YET")

display(
    selection_eval.loc[
        (selection_eval["K_true"] == -1)
        & (selection_eval["min_cluster_rule"] == 5)
        & (
            selection_eval["policy"].isin([
                "quality_only",
                "SA_delta_0.05"
            ])
        ),
        [
            "dataset_id",
            "policy",
            "selected_model",
            "selected_K",
            "selected_silhouette",
            "selected_mean_ARI"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 4. SAVE ALL RESULTS
# ------------------------------------------------------------

stability_summary.to_csv(
    STAB_SIM_OUT / "stability_all_candidates.csv",
    index=False
)

all_scores.to_csv(
    STAB_SIM_OUT / "stability_all_repetitions.csv",
    index=False
)

selection_eval.to_csv(
    STAB_SIM_OUT / "selection_recovery_all_policies.csv",
    index=False
)

paired.to_csv(
    STAB_SIM_OUT / "paired_recovery_comparisons.csv",
    index=False
)

print("\nPAIRED SELECTION PILOT COMPLETED")
print("Datasets:", len(PILOT_INPUTS))
print("Candidate configurations:", len(GRID_SIM))
print("No inferential claim from this pilot.")
print("Output:", STAB_SIM_OUT)


A. COMPUTATIONAL COVERAGE
Expected: 19600
Successful: 19600
Failed: 0

B. PAIRED RECOVERY — MINIMUM CLUSTER SIZE 1


,n_datasets,mean_true_ARI,mean_delta_vs_quality,correct_K_rate
policy,,,,
SA_delta_0.07,12,0.4238,0.3254,0.5833
stability_only,12,0.3624,0.2640,0.5833
SA_delta_0.05,12,0.3010,0.2026,0.4167
SA_delta_0.03,12,0.1944,0.0961,0.2500
quality_only,12,0.0983,0.0000,0.3333



C. EFFECT OF MINIMUM CLUSTER SIZE


n_datasets  mean_true_ARI  \
min_cluster_rule policy                                     
1                SA_delta_0.05          12         0.3010   
                 quality_only           12         0.0983   
5                SA_delta_0.05          12         0.3490   
                 quality_only           12         0.4279   
8                SA_delta_0.05          12         0.3786   
                 quality_only           12         0.4742   

                                mean_delta_vs_quality  correct_K_rate  
min_cluster_rule policy                                                
1                SA_delta_0.05                 0.2026          0.4167  
                 quality_only                  0.0000          0.3333  
5                SA_delta_0.05                -0.0789          0.5000  
                 quality_only                  0.0000          0.5000  
8                SA_delta_0.05                -0.0956          0.5000  
                 quality_only                  0.0000          0.5833


D. SCENARIO-SPECIFIC PAIRED DIFFERENCES


,dataset_id,scenario,selected_model,selected_K,true_ARI,quality_only_true_ARI,delta_true_ARI,eligible_oracle_ARI
11,aligned_K2_r001,aligned_K2,Multi_K2_G16_S7_food50,2,0.6941,0.7759,-0.0818,0.7759
32,aligned_K2_r002,aligned_K2,Multi_K2_G8_S7_access50,2,0.5799,0.6168,-0.0369,0.6549
53,aligned_K3_r001,aligned_K3,Multi_K3_G8_S7_access50,3,0.3597,0.7645,-0.4048,0.7645
74,aligned_K3_r002,aligned_K3,Hierarchical_Average_K2,2,0.0063,0.0063,0.0000,0.5614
95,aligned_K4_r001,aligned_K4,Early_K4_G16_S7,4,0.6778,0.6700,0.0078,0.6778
116,aligned_K4_r002,aligned_K4,Early_K4_G16_S7,4,0.8460,0.8460,0.0000,0.8460
137,conflict_K3_r001,conflict_K3,Multi_K3_G16_S7_climate50,3,0.0465,0.2674,-0.2209,0.4427
158,conflict_K3_r002,conflict_K3,Early_K2_G16_S7,2,0.2053,0.2053,0.0000,0.3932
179,imbalanced_K3_r001,imbalanced_K3,Multi_K2_G16_S7_access50,2,0.5057,0.5131,-0.0075,0.6694
200,imbalanced_K3_r002,imbalanced_K3,Multi_K2_G16_S7_access50,2,0.2253,0.4750,-0.2498,0.5725



E. NULL DATASETS — NO REJECTION YET


,dataset_id,policy,selected_model,selected_K,selected_silhouette,selected_mean_ARI
217,null_gaussian_r001,quality_only,Hierarchical_Complete_K2,2,0.1478,0.2715
221,null_gaussian_r001,SA_delta_0.05,Early_K2_G16_S7,2,0.1321,0.8231
238,null_gaussian_r002,quality_only,Early_K2_G16_S7,2,0.1280,0.6449
242,null_gaussian_r002,SA_delta_0.05,Multi_K3_G16_S7_climate50,3,0.0867,0.7989



PAIRED SELECTION PILOT COMPLETED
Datasets: 14
Candidate configurations: 70
No inferential claim from this pilot.
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/stability_recovery_pilot_v1


# Audit kontribusi stabilitas setelah penyaringan ukuran cluster

In [ ]:

# ============================================================
# CELL 14 — ISOLATING THE INCREMENTAL VALUE OF STABILITY
#
# Descriptive mechanism audit on the SAME 14 pilot datasets.
# Uses saved selection results; no new model fitting.
#
# No confirmatory inference and no final delta tuning.
# ============================================================

from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

STAB_SIM_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "stability_recovery_pilot_v1"
)

RECOVERY_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "recovery_pilot_v1"
)

E = pd.read_csv(
    STAB_SIM_OUT / "selection_recovery_all_policies.csv"
)

G = pd.read_csv(
    RECOVERY_OUT / "candidate_grid.csv"
)

assert E["dataset_id"].nunique() == 14
assert set(E["min_cluster_rule"]) == {1, 5, 8}
assert G["model_id"].is_unique

assert not E.duplicated(
    ["dataset_id", "min_cluster_rule", "policy"]
).any()

nonnull = E.loc[E["K_true"] > 0].copy()
null = E.loc[E["K_true"] == -1].copy()

assert nonnull["dataset_id"].nunique() == 12
assert null["dataset_id"].nunique() == 2

# ------------------------------------------------------------
# A. AUDIT EVERY POLICY AND MINIMUM-SIZE COMBINATION
# ------------------------------------------------------------

summary_all = (
    nonnull.groupby(
        ["min_cluster_rule", "policy"],
        as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "nunique"),
        mean_true_ARI=("true_ARI", "mean"),
        median_true_ARI=("true_ARI", "median"),
        mean_K_accuracy=("correct_K", "mean"),
        mean_selected_silhouette=(
            "selected_silhouette", "mean"
        ),
        mean_selected_stability=(
            "selected_mean_ARI", "mean"
        ),
        mean_selected_min_size=(
            "selected_min_cluster", "mean"
        )
    )
)

print("A. COMPLETE POLICY SENSITIVITY")
display(
    summary_all.sort_values(
        ["min_cluster_rule", "mean_true_ARI"],
        ascending=[True, False]
    ).round(4)
)

# ------------------------------------------------------------
# B. PAIRED DECOMPOSITION FOR delta=0.05
# ------------------------------------------------------------

focus_policies = [
    "quality_only",
    "SA_delta_0.05"
]

focus = nonnull.loc[
    nonnull["policy"].isin(focus_policies)
].copy()

wide = focus.pivot(
    index=["dataset_id", "scenario"],
    columns=["policy", "min_cluster_rule"],
    values="true_ARI"
)

wide.columns = [
    f"{policy}_min{size}"
    for policy, size in wide.columns
]

wide = wide.reset_index()

required = [
    "quality_only_min1",
    "quality_only_min5",
    "quality_only_min8",
    "SA_delta_0.05_min1",
    "SA_delta_0.05_min5",
    "SA_delta_0.05_min8"
]

assert all(c in wide.columns for c in required)
assert len(wide) == 12
assert wide[required].notna().all().all()

wide["gain_filter_1_to_5"] = (
    wide["quality_only_min5"]
    - wide["quality_only_min1"]
)

wide["gain_filter_1_to_8"] = (
    wide["quality_only_min8"]
    - wide["quality_only_min1"]
)

for size in [1, 5, 8]:
    wide[f"gain_SA_given_min{size}"] = (
        wide[f"SA_delta_0.05_min{size}"]
        - wide[f"quality_only_min{size}"]
    )

effects = [
    "gain_filter_1_to_5",
    "gain_filter_1_to_8",
    "gain_SA_given_min1",
    "gain_SA_given_min5",
    "gain_SA_given_min8"
]

print("\nB. DECOMPOSITION OF PERFORMANCE CHANGES")

display(
    wide[effects].agg(
        ["mean", "median", "min", "max"]
    ).round(4)
)

print("\nC. DATASET-LEVEL PAIRED EFFECTS")

display(
    wide[
        ["dataset_id", "scenario"] + effects
    ].round(4)
)

# ------------------------------------------------------------
# D. HOW OFTEN DOES STABILITY CHANGE THE CHOICE?
# ------------------------------------------------------------

comparison_rows = []

for min_size in [1, 5, 8]:

    q = E.loc[
        (E["min_cluster_rule"] == min_size)
        & (E["policy"] == "quality_only"),
        [
            "dataset_id", "selected_model",
            "selected_K", "selected_min_cluster"
        ]
    ].rename(columns={
        "selected_model": "quality_model",
        "selected_K": "quality_K",
        "selected_min_cluster": "quality_min_size"
    })

    for delta in [0.02, 0.03, 0.05, 0.07, 0.10]:

        policy = f"SA_delta_{delta:.2f}"

        sa = E.loc[
            (E["min_cluster_rule"] == min_size)
            & (E["policy"] == policy),
            [
                "dataset_id", "selected_model",
                "selected_K", "selected_min_cluster",
                "true_ARI"
            ]
        ].rename(columns={
            "selected_model": "SA_model",
            "selected_K": "SA_K",
            "selected_min_cluster": "SA_min_size",
            "true_ARI": "SA_true_ARI"
        })

        paired = q.merge(
            sa,
            on="dataset_id",
            validate="one_to_one"
        )

        paired = paired.merge(
            E.loc[
                (E["min_cluster_rule"] == min_size)
                & (E["policy"] == "quality_only"),
                ["dataset_id", "true_ARI"]
            ].rename(columns={
                "true_ARI": "quality_true_ARI"
            }),
            on="dataset_id",
            validate="one_to_one"
        )

        paired = paired.loc[
            paired["dataset_id"].isin(
                nonnull["dataset_id"]
            )
        ].copy()

        assert len(paired) == 12

        comparison_rows.append({
            "min_cluster_rule": min_size,
            "delta": delta,
            "n_datasets": len(paired),
            "fraction_changed_model": float(
                (paired["quality_model"]
                 != paired["SA_model"]).mean()
            ),
            "fraction_changed_K": float(
                (paired["quality_K"]
                 != paired["SA_K"]).mean()
            ),
            "mean_delta_true_ARI": float(
                (
                    paired["SA_true_ARI"]
                    - paired["quality_true_ARI"]
                ).mean()
            ),
            "n_SA_better": int(
                (paired["SA_true_ARI"]
                 > paired["quality_true_ARI"] + 1e-12).sum()
            ),
            "n_SA_worse": int(
                (paired["SA_true_ARI"]
                 < paired["quality_true_ARI"] - 1e-12).sum()
            ),
            "n_equal": int(
                np.isclose(
                    paired["SA_true_ARI"],
                    paired["quality_true_ARI"],
                    rtol=0,
                    atol=1e-12
                ).sum()
            )
        })

comparison_audit = pd.DataFrame(comparison_rows)

assert (
    comparison_audit[
        ["n_SA_better", "n_SA_worse", "n_equal"]
    ].sum(axis=1) == 12
).all()

print("\nD. SA PERFORMANCE ACROSS ALL DELTAS")
display(
    comparison_audit.sort_values(
        ["min_cluster_rule", "delta"]
    ).round(4)
)

# ------------------------------------------------------------
# E. MODEL FAMILY CHOICE — QUALITY VS SA delta=0.05
# ------------------------------------------------------------

family = E.loc[
    (E["policy"].isin([
        "quality_only", "SA_delta_0.05"
    ]))
    & (E["K_true"] > 0)
].merge(
    G[["model_id", "method"]],
    left_on="selected_model",
    right_on="model_id",
    validate="many_to_one"
)

family_table = (
    family.groupby(
        ["min_cluster_rule", "policy", "method"]
    )
    .size()
    .rename("n_selections")
    .reset_index()
)

print("\nE. MODEL FAMILY SELECTION FREQUENCY")
display(family_table)

# ------------------------------------------------------------
# F. NULL DATASETS — STABILITY DOES NOT IMPLY CLUSTERS
# ------------------------------------------------------------

print("\nF. NULL-DATASET SELECTION DIAGNOSTIC")

display(
    null.loc[
        (null["min_cluster_rule"] == 5)
        & (null["policy"].isin([
            "quality_only", "SA_delta_0.05",
            "stability_only"
        ])),
        [
            "dataset_id", "policy",
            "selected_model", "selected_K",
            "selected_silhouette",
            "selected_mean_ARI"
        ]
    ].sort_values(
        ["dataset_id", "policy"]
    ).round(4)
)

# ------------------------------------------------------------
# G. SAVE
# ------------------------------------------------------------

summary_all.to_csv(
    STAB_SIM_OUT / "all_policy_sensitivity_audit.csv",
    index=False
)

wide.to_csv(
    STAB_SIM_OUT / "filter_vs_stability_decomposition.csv",
    index=False
)

comparison_audit.to_csv(
    STAB_SIM_OUT / "delta_incremental_value_audit.csv",
    index=False
)

family_table.to_csv(
    STAB_SIM_OUT / "selected_method_frequency.csv",
    index=False
)

print("\nCELL 14 COMPLETED")
print("No additional clustering fitted.")
print("No selection rule has been finalized.")


A. COMPLETE POLICY SENSITIVITY


,min_cluster_rule,policy,n_datasets,mean_true_ARI,median_true_ARI,mean_K_accuracy,mean_selected_silhouette,mean_selected_stability,mean_selected_min_size
4,1,SA_delta_0.10,12,0.4258,0.4704,0.5833,0.1607,0.8695,28.5833
3,1,SA_delta_0.07,12,0.4238,0.5335,0.5833,0.1766,0.8437,25.6667
6,1,stability_only,12,0.3624,0.3601,0.5833,0.1493,0.8835,26.5000
2,1,SA_delta_0.05,12,0.3010,0.1158,0.4167,0.1988,0.7276,16.5833
1,1,SA_delta_0.03,12,0.1944,0.0023,0.2500,0.2106,0.6594,11.2500
0,1,SA_delta_0.02,12,0.1497,0.0003,0.2500,0.2149,0.6241,8.5833
5,1,quality_only,12,0.0983,0.0003,0.3333,0.2173,0.5546,3.9167
7,5,SA_delta_0.02,12,0.4321,0.5356,0.4167,0.1779,0.8064,26.6667
12,5,quality_only,12,0.4279,0.4941,0.5000,0.1833,0.6969,23.3333
8,5,SA_delta_0.03,12,0.3969,0.3761,0.4167,0.1741,0.8141,27.3333



B. DECOMPOSITION OF PERFORMANCE CHANGES


,gain_filter_1_to_5,gain_filter_1_to_8,gain_SA_given_min1,gain_SA_given_min5,gain_SA_given_min8
mean,0.3296,0.3758,0.2026,-0.0789,-0.0956
median,0.2367,0.3780,0.0000,-0.0037,-0.0222
min,-0.0053,-0.0053,-0.0075,-0.4048,-0.4048
max,0.8466,0.8466,0.8466,0.0303,0.0303



C. DATASET-LEVEL PAIRED EFFECTS


...


E. MODEL FAMILY SELECTION FREQUENCY


,min_cluster_rule,policy,method,n_selections
0,1,SA_delta_0.05,Hierarchical_Average,5
1,1,SA_delta_0.05,Hierarchical_Complete,1
2,1,SA_delta_0.05,Spectral_Early,3
3,1,SA_delta_0.05,Spectral_MultiView,3
4,1,quality_only,Hierarchical_Average,11
5,1,quality_only,Spectral_Early,1
6,5,SA_delta_0.05,Hierarchical_Average,1
7,5,SA_delta_0.05,Spectral_Early,4
8,5,SA_delta_0.05,Spectral_MultiView,7
9,5,quality_only,Hierarchical_Average,2



F. NULL-DATASET SELECTION DIAGNOSTIC


,dataset_id,policy,selected_model,selected_K,selected_silhouette,selected_mean_ARI
221,null_gaussian_r001,SA_delta_0.05,Early_K2_G16_S7,2,0.1321,0.8231
217,null_gaussian_r001,quality_only,Hierarchical_Complete_K2,2,0.1478,0.2715
218,null_gaussian_r001,stability_only,Early_K2_G16_S7,2,0.1321,0.8231
242,null_gaussian_r002,SA_delta_0.05,Multi_K3_G16_S7_climate50,3,0.0867,0.7989
238,null_gaussian_r002,quality_only,Early_K2_G16_S7,2,0.1280,0.6449
239,null_gaussian_r002,stability_only,Multi_K3_G16_S7_climate50,3,0.0867,0.7989



CELL 14 COMPLETED
No additional clustering fitted.
No selection rule has been finalized.


# pilot kalibrasi null

In [ ]:

# ============================================================
# CELL 15 — NULL-CALIBRATED CLUSTERABILITY PILOT
#
# Three diagnostic datasets, 49 Gaussian reference draws each
# 70 candidate configurations evaluated per draw
# Expected new clustering fits: 3 * 49 * 70 = 10,290
#
# No ground truth accessed.
# No policy optimization.
# ============================================================

from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import hashlib
import json
import os

# Existing definitions required from Notebook 06:
assert callable(make_sim_context)
assert callable(fit_sim_candidate)
assert len(GRID_SIM) == 70
assert len(MANIFEST) == 70

MODEL_SEED = 20261018

NULL_SEED = 20261024
B_NULL_PILOT = 49

PILOT_IDS = [
    "aligned_K3_r001",
    "conflict_K3_r001",
    "null_gaussian_r001"
]

MIN_SIZES = [1, 5, 8]

NULL_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "clusterability_null_pilot_v1"
)

BOOT_DIR = NULL_OUT / "null_replicates"
BOOT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


NULL_PROTOCOL = {
    "stage": "clusterability_null_pilot",
    "status": "exploratory_not_confirmatory",
    "dataset_ids": PILOT_IDS,
    "null_family": "multivariate_normal",
    "null_parameter_estimation": (
        "Sample mean and sample covariance of "
        "the observed dataset"
    ),
    "bootstrap_repetitions": B_NULL_PILOT,
    "bootstrap_seed": NULL_SEED,
    "model_seed": MODEL_SEED,
    "candidate_count": len(GRID_SIM),
    "minimum_cluster_sizes": MIN_SIZES,
    "test_statistic": (
        "Maximum full-data R0 silhouette among "
        "all size-eligible candidate partitions"
    ),
    "candidate_grid_sha256": sha256_file(
        RECOVERY_OUT / "candidate_grid.csv"
    ),
    "generator_manifest_sha256": sha256_file(
        SIM_V2 / "simulation_dataset_manifest.csv"
    )
}

protocol_path = NULL_OUT / "protocol.json"

if protocol_path.exists():
    old_protocol = json.loads(
        protocol_path.read_text(encoding="utf-8")
    )
    assert old_protocol == NULL_PROTOCOL
else:
    protocol_path.write_text(
        json.dumps(NULL_PROTOCOL, indent=2),
        encoding="utf-8"
    )


def maximum_quality(quality_table, min_size):
    """
    Search-adjusted statistic over all eligible candidates.

    Returns -inf if no candidate is eligible.
    """

    eligible = quality_table.loc[
        quality_table["smallest_cluster"] >= min_size
    ]

    if eligible.empty:
        return -np.inf

    return float(
        eligible["silhouette_R0"].max()
    )


candidate_plan = GRID_SIM.to_dict("records")

summary_records = []

for dataset_index, dataset_id in enumerate(PILOT_IDS):

    metadata = MANIFEST.loc[
        MANIFEST["dataset_id"] == dataset_id
    ]

    assert len(metadata) == 1
    metadata = metadata.iloc[0]

    input_path = DATA_V2 / f"{dataset_id}.csv"

    assert sha256_file(
        input_path
    ) == metadata["data_sha256"]

    data = pd.read_csv(input_path)

    X_obs = data[FEATURES].to_numpy(
        dtype=float
    )

    assert X_obs.shape == (85, 9)

    # Estimate the working Gaussian null.
    mu_hat = X_obs.mean(axis=0)
    Sigma_hat = np.cov(
        X_obs, rowvar=False, ddof=1
    )

    eigenvalues = np.linalg.eigvalsh(
        Sigma_hat
    )

    if eigenvalues.min() <= 1e-10:
        raise ValueError(
            f"Non-positive covariance: {dataset_id}"
        )

    chol = np.linalg.cholesky(Sigma_hat)

    observed_quality = pd.read_csv(
        QUALITY_DIR / f"{dataset_id}.csv"
    )

    assert len(observed_quality) == 70
    assert observed_quality["model_id"].is_unique
    assert set(observed_quality["model_id"]) == set(
        GRID_SIM["model_id"]
    )

    observed_T = {
        m: maximum_quality(observed_quality, m)
        for m in MIN_SIZES
    }

    assert all(
        np.isfinite(x) for x in observed_T.values()
    )

    dataset_boot_dir = BOOT_DIR / dataset_id
    dataset_boot_dir.mkdir(exist_ok=True)

    # --------------------------------------------------------
    # Parametric Gaussian reference replicates
    # --------------------------------------------------------

    for b in range(1, B_NULL_PILOT + 1):

        boot_file = (
            dataset_boot_dir / f"rep_{b:03d}.csv"
        )

        if boot_file.exists():

            previous = pd.read_csv(boot_file)

            assert len(previous) == 70
            assert previous["model_id"].is_unique
            assert set(previous["model_id"]) == set(
                GRID_SIM["model_id"]
            )
            assert previous[
                "silhouette_R0"
            ].notna().all()

            continue

        rng = np.random.default_rng(
            np.random.SeedSequence([
                NULL_SEED, dataset_index, b
            ])
        )

        # Gaussian draws with the fitted covariance.
        X_null = (
            rng.standard_normal(size=X_obs.shape)
            @ chol.T
            + mu_hat[None, :]
        )

        ctx = make_sim_context(X_null)

        rows = []

        for model in candidate_plan:

            fitted, cluster_sizes, _ = (
                fit_sim_candidate(model, ctx)
            )

            from sklearn.metrics import silhouette_score

            quality = silhouette_score(
                ctx["D"],
                fitted,
                metric="precomputed"
            )

            rows.append({
                "model_id": model["model_id"],
                "K": int(model["K"]),
                "silhouette_R0": float(quality),
                "smallest_cluster": int(
                    cluster_sizes.min()
                )
            })

        results = pd.DataFrame(rows)

        assert len(results) == 70
        assert results["model_id"].is_unique
        assert np.isfinite(
            results["silhouette_R0"]
        ).all()

        temp = boot_file.with_suffix(".tmp")
        results.to_csv(temp, index=False)
        os.replace(temp, boot_file)

    # --------------------------------------------------------
    # Calculate search-adjusted reference statistics
    # --------------------------------------------------------

    null_tables = [
        pd.read_csv(
            dataset_boot_dir / f"rep_{b:03d}.csv"
        )
        for b in range(1, B_NULL_PILOT + 1)
    ]

    assert len(null_tables) == B_NULL_PILOT

    for m in MIN_SIZES:

        T_null = np.array([
            maximum_quality(table, m)
            for table in null_tables
        ])

        T_obs = observed_T[m]

        # If no candidate is size-eligible, T=-inf
        # and cannot exceed a finite observed T.
        exceedances = int(
            np.sum(T_null >= T_obs)
        )

        p_mc = (
            1 + exceedances
        ) / (
            B_NULL_PILOT + 1
        )

        finite_null = T_null[np.isfinite(T_null)]

        summary_records.append({
            "dataset_id": dataset_id,
            "min_cluster_size": m,
            "B_null": B_NULL_PILOT,
            "T_observed": T_obs,
            "n_feasible_null": len(finite_null),
            "mean_T_null": (
                finite_null.mean()
                if len(finite_null) else np.nan
            ),
            "p95_T_null": (
                np.quantile(finite_null, 0.95)
                if len(finite_null) else np.nan
            ),
            "exceedances": exceedances,
            "p_mc_exploratory": p_mc
        })

    print(
        f"Completed null calibration: {dataset_id} "
        f"({B_NULL_PILOT} reference datasets)"
    )


null_pilot_summary = pd.DataFrame(
    summary_records
)

assert len(null_pilot_summary) == (
    len(PILOT_IDS) * len(MIN_SIZES)
)

summary_file = NULL_OUT / "null_calibration_pilot.csv"

null_pilot_summary.to_csv(
    summary_file, index=False
)

print("\nA. NULL-CALIBRATED PILOT RESULTS")
display(
    null_pilot_summary.round(4)
)

print("\nB. COMPUTATIONAL COVERAGE")
print(
    "Expected reference fittings:",
    len(PILOT_IDS) * B_NULL_PILOT * len(GRID_SIM)
)
print(
    "Completed reference files:",
    sum(
        len(list(
            (BOOT_DIR / d).glob("rep_*.csv")
        ))
        for d in PILOT_IDS
    )
)
print("\nNULL PILOT COMPLETED")
print("Exploratory statistics only.")
print("Output:", NULL_OUT)


Completed null calibration: aligned_K3_r001 (49 reference datasets)
Completed null calibration: conflict_K3_r001 (49 reference datasets)
Completed null calibration: null_gaussian_r001 (49 reference datasets)

A. NULL-CALIBRATED PILOT RESULTS


,dataset_id,min_cluster_size,B_null,T_observed,n_feasible_null,mean_T_null,p95_T_null,exceedances,p_mc_exploratory
0,aligned_K3_r001,1,49,0.2357,49,0.2696,0.3768,31,0.64
1,aligned_K3_r001,5,49,0.1674,49,0.1711,0.2171,24,0.50
2,aligned_K3_r001,8,49,0.1674,49,0.1665,0.2028,21,0.44
3,conflict_K3_r001,1,49,0.2221,49,0.2411,0.3366,25,0.52
4,conflict_K3_r001,5,49,0.1340,49,0.1530,0.2144,40,0.82
5,conflict_K3_r001,8,49,0.1340,49,0.1485,0.1802,40,0.82
6,null_gaussian_r001,1,49,0.3098,49,0.2329,0.3184,4,0.10
7,null_gaussian_r001,5,49,0.1478,49,0.1541,0.2066,23,0.48
8,null_gaussian_r001,8,49,0.1478,49,0.1490,0.1986,20,0.42



B. COMPUTATIONAL COVERAGE
Expected reference fittings: 10290
Completed reference files: 147

NULL PILOT COMPLETED
Exploratory statistics only.
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/clusterability_null_pilot_v1


# Audit dekomposisi keragaman kelas laten

In [ ]:

# ============================================================
# CELL 16 — WHY DOES THE GAUSSIAN NULL OVERLAP?
#
# ANOVA scatter decomposition:
# Total scatter = Within-class + Between-class scatter
#
# Ground truth is used ONLY for synthetic generator
# diagnostics, never for unsupervised model selection.
# ============================================================

from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import hashlib

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

SIM_V2 = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v2"
)

DIAG_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "null_mechanism_audit_v1"
)

DIAG_OUT.mkdir(parents=True, exist_ok=True)

DATA_V2 = SIM_V2 / "datasets"
TRUTH_V2 = SIM_V2 / "ground_truth"

M = pd.read_csv(
    SIM_V2 / "simulation_dataset_manifest.csv"
)

assert len(M) == 70
assert M["dataset_id"].is_unique


def scatter_decomposition(Z, labels):
    """
    Exact empirical TSS = WSS + BSS decomposition.

    BSS/TSS measures the fraction of observed
    standardized variance associated with a
    specified known class partition.

    Not a measure of clustering accuracy.
    """

    Z = np.asarray(Z, dtype=float)
    labels = np.asarray(labels)

    grand_mean = Z.mean(axis=0)

    total = float(np.sum(
        (Z - grand_mean) ** 2
    ))

    between = 0.0
    within = 0.0

    for c in np.unique(labels):

        group = Z[labels == c]

        center = group.mean(axis=0)

        between += len(group) * float(
            np.sum((center - grand_mean) ** 2)
        )

        within += float(
            np.sum((group - center) ** 2)
        )

    assert total > 0

    assert np.isclose(
        total,
        within + between,
        rtol=1e-10,
        atol=1e-10
    )

    return {
        "total": total,
        "within": within,
        "between": between,
        "between_fraction": between / total
    }


records = []

for r in M.itertuples(index=False):

    data_file = DATA_V2 / f"{r.dataset_id}.csv"
    truth_file = TRUTH_V2 / f"{r.dataset_id}.csv"

    assert hashlib.sha256(
        data_file.read_bytes()
    ).hexdigest() == r.data_sha256

    assert hashlib.sha256(
        truth_file.read_bytes()
    ).hexdigest() == r.truth_sha256

    data = pd.read_csv(data_file)
    labels = pd.read_csv(truth_file)

    assert np.array_equal(
        data["observation_id"].to_numpy(),
        labels["observation_id"].to_numpy()
    )

    if r.K_true == -1:
        continue

    X = data[
        [
            "F1_ikp_ketersediaan",
            "F2_ikp_keterjangkauan",
            "F3_ikp_pemanfaatan",
            "C1_anomali_hujan_mm",
            "C2_cdd_hari",
            "C3_hari_hujan_ekstrem",
            "A1_persen_tanpa_angkutan",
            "A2_persen_jalan_tanah_lainnya",
            "A3_persen_tanpa_pertokoan_pasar"
        ]
    ].to_numpy(dtype=float)

    assert X.shape == (85, 9)

    # Same feature-wise standardization as R0.
    Z = (
        (X - X.mean(axis=0))
        / X.std(axis=0, ddof=1)
    )

    y = labels["true_cluster"].to_numpy(
        dtype=int
    )

    nuisance = labels[
        "nuisance_climate_group"
    ].to_numpy(dtype=int)

    assert len(np.unique(y)) == r.K_true

    all_result = scatter_decomposition(Z, y)

    food_result = scatter_decomposition(
        Z[:, 0:3], y
    )

    climate_result = scatter_decomposition(
        Z[:, 3:6], y
    )

    access_result = scatter_decomposition(
        Z[:, 6:9], y
    )

    # Used only for explicit climate conflict scenario.
    climate_nuisance_fraction = np.nan

    if r.scenario == "conflict_K3":

        assert len(np.unique(nuisance)) == 3

        climate_nuisance_fraction = (
            scatter_decomposition(
                Z[:, 3:6], nuisance
            )["between_fraction"]
        )

    records.append({
        "dataset_id": r.dataset_id,
        "scenario": r.scenario,
        "K_true": r.K_true,
        "between_fraction_all": (
            all_result["between_fraction"]
        ),
        "between_fraction_food": (
            food_result["between_fraction"]
        ),
        "between_fraction_climate": (
            climate_result["between_fraction"]
        ),
        "between_fraction_access": (
            access_result["between_fraction"]
        ),
        "between_fraction_climate_nuisance": (
            climate_nuisance_fraction
        )
    })


D = pd.DataFrame(records)

assert len(D) == 60

S = (
    D.groupby(
        "scenario", as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "count"),
        mean_between_all=(
            "between_fraction_all", "mean"
        ),
        mean_between_food=(
            "between_fraction_food", "mean"
        ),
        mean_between_climate=(
            "between_fraction_climate", "mean"
        ),
        mean_between_access=(
            "between_fraction_access", "mean"
        ),
        mean_climate_nuisance=(
            "between_fraction_climate_nuisance",
            "mean"
        )
    )
)

print("A. BETWEEN-CLASS SCATTER FRACTIONS")
display(S.round(4))

print("\nB. CONFLICTING CLIMATE VIEW")

conflict = D.loc[
    D["scenario"] == "conflict_K3"
]

display(
    conflict[
        [
            "dataset_id",
            "between_fraction_climate",
            "between_fraction_climate_nuisance"
        ]
    ].round(4)
)

print(
    "\nNuisance explains more Climate scatter:",
    int(
        (
            conflict["between_fraction_climate_nuisance"]
            > conflict["between_fraction_climate"]
        ).sum()
    ),
    "/10 datasets"
)

D.to_csv(
    DIAG_OUT / "scatter_decomposition_all.csv",
    index=False
)

S.to_csv(
    DIAG_OUT / "scatter_decomposition_summary.csv",
    index=False
)

print("\nSCATTER DECOMPOSITION AUDIT COMPLETED")
print("No clustering models were fitted.")
print("Output:", DIAG_OUT)


A. BETWEEN-CLASS SCATTER FRACTIONS


,scenario,n_datasets,mean_between_all,mean_between_food,mean_between_climate,mean_between_access,mean_climate_nuisance
0,aligned_K2,10,0.2362,0.2442,0.2340,0.2306,NaN
1,aligned_K3,10,0.2951,0.2951,0.2994,0.2909,NaN
2,aligned_K4,10,0.3589,0.3613,0.3581,0.3572,NaN
3,conflict_K3,10,0.2010,0.2870,0.0269,0.2892,0.3591
4,imbalanced_K3,10,0.2717,0.2560,0.2863,0.2728,NaN
5,weak_K3,10,0.0737,0.0814,0.0713,0.0684,NaN



B. CONFLICTING CLIMATE VIEW


,dataset_id,between_fraction_climate,between_fraction_climate_nuisance
50,conflict_K3_r001,0.0358,0.2866
51,conflict_K3_r002,0.0340,0.4301
52,conflict_K3_r003,0.0188,0.3374
53,conflict_K3_r004,0.0210,0.3887
54,conflict_K3_r005,0.0168,0.3923
55,conflict_K3_r006,0.0643,0.4026
56,conflict_K3_r007,0.0070,0.3252
57,conflict_K3_r008,0.0270,0.3258
58,conflict_K3_r009,0.0264,0.3023
59,conflict_K3_r010,0.0184,0.3997



Nuisance explains more Climate scatter: 10 /10 datasets

SCATTER DECOMPOSITION AUDIT COMPLETED
No clustering models were fitted.
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/08_monte_carlo_simulation/null_mechanism_audit_v1


# kunci protokol evaluasi independen

In [ ]:

# ============================================================
# CELL 17 — FREEZE OUT-OF-DEVELOPMENT MC PROTOCOL
#
# No model fitting.
# No ground-truth evaluation.
#
# This is a prospective evaluation protocol AFTER
# exploratory development, not preregistration
# before the exploratory analyses.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import json
import hashlib

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

SIM_V2 = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v2"
)

RECOVERY_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "recovery_pilot_v1"
)

MC_OUT = (
    ROOT / "03_outputs"
    / "09_independent_monte_carlo"
    / "simplex_v2_frozen_v1"
)

MC_OUT.mkdir(parents=True, exist_ok=True)


def sha256_locked(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# ------------------------------------------------------------
# 1. VERIFY THE ORIGINAL GENERATOR
# ------------------------------------------------------------

generator_protocol_file = (
    SIM_V2 / "generator_protocol.json"
)

grid_file = (
    RECOVERY_OUT / "candidate_grid.csv"
)

assert generator_protocol_file.exists()
assert grid_file.exists()

generator_protocol = json.loads(
    generator_protocol_file.read_text(
        encoding="utf-8"
    )
)

assert (
    generator_protocol["generator"]
    == "regular_simplex_multiview_v2"
)

assert generator_protocol["n"] == 85
assert len(generator_protocol["features"]) == 9

assert (
    generator_protocol["scenarios"]
    == SCENARIOS_V2
)

assert (
    generator_protocol["noise"]
    == NOISE_V2
)

assert (
    generator_protocol["rotation_angles_xyz_degrees"]
    == ANGLES_V2
)

assert generator_protocol["features"] == FEATURES_V2

assert callable(generate_multiview_v2)
assert callable(simplex_centroids)

candidate_grid = pd.read_csv(grid_file)

assert len(candidate_grid) == 70
assert candidate_grid["model_id"].is_unique
assert set(candidate_grid["K"]) == {2, 3, 4, 5, 6}

# ------------------------------------------------------------
# 2. LOCK THE NEW SIMULATION DESIGN
# ------------------------------------------------------------

N_PER_SCENARIO_MC = 60
B_MC = 20
N_SUBSET_MC = 68

GENERATOR_SEED_WORDS_MC = [20261019, 1001]
SUBSAMPLE_SEED_WORDS_MC = [20261019, 1002]
ORDER_SEED_WORDS_MC = [20261019, 1003]

MODEL_SEED_MC = 20261018

PRIMARY_MIN_CLUSTER = 5
PRIMARY_DELTA = 0.05

assert PRIMARY_MIN_CLUSTER in [1, 5, 8]
assert 0 < PRIMARY_DELTA < 1

MC_PROTOCOL = {
    "experiment": "independent_monte_carlo_v1",
    "development_complete_before_test": True,
    "generator": "regular_simplex_multiview_v2",
    "generator_protocol_sha256": sha256_locked(
        generator_protocol_file
    ),
    "candidate_grid_sha256": sha256_locked(
        grid_file
    ),
    "scenarios": list(SCENARIOS_V2),
    "n_per_scenario": N_PER_SCENARIO_MC,
    "n_datasets": (
        len(SCENARIOS_V2) * N_PER_SCENARIO_MC
    ),
    "n_observations": 85,
    "n_features": 9,
    "n_candidates": 70,
    "generator_seed_words": GENERATOR_SEED_WORDS_MC,
    "subsample_seed_words": SUBSAMPLE_SEED_WORDS_MC,
    "dataset_order_seed_words": ORDER_SEED_WORDS_MC,
    "model_seed": MODEL_SEED_MC,
    "n_subsamples": B_MC,
    "subsample_size": N_SUBSET_MC,
    "subsampling": (
        "simple_random_without_replacement"
    ),
    "stratification": "none",
    "preprocessing": "R0_refit_each_subsample",
    "primary_policy": {
        "baseline": "quality_only",
        "comparison": "quality_constrained_stability",
        "minimum_cluster_size": PRIMARY_MIN_CLUSTER,
        "silhouette_tolerance": PRIMARY_DELTA,
        "stability_measure": "mean_subsample_ARI",
        "outcome": "true_adjusted_rand_index",
        "estimand": "paired_mean_difference"
    },
    "secondary_analyses": {
        "minimum_cluster_sizes": [1, 5, 8],
        "silhouette_tolerances": [
            0.02, 0.03, 0.05, 0.07, 0.10
        ],
        "outcomes": [
            "correct_K",
            "selected_silhouette",
            "selected_mean_ARI",
            "selected_minimum_cluster_size",
            "clusterwise_jaccard"
        ]
    },
    "null_scenario": {
        "purpose": "descriptive_false_structure_diagnostic",
        "ground_truth_ARI": "not_defined",
        "formal_cluster_rejection": False,
        "subsampling_required": False
    },
    "inferential_unit": "independent_simulated_dataset",
    "uncertainty": "dataset_level_paired_bootstrap",
    "no_policy_changes_after_test_inspection": True,
    "status": "FROZEN_BEFORE_NEW_TEST_DATA"
}

PROTOCOL_FILE_MC = MC_OUT / "protocol.json"

if PROTOCOL_FILE_MC.exists():

    existing = json.loads(
        PROTOCOL_FILE_MC.read_text(
            encoding="utf-8"
        )
    )

    if existing != MC_PROTOCOL:
        raise RuntimeError(
            "Frozen Monte Carlo protocol mismatch. "
            "Do not overwrite the existing experiment."
        )

else:

    PROTOCOL_FILE_MC.write_text(
        json.dumps(
            MC_PROTOCOL,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

print("INDEPENDENT MONTE CARLO PROTOCOL FROZEN")
print("Total datasets:", MC_PROTOCOL["n_datasets"])
print("Per scenario:", N_PER_SCENARIO_MC)
print("Candidate configurations:", len(candidate_grid))
print("Subsamples per non-null dataset:", B_MC)
print("Primary minimum cluster:", PRIMARY_MIN_CLUSTER)
print("Primary delta:", PRIMARY_DELTA)
print("Output:", MC_OUT)


INDEPENDENT MONTE CARLO PROTOCOL FROZEN
Total datasets: 420
Per scenario: 60
Candidate configurations: 70
Subsamples per non-null dataset: 20
Primary minimum cluster: 5
Primary delta: 0.05
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1


# Bagian Baru

# Bangkitkan dataset evaluasi baru secara terpisah
Pada tahap ini kita membuat 420 dataset baru menggunakan fungsi generator V2 yang sudah diaudit. Dataset akan diberi identitas D0001, D0002, dan seterusnya, dengan urutan skenario diacak.

In [ ]:

# ============================================================
# CELL 18 — GENERATE NEW BLINDED MONTE CARLO DATASETS
#
# The model-fitting workflow should use ONLY:
#   MC_OUT / "blind_inputs"
#
# Ground truth and scenario metadata are stored separately:
#   MC_OUT / "evaluation_only"
#
# No clustering is performed in this cell.
# ============================================================

import os
import numpy as np
import pandas as pd
import hashlib

BLIND_DIR = MC_OUT / "blind_inputs"
EVAL_DIR = MC_OUT / "evaluation_only"

BLIND_DATA_DIR = BLIND_DIR / "datasets"
EVAL_TRUTH_DIR = EVAL_DIR / "ground_truth"

BLIND_DATA_DIR.mkdir(
    parents=True, exist_ok=True
)

EVAL_TRUTH_DIR.mkdir(
    parents=True, exist_ok=True
)


def write_frozen_csv(path, frame):
    """
    Save once; on rerun require identical CSV bytes.
    This prevents silent replacement of test datasets.
    """

    path = Path(path)

    payload = frame.to_csv(
        index=False,
        float_format="%.17g"
    ).encode("utf-8")

    if path.exists():

        if path.read_bytes() != payload:
            raise RuntimeError(
                f"Frozen file changed: {path}"
            )

    else:

        temp = path.with_suffix(".tmp")
        temp.write_bytes(payload)

        os.replace(temp, path)


# ------------------------------------------------------------
# 1. CREATE A REPRODUCIBLE RANDOM DATASET ORDER
# ------------------------------------------------------------

scenario_names = list(SCENARIOS_V2)

tasks = [
    (scenario_idx, scenario, rep)
    for scenario_idx, scenario in enumerate(scenario_names)
    for rep in range(1, N_PER_SCENARIO_MC + 1)
]

order_rng = np.random.default_rng(
    np.random.SeedSequence(
        ORDER_SEED_WORDS_MC
    )
)

order = order_rng.permutation(len(tasks))

tasks_ordered = [
    tasks[int(i)]
    for i in order
]

assert len(tasks_ordered) == (
    len(SCENARIOS_V2) * N_PER_SCENARIO_MC
)

# ------------------------------------------------------------
# 2. GENERATE EACH DATASET FROM ITS OWN RNG STREAM
# ------------------------------------------------------------

public_records = []
private_records = []

for serial, (
    scenario_idx, scenario, rep
) in enumerate(tasks_ordered, start=1):

    dataset_id = f"D{serial:04d}"

    child_seed = np.random.SeedSequence(
        GENERATOR_SEED_WORDS_MC
        + [scenario_idx, rep]
    )

    rng = np.random.default_rng(
        child_seed
    )

    X, y_true, nuisance = generate_multiview_v2(
        scenario, rng
    )

    assert X.shape == (85, 9)
    assert np.isfinite(X).all()

    K_true = SCENARIOS_V2[scenario]["K"]

    if K_true is None:
        assert np.all(y_true == -1)
    else:
        assert len(np.unique(y_true)) == K_true

    # --------------------------------------------------------
    # PUBLIC / BLINDED FEATURES
    # --------------------------------------------------------

    features_frame = pd.DataFrame(
        X,
        columns=FEATURES_V2
    )

    features_frame.insert(
        0,
        "observation_id",
        np.arange(85, dtype=int)
    )

    # --------------------------------------------------------
    # PRIVATE / EVALUATION-ONLY LABELS
    # --------------------------------------------------------

    truth_frame = pd.DataFrame({
        "observation_id": np.arange(
            85, dtype=int
        ),
        "true_cluster": y_true,
        "nuisance_climate_group": nuisance
    })

    feature_file = (
        BLIND_DATA_DIR / f"{dataset_id}.csv"
    )

    truth_file = (
        EVAL_TRUTH_DIR / f"{dataset_id}.csv"
    )

    write_frozen_csv(
        feature_file,
        features_frame
    )

    write_frozen_csv(
        truth_file,
        truth_frame
    )

    feature_hash = sha256_locked(
        feature_file
    )

    truth_hash = sha256_locked(
        truth_file
    )

    public_records.append({
        "dataset_id": dataset_id,
        "n": 85,
        "p": 9,
        "data_sha256": feature_hash
    })

    private_records.append({
        "dataset_id": dataset_id,
        "scenario": scenario,
        "generator_replicate": rep,
        "K_true": (
            -1 if K_true is None else K_true
        ),
        "data_sha256": feature_hash,
        "truth_sha256": truth_hash,
        "child_seed_state": json.dumps(
            child_seed.generate_state(4).tolist()
        )
    })


# ------------------------------------------------------------
# 3. FREEZE THE PUBLIC AND PRIVATE MANIFESTS
# ------------------------------------------------------------

PUBLIC_MANIFEST_MC = pd.DataFrame(
    public_records
)

PRIVATE_MANIFEST_MC = pd.DataFrame(
    private_records
)

assert len(PUBLIC_MANIFEST_MC) == 420
assert len(PRIVATE_MANIFEST_MC) == 420

assert PUBLIC_MANIFEST_MC[
    "dataset_id"
].is_unique

assert PRIVATE_MANIFEST_MC[
    "dataset_id"
].is_unique

public_manifest_file = (
    BLIND_DIR / "dataset_manifest.csv"
)

private_manifest_file = (
    EVAL_DIR / "generator_truth_manifest.csv"
)

write_frozen_csv(
    public_manifest_file,
    PUBLIC_MANIFEST_MC
)

write_frozen_csv(
    private_manifest_file,
    PRIVATE_MANIFEST_MC
)

# ------------------------------------------------------------
# 4. READ-ONLY INTEGRITY AUDIT
# ------------------------------------------------------------

assert len(list(
    BLIND_DATA_DIR.glob("D*.csv")
)) == 420

assert len(list(
    EVAL_TRUTH_DIR.glob("D*.csv")
)) == 420

assert PRIVATE_MANIFEST_MC.groupby(
    "scenario"
).size().eq(
    N_PER_SCENARIO_MC
).all()

for r in PUBLIC_MANIFEST_MC.itertuples(index=False):

    path = BLIND_DATA_DIR / f"{r.dataset_id}.csv"

    assert sha256_locked(path) == r.data_sha256

print("INDEPENDENT DATA GENERATION COMPLETED")
print("Public feature datasets:", len(PUBLIC_MANIFEST_MC))
print("Private truth datasets:", len(PRIVATE_MANIFEST_MC))
print("Dataset order randomized.")
print("No clustering models fitted.")
print()
print("Blind inputs:", BLIND_DIR)
print("Evaluation metadata:", EVAL_DIR)

print("\nGENERATOR SCENARIO COVERAGE — AUDIT ONLY")
display(
    PRIVATE_MANIFEST_MC.groupby(
        ["scenario", "K_true"],
        as_index=False
    ).size().rename(
        columns={"size": "n_datasets"}
    )
)


INDEPENDENT DATA GENERATION COMPLETED
Public feature datasets: 420
Private truth datasets: 420
Dataset order randomized.
No clustering models fitted.

Blind inputs: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/blind_inputs
Evaluation metadata: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/evaluation_only

GENERATOR SCENARIO COVERAGE — AUDIT ONLY


,scenario,K_true,n_datasets
0,aligned_K2,2,60
1,aligned_K3,3,60
2,aligned_K4,4,60
3,conflict_K3,3,60
4,imbalanced_K3,3,60
5,null_gaussian,-1,60
6,weak_K3,3,60


# Audit reproduksibilitas sebelum fitting independen

In [ ]:

# ============================================================
# CELL 19 — FROZEN FITTING PRE-FLIGHT AUDIT
#
# Verifies:
# 1. Frozen Monte Carlo protocol and blind inputs
# 2. Original candidate grid and library versions
# 3. 70/70 pilot reference partitions and silhouettes
#
# No independent-test ground truth is accessed.
# ============================================================

from pathlib import Path
from importlib.metadata import version
from sklearn.metrics import adjusted_rand_score, silhouette_score
from IPython.display import display

import numpy as np
import pandas as pd
import sklearn
import scipy
import kmedoids
import hashlib
import json
import os

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

MC_OUT = (
    ROOT / "03_outputs"
    / "09_independent_monte_carlo"
    / "simplex_v2_frozen_v1"
)

RECOVERY_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "recovery_pilot_v1"
)

SIM_V2 = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v2"
)

BLIND_DIR = MC_OUT / "blind_inputs"
BLIND_DATA_DIR = BLIND_DIR / "datasets"

FULL_OUT = MC_OUT / "full_data_fits"
FULL_OUT.mkdir(parents=True, exist_ok=True)

def sha256_checked(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

frozen = json.loads(
    (MC_OUT / "protocol.json").read_text(
        encoding="utf-8"
    )
)

recovery_protocol = json.loads(
    (RECOVERY_OUT / "protocol.json").read_text(
        encoding="utf-8"
    )
)

PUBLIC_MC = pd.read_csv(
    BLIND_DIR / "dataset_manifest.csv"
)

GRID_MC = pd.read_csv(
    RECOVERY_OUT / "candidate_grid.csv"
)

assert frozen["n_datasets"] == 420
assert frozen["n_candidates"] == 70
assert frozen["model_seed"] == 20261018
assert frozen["primary_policy"]["minimum_cluster_size"] == 5
assert frozen["primary_policy"]["silhouette_tolerance"] == 0.05

assert len(PUBLIC_MC) == 420
assert PUBLIC_MC["dataset_id"].is_unique
assert len(GRID_MC) == 70
assert GRID_MC["model_id"].is_unique

assert sha256_checked(
    RECOVERY_OUT / "candidate_grid.csv"
) == frozen["candidate_grid_sha256"]

assert sha256_checked(
    SIM_V2 / "generator_protocol.json"
) == frozen["generator_protocol_sha256"]

assert callable(make_sim_context)
assert callable(fit_sim_candidate)

MODEL_SEED = frozen["model_seed"]

assert set(KG_GRID) == {8, 16}
assert KS_FIXED == 7
assert VIEW_NAMES == [
    "Food", "Climate", "Accessibility"
]

assert WEIGHTS == {
    "equal": (1/3, 1/3, 1/3),
    "food50": (0.50, 0.25, 0.25),
    "climate50": (0.25, 0.50, 0.25),
    "access50": (0.25, 0.25, 0.50)
}

assert FEATURES == FEATURES_V2

current_versions = {
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "kmedoids": version("kmedoids")
}

assert current_versions == recovery_protocol["versions"], (
    "Library versions differ from recovery pilot."
)

# Verify public input hashes without loading truth.
for r in PUBLIC_MC.itertuples(index=False):
    path = BLIND_DATA_DIR / f"{r.dataset_id}.csv"
    assert path.exists()
    assert sha256_checked(path) == r.data_sha256

# ------------------------------------------------------------
# Reproduce original pilot partition references
# ------------------------------------------------------------

TEST_PILOT_ID = "aligned_K3_r001"

pilot_data = pd.read_csv(
    SIM_V2 / "datasets" / f"{TEST_PILOT_ID}.csv"
)

pilot_ref = pd.read_csv(
    RECOVERY_OUT / "labels" / f"{TEST_PILOT_ID}.csv"
)

pilot_quality = pd.read_csv(
    RECOVERY_OUT / "quality" / f"{TEST_PILOT_ID}.csv"
).set_index("model_id")

assert np.array_equal(
    pilot_data["observation_id"],
    pilot_ref["observation_id"]
)

X_pilot = pilot_data[FEATURES].to_numpy(
    dtype=float
)

ctx_pilot = make_sim_context(X_pilot)

preflight = []

for row in GRID_MC.to_dict("records"):

    model_id = row["model_id"]

    y_fit, _, _ = fit_sim_candidate(
        row, ctx_pilot
    )

    y_ref = pilot_ref[
        model_id
    ].to_numpy(dtype=int)

    ari = adjusted_rand_score(y_ref, y_fit)

    silhouette = silhouette_score(
        ctx_pilot["D"],
        y_fit,
        metric="precomputed"
    )

    reference_sil = float(
        pilot_quality.loc[
            model_id, "silhouette_R0"
        ]
    )

    preflight.append({
        "model_id": model_id,
        "method": row["method"],
        "ARI": float(ari),
        "silhouette_error": float(
            abs(silhouette - reference_sil)
        )
    })

PREFLIGHT = pd.DataFrame(preflight)

assert len(PREFLIGHT) == 70
assert np.allclose(
    PREFLIGHT["ARI"], 1.0,
    rtol=0, atol=1e-12
), "Pilot partition reproduction failed."

assert (
    PREFLIGHT["silhouette_error"].max() < 1e-10
), "Pilot quality reproduction failed."

print("A. FULL-DATA REPRODUCTION AUDIT")

display(
    PREFLIGHT.groupby(
        "method", as_index=False
    ).agg(
        n_models=("model_id", "count"),
        min_ARI=("ARI", "min"),
        max_silhouette_error=(
            "silhouette_error", "max"
        )
    )
)

print("\nB. INDEPENDENT TEST INTEGRITY")
print("Blind datasets:", len(PUBLIC_MC))
print("Candidate configurations:", len(GRID_MC))
print("Pilot models reproduced:", len(PREFLIGHT))
print("Model seed:", MODEL_SEED)
print("Versions:", current_versions)

FIT_SPEC = {
    "stage": "independent_mc_full_data_fitting",
    "frozen_protocol_sha256": sha256_checked(
        MC_OUT / "protocol.json"
    ),
    "public_manifest_sha256": sha256_checked(
        BLIND_DIR / "dataset_manifest.csv"
    ),
    "candidate_grid_sha256": sha256_checked(
        RECOVERY_OUT / "candidate_grid.csv"
    ),
    "model_seed": MODEL_SEED,
    "n_datasets": 420,
    "n_candidates": 70,
    "versions": current_versions,
    "ground_truth_access": False
}

fit_spec_path = FULL_OUT / "fitting_spec.json"

if fit_spec_path.exists():
    assert json.loads(
        fit_spec_path.read_text(encoding="utf-8")
    ) == FIT_SPEC
else:
    fit_spec_path.write_text(
        json.dumps(FIT_SPEC, indent=2),
        encoding="utf-8"
    )

PREFLIGHT.to_csv(
    FULL_OUT / "preflight_reproduction.csv",
    index=False
)

print("\nPRE-FLIGHT AUDIT PASSED")
print("Ready for independent full-data fitting.")


A. FULL-DATA REPRODUCTION AUDIT


,method,n_models,min_ARI,max_silhouette_error
0,Hierarchical_Average,5,1.0,8.326673e-17
1,Hierarchical_Complete,5,1.0,8.326673e-17
2,Hierarchical_Ward,5,1.0,5.551115e-17
3,KMedoids_PAM,5,1.0,9.714451e-17
4,Spectral_Early,10,1.0,5.551115e-17
5,Spectral_MultiView,40,1.0,8.326673e-17



B. INDEPENDENT TEST INTEGRITY
Blind datasets: 420
Candidate configurations: 70
Pilot models reproduced: 70
Model seed: 20261018
Versions: {'numpy': '2.1.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'kmedoids': '0.5.5'}

PRE-FLIGHT AUDIT PASSED
Ready for independent full-data fitting.


# Fitting 420 dataset dengan checkpoint
Cell berikut melaksanakan full-data fitting secara bertahap.
Setiap dataset mempunyai tiga file: skor kualitas, label cluster, dan manifest checkpoint berisi SHA-256. Dataset hanya dianggap selesai jika ketiganya valid. Jika suatu konfigurasi gagal, program mencatat error dan menghentikan batch sehingga kegagalan tidak tersembunyi.
Untuk pemeriksaan awal gunakan MAX_NEW_DATASETS = 10. Setelah batch pertama berhasil, angka tersebut bisa dinaikkan tanpa mengubah seed, grid, ataupun aturan pemilihan model.

In [ ]:

# ============================================================
# CELL 20 — CHECKPOINTED INDEPENDENT FULL-DATA FITTING
#
# No ground-truth reads.
# No model selection.
#
# 420 datasets × 70 configurations = 29,400 fits.
# Run in bounded batches; reruns skip verified checkpoints.
# ============================================================

from sklearn.metrics import silhouette_samples
from IPython.display import display

import os
import json
import numpy as np
import pandas as pd

MAX_NEW_DATASETS = 10

DATASET_OUT = FULL_OUT / "datasets"
FAILURE_OUT = FULL_OUT / "failures"

DATASET_OUT.mkdir(exist_ok=True)
FAILURE_OUT.mkdir(exist_ok=True)

assert MAX_NEW_DATASETS >= 1
assert len(PUBLIC_MC) == 420
assert len(GRID_MC) == 70

MODEL_ROWS = GRID_MC.to_dict("records")
ALL_IDS = PUBLIC_MC["dataset_id"].tolist()

def verify_complete_checkpoint(dataset_id, expected_hash):

    directory = DATASET_OUT / dataset_id

    quality_path = directory / "quality.csv"
    labels_path = directory / "labels.csv"
    commit_path = directory / "checkpoint.json"

    if not commit_path.exists():
        if (
            quality_path.exists()
            or labels_path.exists()
        ):
            raise RuntimeError(
                f"Incomplete checkpoint: {dataset_id}. "
                "Inspect the dataset directory before retrying."
            )
        return False

    commit = json.loads(
        commit_path.read_text(encoding="utf-8")
    )

    assert quality_path.exists()
    assert labels_path.exists()

    assert commit["dataset_id"] == dataset_id
    assert commit["data_sha256"] == expected_hash
    assert commit["grid_sha256"] == FIT_SPEC[
        "candidate_grid_sha256"
    ]
    assert commit["model_seed"] == MODEL_SEED

    assert sha256_checked(
        quality_path
    ) == commit["quality_sha256"]

    assert sha256_checked(
        labels_path
    ) == commit["labels_sha256"]

    quality = pd.read_csv(quality_path)
    labels = pd.read_csv(labels_path)

    assert len(quality) == 70
    assert len(labels) == 85

    assert quality["model_id"].is_unique
    assert set(quality["model_id"]) == set(
        GRID_MC["model_id"]
    )

    assert set(labels.columns) == (
        set(GRID_MC["model_id"])
        | {"observation_id"}
    )

    assert np.isfinite(
        quality["silhouette_R0"]
    ).all()

    return True


completed_before = 0
pending = []

for r in PUBLIC_MC.itertuples(index=False):

    if verify_complete_checkpoint(
        r.dataset_id, r.data_sha256
    ):
        completed_before += 1
    else:
        pending.append(r)

print("FULL-DATA FITTING BATCH")
print("Already verified:", completed_before)
print("Remaining:", len(pending))
print(
    "New datasets in this batch:",
    min(len(pending), MAX_NEW_DATASETS)
)

new_completed = 0

for metadata in pending[:MAX_NEW_DATASETS]:

    dataset_id = metadata.dataset_id
    input_path = (
        BLIND_DATA_DIR / f"{dataset_id}.csv"
    )

    assert sha256_checked(
        input_path
    ) == metadata.data_sha256

    frame = pd.read_csv(input_path)

    assert list(frame.columns) == (
        ["observation_id"] + FEATURES
    )

    assert np.array_equal(
        frame["observation_id"].to_numpy(),
        np.arange(85)
    )

    X = frame[FEATURES].to_numpy(
        dtype=float
    )

    assert X.shape == (85, 9)
    assert np.isfinite(X).all()

    ctx = make_sim_context(X)

    quality_rows = []
    label_columns = {
        "observation_id": np.arange(85)
    }

    try:

        for row in MODEL_ROWS:

            model_id = row["model_id"]
            K = int(row["K"])

            fitted, cluster_sizes, density = (
                fit_sim_candidate(row, ctx)
            )

            fitted = np.asarray(
                fitted, dtype=int
            )

            assert len(fitted) == 85
            assert len(np.unique(fitted)) == K

            sil_values = silhouette_samples(
                ctx["D"],
                fitted,
                metric="precomputed"
            )

            quality_rows.append({
                **row,
                "dataset_id": dataset_id,
                "silhouette_R0": float(
                    sil_values.mean()
                ),
                "negative_silhouette_pct": float(
                    100 * np.mean(sil_values < 0)
                ),
                "smallest_cluster": int(
                    cluster_sizes.min()
                ),
                "largest_cluster": int(
                    cluster_sizes.max()
                ),
                "graph_density": density
            })

            label_columns[model_id] = fitted

    except Exception as exc:

        error_record = {
            "dataset_id": dataset_id,
            "failed_model": model_id,
            "error_type": type(exc).__name__,
            "error": str(exc)
        }

        error_path = (
            FAILURE_OUT / f"{dataset_id}.json"
        )

        error_path.write_text(
            json.dumps(
                error_record, indent=2
            ),
            encoding="utf-8"
        )

        raise RuntimeError(
            f"Fitting failed for {dataset_id}, "
            f"model {model_id}. See {error_path}"
        ) from exc

    quality_df = pd.DataFrame(
        quality_rows
    )

    labels_df = pd.DataFrame(
        label_columns
    )

    assert len(quality_df) == 70
    assert quality_df["model_id"].is_unique

    assert set(quality_df["model_id"]) == set(
        GRID_MC["model_id"]
    )

    assert labels_df.shape == (85, 71)
    assert quality_df[
        "silhouette_R0"
    ].notna().all()

    # --------------------------------------------------------
    # Save data before writing the completion marker.
    # --------------------------------------------------------

    dataset_dir = DATASET_OUT / dataset_id

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    quality_path = (
        dataset_dir / "quality.csv"
    )

    labels_path = (
        dataset_dir / "labels.csv"
    )

    commit_path = (
        dataset_dir / "checkpoint.json"
    )

    if (
        quality_path.exists()
        or labels_path.exists()
        or commit_path.exists()
    ):
        raise RuntimeError(
            f"Unexpected existing files: {dataset_id}"
        )

    q_tmp = quality_path.with_suffix(".tmp")
    l_tmp = labels_path.with_suffix(".tmp")

    quality_df.to_csv(
        q_tmp,
        index=False,
        float_format="%.17g"
    )

    labels_df.to_csv(
        l_tmp,
        index=False
    )

    os.replace(q_tmp, quality_path)
    os.replace(l_tmp, labels_path)

    checkpoint = {
        "dataset_id": dataset_id,
        "data_sha256": metadata.data_sha256,
        "grid_sha256": FIT_SPEC[
            "candidate_grid_sha256"
        ],
        "model_seed": MODEL_SEED,
        "n_models": 70,
        "n_observations": 85,
        "quality_sha256": sha256_checked(
            quality_path
        ),
        "labels_sha256": sha256_checked(
            labels_path
        ),
        "status": "SUCCESS"
    }

    c_tmp = commit_path.with_suffix(".tmp")

    c_tmp.write_text(
        json.dumps(checkpoint, indent=2),
        encoding="utf-8"
    )

    os.replace(c_tmp, commit_path)

    assert verify_complete_checkpoint(
        dataset_id,
        metadata.data_sha256
    )

    new_completed += 1

    print(
        f"{dataset_id}: 70/70 configurations "
        "completed and checkpointed"
    )

print("\nBATCH COMPLETED")
print("Previously completed:", completed_before)
print("Newly completed:", new_completed)
print(
    "Total completed:",
    completed_before + new_completed,
    "/420"
)
print(
    "Remaining:",
    420 - completed_before - new_completed
)
print("Output:", DATASET_OUT)


FULL-DATA FITTING BATCH
Already verified: 10
Remaining: 410
New datasets in this batch: 10
D0011: 70/70 configurations completed and checkpointed
D0012: 70/70 configurations completed and checkpointed
D0013: 70/70 configurations completed and checkpointed
D0014: 70/70 configurations completed and checkpointed
D0015: 70/70 configurations completed and checkpointed
D0016: 70/70 configurations completed and checkpointed
D0017: 70/70 configurations completed and checkpointed
D0018: 70/70 configurations completed and checkpointed
D0019: 70/70 configurations completed and checkpointed
D0020: 70/70 configurations completed and checkpointed

BATCH COMPLETED
Previously completed: 10
Newly completed: 10
Total completed: 20 /420
Remaining: 400
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/full_data_fits/datasets


## 20A — Monitor progres tanpa menghentikan notebook

In [ ]:

# ============================================================
# CELL 20A — FULL-DATA FITTING PROGRESS MONITOR
# Safe to run at any point.
# No ground truth accessed.
# ============================================================

verified_ids = []
remaining_ids = []

for record in PUBLIC_MC.itertuples(index=False):

    valid = verify_complete_checkpoint(
        record.dataset_id,
        record.data_sha256
    )

    if valid:
        verified_ids.append(record.dataset_id)
    else:
        remaining_ids.append(record.dataset_id)

total = len(PUBLIC_MC)
completed = len(verified_ids)
remaining = len(remaining_ids)

print("=" * 55)
print("INDEPENDENT MONTE CARLO — FULL-DATA STATUS")
print("=" * 55)

print(f"Dataset expected  : {total}")
print(f"Dataset completed : {completed}")
print(f"Dataset remaining : {remaining}")
print(f"Completion        : {100 * completed / total:.2f}%")

print(f"\nVerified model fits : {completed * 70:,}")
print(f"Remaining model fits: {remaining * 70:,}")

if completed == total:
    print("\nSTATUS: ALL DATASETS COMPLETE")
    print("Cell 21 can now be executed.")
else:
    print("\nSTATUS: IN PROGRESS")
    print("Continue running Cell 20.")

print("=" * 55)


INDEPENDENT MONTE CARLO — FULL-DATA STATUS
Dataset expected  : 420
Dataset completed : 20
Dataset remaining : 400
Completion        : 4.76%

Verified model fits : 1,400
Remaining model fits: 28,000

STATUS: IN PROGRESS
Continue running Cell 20.


# Independent Monte Carlo Full-Data Fitting (Auto-Resume)

In [26]:

# ================================================================
# INDEPENDENT MONTE CARLO
# COMPLETE AUTOMATIC FULL-DATA FITTING + CHECKPOINT + FINAL AUDIT
# ================================================================
#
# Research:
# A Stability-Aware Multi-View Clustering Framework for
# Climate-Related Food Security Vulnerability
#
# Target:
#   420 independent simulated datasets
#    70 candidate configurations per dataset
# 29,400 full-data model fittings
#
# Characteristics:
# 1. Self-contained Colab cell
# 2. Automatically resumes completed checkpoints
# 3. Uses the frozen R0 clustering specification
# 4. Does not read any independent-test ground truth
# 5. Reproduces original pilot before proceeding
# 6. Saves checkpoint after EACH successful dataset
# 7. Audits all 420 datasets upon completion
#
# IMPORTANT:
# Do not change the frozen model parameters or selection rules.
# ================================================================


# ================================================================
# SECTION 1 — IMPORTS AND GOOGLE DRIVE
# ================================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from importlib.metadata import version

import os
import sys
import json
import hashlib
import subprocess

import numpy as np
import pandas as pd
import scipy
import sklearn

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.sparse.csgraph import connected_components

from sklearn.cluster import SpectralClustering
from sklearn.metrics import (
    silhouette_samples,
    silhouette_score,
    adjusted_rand_score
)

try:
    import kmedoids
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable,
        "-m", "pip", "install",
        "kmedoids==0.5.5"
    ])
    import kmedoids


# ================================================================
# SECTION 2 — RESEARCH DIRECTORIES
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

SIM_V2 = (
    ROOT
    / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v2"
)

RECOVERY_OUT = (
    ROOT
    / "03_outputs"
    / "08_monte_carlo_simulation"
    / "recovery_pilot_v1"
)

MC_OUT = (
    ROOT
    / "03_outputs"
    / "09_independent_monte_carlo"
    / "simplex_v2_frozen_v1"
)

BLIND_DIR = MC_OUT / "blind_inputs"
BLIND_DATA_DIR = BLIND_DIR / "datasets"

FULL_OUT = MC_OUT / "full_data_fits"

DATASET_OUT = FULL_OUT / "datasets"
FAILURE_OUT = FULL_OUT / "failures"

DATASET_OUT.mkdir(parents=True, exist_ok=True)
FAILURE_OUT.mkdir(parents=True, exist_ok=True)


# ================================================================
# SECTION 3 — HELPER FUNCTIONS
# ================================================================

def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def atomic_csv(frame, path, **kwargs):
    path = Path(path)

    temp = path.with_suffix(".tmp")

    frame.to_csv(
        temp,
        index=False,
        **kwargs
    )

    os.replace(temp, path)


def atomic_json(obj, path):
    path = Path(path)

    temp = path.with_suffix(".tmp")

    temp.write_text(
        json.dumps(obj, indent=2),
        encoding="utf-8"
    )

    os.replace(temp, path)


# ================================================================
# SECTION 4 — LOAD AND VALIDATE FROZEN PROTOCOL
# ================================================================

protocol_file = MC_OUT / "protocol.json"

fit_spec_file = FULL_OUT / "fitting_spec.json"

recovery_protocol_file = (
    RECOVERY_OUT / "protocol.json"
)

generator_protocol_file = (
    SIM_V2 / "generator_protocol.json"
)

grid_file = (
    RECOVERY_OUT / "candidate_grid.csv"
)

public_manifest_file = (
    BLIND_DIR / "dataset_manifest.csv"
)

required_files = [
    protocol_file,
    fit_spec_file,
    recovery_protocol_file,
    generator_protocol_file,
    grid_file,
    public_manifest_file
]

for path in required_files:
    assert path.exists(), (
        f"Required file missing: {path}"
    )

FROZEN = read_json(protocol_file)
FIT_SPEC = read_json(fit_spec_file)
RECOVERY_SPEC = read_json(recovery_protocol_file)
GENERATOR_SPEC = read_json(generator_protocol_file)

PUBLIC_MC = pd.read_csv(public_manifest_file)
GRID_MC = pd.read_csv(grid_file)

assert FROZEN["n_datasets"] == 420
assert FROZEN["n_candidates"] == 70

assert FROZEN["model_seed"] == 20261018

assert (
    FROZEN["primary_policy"]["minimum_cluster_size"]
    == 5
)

assert np.isclose(
    FROZEN["primary_policy"]["silhouette_tolerance"],
    0.05
)

assert FROZEN["subsampling"] == (
    "simple_random_without_replacement"
)

assert FROZEN["n_subsamples"] == 20
assert FROZEN["subsample_size"] == 68

assert len(PUBLIC_MC) == 420
assert PUBLIC_MC["dataset_id"].is_unique

assert len(GRID_MC) == 70
assert GRID_MC["model_id"].is_unique

assert set(GRID_MC["K"]) == {
    2, 3, 4, 5, 6
}

assert sha256_file(grid_file) == (
    FROZEN["candidate_grid_sha256"]
)

assert sha256_file(generator_protocol_file) == (
    FROZEN["generator_protocol_sha256"]
)

assert sha256_file(protocol_file) == (
    FIT_SPEC["frozen_protocol_sha256"]
)

assert sha256_file(public_manifest_file) == (
    FIT_SPEC["public_manifest_sha256"]
)

assert sha256_file(grid_file) == (
    FIT_SPEC["candidate_grid_sha256"]
)

assert FIT_SPEC["model_seed"] == (
    FROZEN["model_seed"]
)

assert GENERATOR_SPEC["generator"] == (
    "regular_simplex_multiview_v2"
)


# ================================================================
# SECTION 5 — VERIFY SOFTWARE ENVIRONMENT
# ================================================================

CURRENT_VERSIONS = {
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "kmedoids": version("kmedoids")
}

EXPECTED_VERSIONS = RECOVERY_SPEC["versions"]

if CURRENT_VERSIONS != EXPECTED_VERSIONS:
    raise RuntimeError(
        "Library versions do not match the validated "
        "recovery pilot.\n"
        f"Expected: {EXPECTED_VERSIONS}\n"
        f"Current: {CURRENT_VERSIONS}"
    )

assert CURRENT_VERSIONS == (
    FIT_SPEC["versions"]
)

MODEL_SEED = 20261018

print("=" * 65)
print("A. FROZEN PROTOCOL VALIDATION")
print("=" * 65)

print("Datasets             :", len(PUBLIC_MC))
print("Candidate models     :", len(GRID_MC))
print("Expected fittings    :", 420 * 70)
print("Model seed           :", MODEL_SEED)
print("Software versions    :", CURRENT_VERSIONS)
print("Protocol status      : VERIFIED")


# ================================================================
# SECTION 6 — LOCKED REPRESENTATION R0
# ================================================================

FEATURES = GENERATOR_SPEC["features"]

assert len(FEATURES) == 9

VIEW_NAMES = [
    "Food",
    "Climate",
    "Accessibility"
]

WEIGHTS = {
    "equal": (
        1/3, 1/3, 1/3
    ),
    "food50": (
        0.50, 0.25, 0.25
    ),
    "climate50": (
        0.25, 0.50, 0.25
    ),
    "access50": (
        0.25, 0.25, 0.50
    )
}

KG_GRID = [8, 16]
KS_FIXED = 7


def representation_R0_locked(X):
    """
    Replicates audited empirical/synthetic R0 geometry.

    1. Feature-wise sample z-score (ddof=1)
    2. View Euclidean distances / sqrt(3)
    3. Normalize each view RMS to sqrt(2)
    4. Equal-view fused Euclidean geometry
    """

    X = np.asarray(X, dtype=float)

    assert X.ndim == 2
    assert X.shape[1] == 9
    assert np.isfinite(X).all()

    n = X.shape[0]

    mean = X.mean(axis=0)
    sd = X.std(axis=0, ddof=1)

    if np.any(sd <= 0):
        raise ValueError(
            "Constant feature encountered."
        )

    Z = (X - mean) / sd

    blocks = []
    view_distances = {}

    for i, view in enumerate(VIEW_NAMES):

        A = Z[:, 3*i:3*i+3]

        D = (
            squareform(pdist(A))
            / np.sqrt(3)
        )

        upper = D[
            np.triu_indices(n, k=1)
        ]

        rms = np.sqrt(
            np.mean(upper ** 2)
        )

        if not np.isfinite(rms) or rms <= 0:
            raise ValueError(
                "Invalid view RMS."
            )

        factor = np.sqrt(2.0) / rms

        view_distances[view] = (
            D * factor
        )

        blocks.append(
            A * factor / 3.0
        )

    Phi = np.column_stack(blocks)

    fused_D = squareform(
        pdist(Phi)
    )

    reconstructed = np.sqrt(
        sum(
            D ** 2 / 3.0
            for D in view_distances.values()
        )
    )

    assert np.allclose(
        fused_D,
        reconstructed,
        atol=1e-10
    )

    assert np.allclose(
        fused_D,
        fused_D.T,
        atol=1e-12
    )

    assert np.allclose(
        np.diag(fused_D),
        0,
        atol=1e-12
    )

    return Phi, fused_D, view_distances


# ================================================================
# SECTION 7 — LOCKED SPECTRAL AFFINITY CONSTRUCTION
# ================================================================

def make_graph_locked(D, kg, ks):

    D = np.asarray(D, dtype=float)

    n = D.shape[0]

    assert D.shape == (n, n)
    assert np.isfinite(D).all()
    assert np.allclose(
        D, D.T, atol=1e-12
    )

    if np.any(D < 0):
        raise ValueError(
            "Negative distances encountered."
        )

    if not 1 <= max(kg, ks) < n:
        raise ValueError(
            "Invalid graph neighborhood parameters."
        )

    A = D.copy()

    np.fill_diagonal(
        A, np.inf
    )

    order = np.argsort(
        A,
        axis=1,
        kind="stable"
    )

    sigma = A[
        np.arange(n),
        order[:, ks - 1]
    ]

    if np.any(sigma <= 0):
        raise ValueError(
            "Invalid local scale bandwidth."
        )

    adjacency = np.zeros(
        (n, n),
        dtype=bool
    )

    adjacency[
        np.arange(n)[:, None],
        order[:, :kg]
    ] = True

    adjacency |= adjacency.T

    W = np.exp(
        -(D ** 2)
        / (
            sigma[:, None]
            * sigma[None, :]
        )
    )

    W = np.where(
        adjacency,
        W,
        0.0
    )

    np.fill_diagonal(
        W, 0.0
    )

    components = connected_components(
        (W > 0).astype(np.uint8),
        directed=False
    )[0]

    if components != 1:
        raise ValueError(
            f"Disconnected graph: {components}"
        )

    return W


# ================================================================
# SECTION 8 — BUILD MODEL CONTEXT
# ================================================================

def make_context_locked(X):

    Phi, D, D_views = (
        representation_R0_locked(X)
    )

    trees = {
        "Hierarchical_Ward": linkage(
            Phi,
            method="ward",
            metric="euclidean"
        ),
        "Hierarchical_Average": linkage(
            squareform(D, checks=True),
            method="average"
        ),
        "Hierarchical_Complete": linkage(
            squareform(D, checks=True),
            method="complete"
        )
    }

    graph_cache = {}

    for kg in KG_GRID:

        graph_cache[("Early", kg)] = (
            make_graph_locked(
                D, kg, KS_FIXED
            )
        )

        for view in VIEW_NAMES:

            Wv = make_graph_locked(
                D_views[view],
                kg,
                KS_FIXED
            )

            norm = np.linalg.norm(
                Wv, ord="fro"
            )

            if not np.isfinite(norm) or norm <= 0:
                raise ValueError(
                    "Invalid graph Frobenius norm."
                )

            graph_cache[(view, kg)] = (
                Wv / norm
            )

    return {
        "Phi": Phi,
        "D": D,
        "D_views": D_views,
        "trees": trees,
        "graphs": graph_cache
    }


# ================================================================
# SECTION 9 — LOCKED CLUSTERING ALGORITHMS
# ================================================================

def fit_candidate_locked(row, ctx):

    method = row["method"]
    K = int(row["K"])

    density = np.nan

    if method == "KMedoids_PAM":

        labels = kmedoids.pam(
            ctx["D"],
            K,
            init="build",
            max_iter=300,
            random_state=MODEL_SEED
        ).labels

    elif method in ctx["trees"]:

        labels = cut_tree(
            ctx["trees"][method],
            n_clusters=K
        ).ravel()

    elif method.startswith("Spectral_"):

        kg = int(
            row["k_graph"]
        )

        if method == "Spectral_Early":

            W = ctx["graphs"][
                ("Early", kg)
            ]

        elif method == "Spectral_MultiView":

            weights = WEIGHTS[
                row["weight_id"]
            ]

            W = sum(
                weights[i]
                * ctx["graphs"][(view, kg)]
                for i, view in enumerate(VIEW_NAMES)
            )

        else:
            raise ValueError(
                f"Unknown spectral method: {method}"
            )

        estimator = SpectralClustering(
            n_clusters=K,
            affinity="precomputed",
            assign_labels="kmeans",
            eigen_solver="arpack",
            n_init=20,
            random_state=MODEL_SEED
        )

        labels = estimator.fit_predict(
            W
        )

        n = len(W)

        density = (
            np.count_nonzero(
                np.triu(W, k=1)
            )
            / (n * (n - 1) / 2)
        )

    else:
        raise ValueError(
            f"Unknown model method: {method}"
        )

    labels = np.asarray(
        labels,
        dtype=int
    )

    _, cluster_sizes = np.unique(
        labels,
        return_counts=True
    )

    if len(cluster_sizes) != K:
        raise ValueError(
            f"Expected {K} clusters; "
            f"obtained {len(cluster_sizes)}"
        )

    return (
        labels,
        cluster_sizes,
        density
    )


# ================================================================
# SECTION 10 — 70-MODEL REPRODUCTION PREFLIGHT
# ================================================================

PILOT_ID = "aligned_K3_r001"

pilot_input = pd.read_csv(
    SIM_V2 / "datasets" / f"{PILOT_ID}.csv"
)

pilot_labels = pd.read_csv(
    RECOVERY_OUT / "labels" / f"{PILOT_ID}.csv"
)

pilot_quality = pd.read_csv(
    RECOVERY_OUT / "quality" / f"{PILOT_ID}.csv"
).set_index("model_id")

assert np.array_equal(
    pilot_input["observation_id"],
    pilot_labels["observation_id"]
)

pilot_X = pilot_input[
    FEATURES
].to_numpy(dtype=float)

pilot_ctx = make_context_locked(
    pilot_X
)

MODEL_ROWS = GRID_MC.to_dict(
    "records"
)

preflight_records = []

for row in MODEL_ROWS:

    model_id = row["model_id"]

    fitted, _, _ = fit_candidate_locked(
        row, pilot_ctx
    )

    reference = pilot_labels[
        model_id
    ].to_numpy(dtype=int)

    ari = adjusted_rand_score(
        reference,
        fitted
    )

    current_sil = silhouette_score(
        pilot_ctx["D"],
        fitted,
        metric="precomputed"
    )

    reference_sil = float(
        pilot_quality.loc[
            model_id,
            "silhouette_R0"
        ]
    )

    sil_error = abs(
        current_sil - reference_sil
    )

    preflight_records.append({
        "model_id": model_id,
        "ARI": ari,
        "silhouette_error": sil_error
    })


preflight = pd.DataFrame(
    preflight_records
)

assert len(preflight) == 70

assert np.allclose(
    preflight["ARI"],
    1.0,
    rtol=0,
    atol=1e-12
)

assert (
    preflight["silhouette_error"].max()
    < 1e-10
)

print("\n" + "=" * 65)
print("B. REPRODUCTION PREFLIGHT")
print("=" * 65)

print("Models reproduced  :", len(preflight))
print("Minimum ARI        :", preflight["ARI"].min())
print(
    "Maximum Sil error  :",
    preflight["silhouette_error"].max()
)
print("Preflight status   : PASSED")


# ================================================================
# SECTION 11 — VERIFY EXISTING DATASET CHECKPOINTS
# ================================================================

GRID_IDS = set(
    GRID_MC["model_id"]
)

MODEL_K_LOOKUP = dict(
    zip(
        GRID_MC["model_id"],
        GRID_MC["K"]
    )
)


def verify_checkpoint(dataset_id, expected_data_hash):

    dataset_dir = (
        DATASET_OUT / dataset_id
    )

    quality_file = dataset_dir / "quality.csv"
    labels_file = dataset_dir / "labels.csv"
    commit_file = dataset_dir / "checkpoint.json"

    if not commit_file.exists():

        if (
            quality_file.exists()
            or labels_file.exists()
        ):
            raise RuntimeError(
                "Incomplete checkpoint detected: "
                f"{dataset_id}. "
                "Do not overwrite files without inspection."
            )

        return False

    if not (
        quality_file.exists()
        and labels_file.exists()
    ):
        raise RuntimeError(
            f"Checkpoint files missing: {dataset_id}"
        )

    checkpoint = read_json(
        commit_file
    )

    assert checkpoint["dataset_id"] == dataset_id

    assert checkpoint["data_sha256"] == (
        expected_data_hash
    )

    assert checkpoint["grid_sha256"] == (
        FROZEN["candidate_grid_sha256"]
    )

    assert checkpoint["model_seed"] == (
        MODEL_SEED
    )

    assert checkpoint["status"] == "SUCCESS"

    assert sha256_file(quality_file) == (
        checkpoint["quality_sha256"]
    )

    assert sha256_file(labels_file) == (
        checkpoint["labels_sha256"]
    )

    quality = pd.read_csv(
        quality_file
    )

    labels = pd.read_csv(
        labels_file
    )

    assert quality.shape[0] == 70
    assert labels.shape == (85, 71)

    assert quality["model_id"].is_unique

    assert set(quality["model_id"]) == GRID_IDS

    assert set(labels.columns) == (
        GRID_IDS | {"observation_id"}
    )

    assert quality["dataset_id"].eq(
        dataset_id
    ).all()

    assert np.array_equal(
        labels["observation_id"].to_numpy(),
        np.arange(85)
    )

    assert np.isfinite(
        quality["silhouette_R0"]
    ).all()

    assert quality[
        "smallest_cluster"
    ].ge(1).all()

    assert quality[
        "largest_cluster"
    ].le(85).all()

    # Structural validation of all 70 stored partitions.
    for model_id, K in MODEL_K_LOOKUP.items():

        y = labels[
            model_id
        ].to_numpy()

        _, sizes = np.unique(
            y,
            return_counts=True
        )

        assert len(sizes) == int(K), (
            f"Wrong K in {dataset_id}, {model_id}"
        )

    return True


verified_before = []
pending_records = []

for record in PUBLIC_MC.itertuples(index=False):

    if verify_checkpoint(
        record.dataset_id,
        record.data_sha256
    ):
        verified_before.append(
            record.dataset_id
        )
    else:
        pending_records.append(record)


print("\n" + "=" * 65)
print("C. EXISTING CHECKPOINT STATUS")
print("=" * 65)

print(
    "Already completed :",
    len(verified_before)
)

print(
    "Remaining datasets:",
    len(pending_records)
)

print(
    "Remaining fits    :",
    len(pending_records) * 70
)


# ================================================================
# SECTION 12 — AUTOMATIC FITTING OF ALL REMAINING DATASETS
# ================================================================

# This code processes ALL remaining datasets in one execution.
# No repeated manual execution of Cell 20 is necessary.
#
# Results are committed after EACH dataset.
# A Colab interruption does not erase completed checkpoints.
#
# Printing every 10 datasets keeps Colab output manageable.

PRINT_EVERY = 10

new_completed = 0

for record in pending_records:

    dataset_id = record.dataset_id

    input_file = (
        BLIND_DATA_DIR
        / f"{dataset_id}.csv"
    )

    failed_model = "<context_initialization>"

    try:

        # ----------------------------------------------------
        # INPUT INTEGRITY
        # ----------------------------------------------------

        assert input_file.exists()

        assert sha256_file(
            input_file
        ) == record.data_sha256

        frame = pd.read_csv(
            input_file
        )

        assert list(frame.columns) == (
            ["observation_id"] + FEATURES
        )

        assert np.array_equal(
            frame["observation_id"].to_numpy(),
            np.arange(85)
        )

        X = frame[
            FEATURES
        ].to_numpy(dtype=float)

        assert X.shape == (85, 9)
        assert np.isfinite(X).all()

        # ----------------------------------------------------
        # DATASET-SPECIFIC REPRESENTATION AND AFFINITIES
        # ----------------------------------------------------

        ctx = make_context_locked(
            X
        )

        # ----------------------------------------------------
        # FIT ALL 70 CANDIDATE CONFIGURATIONS
        # ----------------------------------------------------

        quality_rows = []

        label_columns = {
            "observation_id": np.arange(
                85, dtype=int
            )
        }

        for model in MODEL_ROWS:

            model_id = model["model_id"]
            failed_model = model_id

            K = int(
                model["K"]
            )

            fitted, sizes, density = (
                fit_candidate_locked(
                    model,
                    ctx
                )
            )

            assert len(fitted) == 85

            assert len(
                np.unique(fitted)
            ) == K

            individual_silhouette = silhouette_samples(
                ctx["D"],
                fitted,
                metric="precomputed"
            )

            assert np.isfinite(
                individual_silhouette
            ).all()

            quality_rows.append({
                **model,
                "dataset_id": dataset_id,
                "silhouette_R0": float(
                    individual_silhouette.mean()
                ),
                "negative_silhouette_pct": float(
                    100
                    * np.mean(
                        individual_silhouette < 0
                    )
                ),
                "smallest_cluster": int(
                    sizes.min()
                ),
                "largest_cluster": int(
                    sizes.max()
                ),
                "graph_density": density
            })

            label_columns[
                model_id
            ] = fitted

        # ----------------------------------------------------
        # COMPLETE DATASET VALIDATION
        # ----------------------------------------------------

        quality_df = pd.DataFrame(
            quality_rows
        )

        labels_df = pd.DataFrame(
            label_columns
        )

        assert len(quality_df) == 70

        assert quality_df[
            "model_id"
        ].is_unique

        assert set(
            quality_df["model_id"]
        ) == GRID_IDS

        assert labels_df.shape == (
            85, 71
        )

        assert np.isfinite(
            quality_df["silhouette_R0"]
        ).all()

        assert quality_df[
            "smallest_cluster"
        ].ge(1).all()

        # ----------------------------------------------------
        # CHECKPOINT STORAGE
        # ----------------------------------------------------

        output_dir = (
            DATASET_OUT / dataset_id
        )

        output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        quality_path = (
            output_dir / "quality.csv"
        )

        labels_path = (
            output_dir / "labels.csv"
        )

        checkpoint_path = (
            output_dir / "checkpoint.json"
        )

        if (
            quality_path.exists()
            or labels_path.exists()
            or checkpoint_path.exists()
        ):
            raise RuntimeError(
                "Unexpected existing checkpoint files "
                f"for {dataset_id}"
            )

        atomic_csv(
            quality_df,
            quality_path,
            float_format="%.17g"
        )

        atomic_csv(
            labels_df,
            labels_path
        )

        checkpoint = {
            "dataset_id": dataset_id,
            "data_sha256": record.data_sha256,
            "grid_sha256": FROZEN[
                "candidate_grid_sha256"
            ],
            "model_seed": MODEL_SEED,
            "n_models": 70,
            "n_observations": 85,
            "quality_sha256": sha256_file(
                quality_path
            ),
            "labels_sha256": sha256_file(
                labels_path
            ),
            "status": "SUCCESS"
        }

        # Written LAST: only fully saved datasets
        # are considered complete.

        atomic_json(
            checkpoint,
            checkpoint_path
        )

        assert verify_checkpoint(
            dataset_id,
            record.data_sha256
        )

        new_completed += 1

        total_completed = (
            len(verified_before)
            + new_completed
        )

        if (
            new_completed % PRINT_EVERY == 0
            or total_completed == 420
        ):

            percentage = (
                100 * total_completed / 420
            )

            print(
                f"[{total_completed:03d}/420] "
                f"{percentage:6.2f}% | "
                f"verified fits: "
                f"{total_completed * 70:,} | "
                f"remaining: "
                f"{420 - total_completed}"
            )

    except Exception as exc:

        failure_record = {
            "dataset_id": dataset_id,
            "failed_model": failed_model,
            "error_type": type(exc).__name__,
            "error": str(exc)
        }

        failure_file = (
            FAILURE_OUT / f"{dataset_id}.json"
        )

        atomic_json(
            failure_record,
            failure_file
        )

        print("\nMODEL FITTING FAILED")
        print("Dataset:", dataset_id)
        print("Model:", failed_model)
        print("Error:", repr(exc))
        print("Completed checkpoints remain preserved.")
        print("Failure log:", failure_file)

        raise


# ================================================================
# SECTION 13 — FINAL CHECKPOINT AUDIT
# ================================================================

print("\n" + "=" * 65)
print("D. FINAL COMPUTATIONAL COVERAGE")
print("=" * 65)

verified_all = []
quality_all = []

for record in PUBLIC_MC.itertuples(index=False):

    valid = verify_checkpoint(
        record.dataset_id,
        record.data_sha256
    )

    if not valid:
        raise RuntimeError(
            f"Dataset still incomplete: "
            f"{record.dataset_id}"
        )

    verified_all.append(
        record.dataset_id
    )

    q = pd.read_csv(
        DATASET_OUT
        / record.dataset_id
        / "quality.csv"
    )

    assert len(q) == 70

    quality_all.append(q)


N_COMPLETED = len(
    verified_all
)

N_EXPECTED = len(
    PUBLIC_MC
)

print(
    "Expected datasets  :",
    N_EXPECTED
)

print(
    "Verified datasets  :",
    N_COMPLETED
)

print(
    "Expected model fits:",
    420 * 70
)

print(
    "Verified model fits:",
    N_COMPLETED * 70
)

assert N_COMPLETED == 420

Q_ALL_MC = pd.concat(
    quality_all,
    ignore_index=True
)

assert len(Q_ALL_MC) == 29400

assert not Q_ALL_MC.duplicated(
    ["dataset_id", "model_id"]
).any()

assert Q_ALL_MC.groupby(
    "dataset_id"
).size().eq(70).all()

assert np.isfinite(
    Q_ALL_MC["silhouette_R0"]
).all()

assert Q_ALL_MC[
    "smallest_cluster"
].ge(1).all()

assert Q_ALL_MC[
    "largest_cluster"
].le(85).all()


# ================================================================
# SECTION 14 — METHOD COVERAGE SUMMARY
# ================================================================

method_coverage = (
    Q_ALL_MC.groupby(
        "method",
        as_index=False
    )
    .agg(
        n_datasets=(
            "dataset_id",
            "nunique"
        ),
        n_fits=(
            "model_id",
            "count"
        ),
        smallest_observed_cluster=(
            "smallest_cluster",
            "min"
        )
    )
)

print("\nE. COVERAGE BY CLUSTERING METHOD")

display(
    method_coverage
)


# ================================================================
# SECTION 15 — CLUSTER-SIZE ELIGIBILITY AUDIT
# ================================================================

eligibility_rows = []

for minimum_size in [1, 5, 8]:

    eligible = Q_ALL_MC.loc[
        Q_ALL_MC[
            "smallest_cluster"
        ] >= minimum_size
    ]

    eligibility_rows.append({
        "minimum_cluster_size": minimum_size,
        "eligible_fits": len(eligible),
        "eligible_fraction": (
            len(eligible)
            / len(Q_ALL_MC)
        ),
        "datasets_with_eligible_candidates": (
            eligible["dataset_id"].nunique()
        )
    })

eligibility = pd.DataFrame(
    eligibility_rows
)

assert (
    eligibility[
        "datasets_with_eligible_candidates"
    ] == 420
).all()

print("\nF. CANDIDATE ELIGIBILITY")

display(
    eligibility.round(4)
)


# ================================================================
# SECTION 16 — SAVE FINAL AGGREGATED RESULTS
# ================================================================

atomic_csv(
    Q_ALL_MC,
    FULL_OUT / "quality_all_420.csv",
    float_format="%.17g"
)

atomic_csv(
    method_coverage,
    FULL_OUT / "method_coverage.csv"
)

atomic_csv(
    eligibility,
    FULL_OUT / "candidate_eligibility.csv"
)


# ================================================================
# SECTION 17 — FINAL SUCCESS REPORT
# ================================================================

print("\n" + "=" * 65)
print("INDEPENDENT MONTE CARLO FULL-DATA FITTING COMPLETED")
print("=" * 65)

print("Status              : SUCCESS")
print("Datasets            : 420/420")
print("Candidate models    : 70")
print("Verified fits       : 29,400/29,400")
print("Model seed          :", MODEL_SEED)
print("Representation      : R0")
print("Ground truth access : NONE")
print("Checkpoint audit    : PASSED")

print("\nSaved outputs:")
print(FULL_OUT / "quality_all_420.csv")
print(FULL_OUT / "method_coverage.csv")
print(FULL_OUT / "candidate_eligibility.csv")

print("\nNext stage:")
print("Independent Monte Carlo subsampling stability.")
print("=" * 65)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
A. FROZEN PROTOCOL VALIDATION
Datasets             : 420
Candidate models     : 70
Expected fittings    : 29400
Model seed           : 20261018
Software versions    : {'numpy': '2.1.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'kmedoids': '0.5.5'}
Protocol status      : VERIFIED

B. REPRODUCTION PREFLIGHT
Models reproduced  : 70
Minimum ARI        : 1.0
Maximum Sil error  : 9.71445146547012e-17
Preflight status   : PASSED

C. EXISTING CHECKPOINT STATUS
Already completed : 20
Remaining datasets: 400
Remaining fits    : 28000
[030/420]   7.14% | verified fits: 2,100 | remaining: 390
[040/420]   9.52% | verified fits: 2,800 | remaining: 380
[050/420]  11.90% | verified fits: 3,500 | remaining: 370
[060/420]  14.29% | verified fits: 4,200 | remaining: 360
[070/420]  16.67% | verified fits: 4,900 | remaining: 350
[080/420]  19.05% | verified fits: 5,600 | remaining:

,method,n_datasets,n_fits,smallest_observed_cluster
0,Hierarchical_Average,420,2100,1
1,Hierarchical_Complete,420,2100,1
2,Hierarchical_Ward,420,2100,2
3,KMedoids_PAM,420,2100,3
4,Spectral_Early,420,4200,5
5,Spectral_MultiView,420,16800,4



F. CANDIDATE ELIGIBILITY


,minimum_cluster_size,eligible_fits,eligible_fraction,datasets_with_eligible_candidates
0,1,29400,1.0000,420
1,5,26736,0.9094,420
2,8,24969,0.8493,420



INDEPENDENT MONTE CARLO FULL-DATA FITTING COMPLETED
Status              : SUCCESS
Datasets            : 420/420
Candidate models    : 70
Verified fits       : 29,400/29,400
Model seed          : 20261018
Representation      : R0
Ground truth access : NONE
Checkpoint audit    : PASSED

Saved outputs:
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/full_data_fits/quality_all_420.csv
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/full_data_fits/method_coverage.csv
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/full_data_fits/candidate_eligibility.csv

Next stage:
Independent Monte Carlo subsampling stability.


# Audit seluruh hasil full-data fitting

In [27]:

# ============================================================
# CELL 21 — FINAL FULL-DATA FITTING COVERAGE AUDIT
#
# Run only when all 420 datasets are complete.
# No ground truth reads.
# ============================================================

import numpy as np
import pandas as pd

verified = []
all_quality = []

for r in PUBLIC_MC.itertuples(index=False):

    if not verify_complete_checkpoint(
        r.dataset_id, r.data_sha256
    ):
        continue

    verified.append(r.dataset_id)

    q = pd.read_csv(
        DATASET_OUT / r.dataset_id / "quality.csv"
    )

    assert q["dataset_id"].eq(
        r.dataset_id
    ).all()

    assert len(q) == len(GRID_MC)

    all_quality.append(q)

print("A. COMPUTATIONAL COVERAGE")
print("Expected datasets:", 420)
print("Verified datasets:", len(verified))
print("Expected configurations per dataset:", 70)
print("Verified full-data fits:", 70 * len(verified))

if len(verified) != 420:
    raise RuntimeError(
        "Full-data fitting is incomplete. "
        "Continue Cell 20 before finalizing."
    )

Q_ALL_MC = pd.concat(
    all_quality,
    ignore_index=True
)

assert len(Q_ALL_MC) == 29400

assert not Q_ALL_MC.duplicated(
    ["dataset_id", "model_id"]
).any()

assert Q_ALL_MC.groupby(
    "dataset_id"
).size().eq(70).all()

assert np.isfinite(
    Q_ALL_MC["silhouette_R0"]
).all()

assert Q_ALL_MC["smallest_cluster"].ge(1).all()

print("\nB. COVERAGE BY CLUSTERING METHOD")

method_coverage = (
    Q_ALL_MC.groupby(
        "method", as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "nunique"),
        n_fits=("model_id", "count"),
        smallest_observed_cluster=(
            "smallest_cluster", "min"
        )
    )
)

display(method_coverage)

print("\nC. CANDIDATE ELIGIBILITY")

eligibility = []

for minimum in [1, 5, 8]:

    subset = Q_ALL_MC.loc[
        Q_ALL_MC["smallest_cluster"]
        >= minimum
    ]

    eligibility.append({
        "minimum_cluster_size": minimum,
        "eligible_fits": len(subset),
        "eligible_fraction": (
            len(subset) / len(Q_ALL_MC)
        ),
        "datasets_with_eligible_candidates": (
            subset["dataset_id"].nunique()
        )
    })

eligibility = pd.DataFrame(eligibility)
display(eligibility.round(4))

assert (
    eligibility[
        "datasets_with_eligible_candidates"
    ] == 420
).all(), (
    "Some datasets have no candidates "
    "under a size rule."
)

# Preserve results without using evaluation labels.
Q_ALL_MC.to_csv(
    FULL_OUT / "quality_all_420.csv",
    index=False,
    float_format="%.17g"
)

method_coverage.to_csv(
    FULL_OUT / "method_coverage.csv",
    index=False
)

eligibility.to_csv(
    FULL_OUT / "candidate_eligibility.csv",
    index=False
)

print("\nFULL-DATA FITTING AUDIT COMPLETED")
print("Total fits verified:", len(Q_ALL_MC))
print("All 420 datasets covered.")
print("Ground truth was not accessed.")
print("Output:", FULL_OUT)


A. COMPUTATIONAL COVERAGE
Expected datasets: 420
Verified datasets: 420
Expected configurations per dataset: 70
Verified full-data fits: 29400

B. COVERAGE BY CLUSTERING METHOD


,method,n_datasets,n_fits,smallest_observed_cluster
0,Hierarchical_Average,420,2100,1
1,Hierarchical_Complete,420,2100,1
2,Hierarchical_Ward,420,2100,2
3,KMedoids_PAM,420,2100,3
4,Spectral_Early,420,4200,5
5,Spectral_MultiView,420,16800,4



C. CANDIDATE ELIGIBILITY


,minimum_cluster_size,eligible_fits,eligible_fraction,datasets_with_eligible_candidates
0,1,29400,1.0000,420
1,5,26736,0.9094,420
2,8,24969,0.8493,420



FULL-DATA FITTING AUDIT COMPLETED
Total fits verified: 29400
All 420 datasets covered.
Ground truth was not accessed.
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/full_data_fits


# Bagian Baru

# Independent Monte Carlo Subsampling Stability

In [28]:

# ================================================================
# INDEPENDENT MONTE CARLO
# AUTOMATIC SUBSAMPLING STABILITY WITH CHECKPOINT / AUTO-RESUME
# ================================================================
#
# DATA:
#   420 blinded synthetic datasets
#   85 observations, 9 indicators, 3 views
#
# MODELS:
#   70 frozen clustering configurations
#
# SUBSAMPLING:
#   20 random subsets per dataset
#   68/85 observations without replacement
#   No stratification and no ground-truth access
#
# EXPECTED REFITS:
#   420 * 20 * 70 = 588,000
#
# PROCEDURE:
#   1. Verify frozen experiment and software versions
#   2. Reproduce all 70 full-data partitions on D0001
#   3. Generate deterministic dataset-specific subsets
#   4. Refit representation and clustering on each subset
#   5. Compare refits with restricted full-data labels
#   6. Save atomic checkpoint after EACH subset
#   7. Resume unfinished datasets without repeating work
#   8. Audit and summarize all stability results
#
# No source-code file or new notebook is created.
# CSV / JSON research outputs are saved to Google Drive.
# ================================================================


# ================================================================
# 1. ENVIRONMENT
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from importlib.metadata import version

import os
import sys
import json
import hashlib
import subprocess
import traceback

import numpy as np
import pandas as pd
import scipy
import sklearn

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.sparse.csgraph import connected_components

from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score

try:
    import kmedoids
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "kmedoids==0.5.5"
    ])
    import kmedoids


# ================================================================
# 2. PATHS
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

MC_OUT = (
    ROOT / "03_outputs"
    / "09_independent_monte_carlo"
    / "simplex_v2_frozen_v1"
)

SIM_V2 = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "generator_pilot_v2"
)

RECOVERY_OUT = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "recovery_pilot_v1"
)

BLIND = MC_OUT / "blind_inputs"
DATA_DIR = BLIND / "datasets"

FULL_OUT = MC_OUT / "full_data_fits"
FULL_DATA_DIR = FULL_OUT / "datasets"

STAB_OUT = (
    MC_OUT / "subsampling_stability_all420_v1"
)

INDICES_DIR = STAB_OUT / "subset_indices"
SCORES_DIR = STAB_OUT / "scores"
CHECKPOINT_DIR = STAB_OUT / "checkpoints"
FAILURE_DIR = STAB_OUT / "failures"

for folder in [
    INDICES_DIR,
    SCORES_DIR,
    CHECKPOINT_DIR,
    FAILURE_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# ================================================================
# 3. UTILITY FUNCTIONS
# ================================================================

def file_sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def read_json(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )


def atomic_csv(df, path, **kwargs):
    path = Path(path)
    temp = path.with_suffix(".tmp")

    df.to_csv(
        temp,
        index=False,
        **kwargs
    )

    os.replace(temp, path)


def atomic_json(obj, path):
    path = Path(path)
    temp = path.with_suffix(".tmp")

    temp.write_text(
        json.dumps(obj, indent=2),
        encoding="utf-8"
    )

    os.replace(temp, path)


def subset_hash(indices):
    indices = np.asarray(indices, dtype="<i8")

    return hashlib.sha256(
        indices.tobytes()
    ).hexdigest()


# ================================================================
# 4. LOAD FROZEN PROTOCOL
# ================================================================

FROZEN = read_json(
    MC_OUT / "protocol.json"
)

FIT_SPEC = read_json(
    FULL_OUT / "fitting_spec.json"
)

RECOVERY_SPEC = read_json(
    RECOVERY_OUT / "protocol.json"
)

GENERATOR_SPEC = read_json(
    SIM_V2 / "generator_protocol.json"
)

PUBLIC = pd.read_csv(
    BLIND / "dataset_manifest.csv"
)

GRID = pd.read_csv(
    RECOVERY_OUT / "candidate_grid.csv"
)

assert FROZEN["n_datasets"] == 420
assert FROZEN["n_candidates"] == 70
assert FROZEN["n_subsamples"] == 20
assert FROZEN["subsample_size"] == 68
assert FROZEN["model_seed"] == 20261018

assert FROZEN["primary_policy"][
    "minimum_cluster_size"
] == 5

assert np.isclose(
    FROZEN["primary_policy"][
        "silhouette_tolerance"
    ],
    0.05
)

assert FROZEN["subsampling"] == (
    "simple_random_without_replacement"
)

assert len(PUBLIC) == 420
assert PUBLIC["dataset_id"].is_unique

assert len(GRID) == 70
assert GRID["model_id"].is_unique
assert set(GRID["K"]) == {2, 3, 4, 5, 6}

assert file_sha256(
    RECOVERY_OUT / "candidate_grid.csv"
) == FROZEN["candidate_grid_sha256"]

assert file_sha256(
    SIM_V2 / "generator_protocol.json"
) == FROZEN["generator_protocol_sha256"]

assert file_sha256(
    MC_OUT / "protocol.json"
) == FIT_SPEC["frozen_protocol_sha256"]

assert file_sha256(
    BLIND / "dataset_manifest.csv"
) == FIT_SPEC["public_manifest_sha256"]

CURRENT_VERSIONS = {
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "kmedoids": version("kmedoids")
}

if CURRENT_VERSIONS != RECOVERY_SPEC["versions"]:
    raise RuntimeError(
        "Software versions changed.\n"
        f"Expected: {RECOVERY_SPEC['versions']}\n"
        f"Current : {CURRENT_VERSIONS}"
    )

assert CURRENT_VERSIONS == FIT_SPEC["versions"]

MODEL_SEED = int(FROZEN["model_seed"])

B = int(FROZEN["n_subsamples"])
SUBSET_SIZE = int(FROZEN["subsample_size"])

SEED_WORDS = list(
    FROZEN["subsample_seed_words"]
)

assert SEED_WORDS == [20261019, 1002]
assert B == 20
assert SUBSET_SIZE == 68

FEATURES = list(
    GENERATOR_SPEC["features"]
)

assert len(FEATURES) == 9

VIEWS = [
    "Food",
    "Climate",
    "Accessibility"
]

WEIGHTS = {
    "equal": (1/3, 1/3, 1/3),
    "food50": (0.50, 0.25, 0.25),
    "climate50": (0.25, 0.50, 0.25),
    "access50": (0.25, 0.25, 0.50)
}

K_GRAPH_GRID = [8, 16]
K_SCALE = 7

MODEL_ROWS = GRID.to_dict("records")
MODEL_IDS = set(GRID["model_id"])

assert set(GRID["method"]) == {
    "KMedoids_PAM",
    "Hierarchical_Ward",
    "Hierarchical_Average",
    "Hierarchical_Complete",
    "Spectral_Early",
    "Spectral_MultiView"
}

assert set(
    GRID.loc[
        GRID["method"].str.startswith("Spectral"),
        "k_graph"
    ].astype(int)
) == set(K_GRAPH_GRID)

assert set(
    GRID.loc[
        GRID["method"].str.startswith("Spectral"),
        "k_scale"
    ].astype(int)
) == {K_SCALE}

assert set(
    GRID.loc[
        GRID["method"] == "Spectral_MultiView",
        "weight_id"
    ]
) == set(WEIGHTS)


# ================================================================
# 5. RECORD BLINDNESS-PRESERVING COMPUTATIONAL AMENDMENT
# ================================================================
#
# The original primary statistical policy is unchanged.
# All 420 datasets are refitted because the fitting stage
# must not use scenario labels to distinguish null/non-null.
#
# Null-dataset stability will be descriptive only.
# ================================================================

AMENDMENT = {
    "experiment": "independent_monte_carlo_v1",
    "stage": "subsampling_stability",
    "implementation": "R0_subsampling_all420_v1",
    "reason_for_amendment": (
        "Run subsampling on all blinded datasets "
        "to avoid using true scenario labels when "
        "selecting the fitting workload."
    ),
    "original_primary_selection_unchanged": True,
    "primary_minimum_cluster_size": 5,
    "primary_delta": 0.05,
    "original_null_scope": (
        "Full-data descriptive only"
    ),
    "amended_null_scope": (
        "Full-data and subsampling descriptive only"
    ),
    "n_datasets": 420,
    "n_models": 70,
    "n_subsamples": 20,
    "subset_size": 68,
    "sampling": "simple_random_without_replacement",
    "stratification": "none",
    "model_seed": MODEL_SEED,
    "subset_seed_words": SEED_WORDS,
    "seed_derivation": (
        "SHA256(dataset_id) first four bytes "
        "little-endian appended to subset_seed_words"
    ),
    "reference": (
        "Full-data labels restricted to subset"
    ),
    "preprocessing": "Refit R0 for every subset",
    "expected_refits": 420 * 20 * 70,
    "ground_truth_access": False,
    "frozen_protocol_sha256": file_sha256(
        MC_OUT / "protocol.json"
    ),
    "public_manifest_sha256": file_sha256(
        BLIND / "dataset_manifest.csv"
    ),
    "candidate_grid_sha256": file_sha256(
        RECOVERY_OUT / "candidate_grid.csv"
    ),
    "software_versions": CURRENT_VERSIONS
}

AMENDMENT_FILE = (
    STAB_OUT / "computational_amendment.json"
)

if AMENDMENT_FILE.exists():
    assert read_json(AMENDMENT_FILE) == AMENDMENT
else:
    atomic_json(AMENDMENT, AMENDMENT_FILE)

AMENDMENT_HASH = file_sha256(AMENDMENT_FILE)


# ================================================================
# 6. REPRESENTATION R0
# ================================================================

def representation_R0_subset(X):

    X = np.asarray(X, dtype=float)

    n, p = X.shape

    assert p == 9
    assert n in (68, 85)
    assert np.isfinite(X).all()

    mean = X.mean(axis=0)
    sd = X.std(axis=0, ddof=1)

    if np.any(sd <= 0):
        raise ValueError("Constant indicator.")

    Z = (X - mean) / sd

    blocks = []
    D_views = {}

    for i, view in enumerate(VIEWS):

        A = Z[:, 3*i:3*i+3]

        D = (
            squareform(pdist(A))
            / np.sqrt(3)
        )

        upper = D[
            np.triu_indices(n, 1)
        ]

        rms = np.sqrt(
            np.mean(upper ** 2)
        )

        if not np.isfinite(rms) or rms <= 0:
            raise ValueError(
                f"Invalid RMS in view {view}"
            )

        factor = np.sqrt(2.0) / rms

        D_views[view] = D * factor

        blocks.append(
            A * factor / 3.0
        )

    Phi = np.column_stack(blocks)

    D_fused = squareform(
        pdist(Phi)
    )

    D_rebuilt = np.sqrt(
        sum(
            D ** 2 / 3.0
            for D in D_views.values()
        )
    )

    assert np.allclose(
        D_fused,
        D_rebuilt,
        atol=1e-10
    )

    assert np.allclose(
        D_fused,
        D_fused.T,
        atol=1e-12
    )

    assert np.allclose(
        np.diag(D_fused),
        0,
        atol=1e-12
    )

    return Phi, D_fused, D_views


# ================================================================
# 7. GRAPH CONSTRUCTION
# ================================================================

def construct_graph(D, k_graph, k_scale):

    D = np.asarray(D, dtype=float)
    n = D.shape[0]

    assert D.shape == (n, n)
    assert np.isfinite(D).all()
    assert np.allclose(
        D, D.T, atol=1e-12
    )

    if np.any(D < 0):
        raise ValueError("Negative distance.")

    if not 1 <= max(k_graph, k_scale) < n:
        raise ValueError(
            "Invalid neighborhood settings."
        )

    A = D.copy()
    np.fill_diagonal(A, np.inf)

    order = np.argsort(
        A,
        axis=1,
        kind="stable"
    )

    sigma = A[
        np.arange(n),
        order[:, k_scale - 1]
    ]

    if np.any(sigma <= 0):
        raise ValueError(
            "Invalid local bandwidth."
        )

    adjacency = np.zeros(
        (n, n), dtype=bool
    )

    adjacency[
        np.arange(n)[:, None],
        order[:, :k_graph]
    ] = True

    adjacency |= adjacency.T

    W = np.exp(
        -(D ** 2)
        / (
            sigma[:, None]
            * sigma[None, :]
        )
    )

    W = np.where(
        adjacency,
        W,
        0.0
    )

    np.fill_diagonal(W, 0.0)

    n_components = connected_components(
        (W > 0).astype(np.uint8),
        directed=False
    )[0]

    if n_components != 1:
        raise ValueError(
            f"Disconnected graph: {n_components}"
        )

    return W


# ================================================================
# 8. FIT CONTEXT
# ================================================================

def create_fit_context(X):

    Phi, D, D_views = (
        representation_R0_subset(X)
    )

    trees = {
        "Hierarchical_Ward": linkage(
            Phi,
            method="ward",
            metric="euclidean"
        ),
        "Hierarchical_Average": linkage(
            squareform(D, checks=True),
            method="average"
        ),
        "Hierarchical_Complete": linkage(
            squareform(D, checks=True),
            method="complete"
        )
    }

    return {
        "D": D,
        "D_views": D_views,
        "trees": trees,
        "graph_cache": {}
    }


# ================================================================
# 9. CACHED AFFINITY CONSTRUCTION
# ================================================================

def get_affinity(ctx, row):

    method = row["method"]

    kg = int(row["k_graph"])
    ks = int(row["k_scale"])

    cache = ctx["graph_cache"]

    if method == "Spectral_Early":

        key = ("early", kg, ks)

        if key not in cache:

            cache[key] = construct_graph(
                ctx["D"], kg, ks
            )

        return cache[key]

    if method == "Spectral_MultiView":

        weight_id = row["weight_id"]

        fused_key = (
            "fused", kg, ks, weight_id
        )

        if fused_key not in cache:

            W = np.zeros_like(ctx["D"])

            weights = WEIGHTS[weight_id]

            for i, view in enumerate(VIEWS):

                view_key = (
                    "view", view, kg, ks
                )

                if view_key not in cache:

                    Wv = construct_graph(
                        ctx["D_views"][view],
                        kg, ks
                    )

                    fro = np.linalg.norm(
                        Wv, ord="fro"
                    )

                    if not np.isfinite(fro) or fro <= 0:
                        raise ValueError(
                            "Invalid Frobenius norm."
                        )

                    cache[view_key] = Wv / fro

                W += (
                    weights[i]
                    * cache[view_key]
                )

            cache[fused_key] = W

        return cache[fused_key]

    raise ValueError(
        f"Unknown spectral method: {method}"
    )


# ================================================================
# 10. CLUSTERING FIT
# ================================================================

def fit_candidate(row, ctx):

    method = row["method"]
    K = int(row["K"])

    if method == "KMedoids_PAM":

        labels = kmedoids.pam(
            ctx["D"],
            K,
            init="build",
            max_iter=300,
            random_state=MODEL_SEED
        ).labels

    elif method in ctx["trees"]:

        labels = cut_tree(
            ctx["trees"][method],
            n_clusters=K
        ).ravel()

    elif method.startswith("Spectral_"):

        W = get_affinity(ctx, row)

        labels = SpectralClustering(
            n_clusters=K,
            affinity="precomputed",
            assign_labels="kmeans",
            eigen_solver="arpack",
            n_init=20,
            random_state=MODEL_SEED
        ).fit_predict(W)

    else:
        raise ValueError(
            f"Unknown clustering method: {method}"
        )

    labels = np.asarray(
        labels, dtype=int
    )

    if len(labels) != len(ctx["D"]):
        raise ValueError(
            "Unexpected number of fitted labels."
        )

    if len(np.unique(labels)) != K:
        raise ValueError(
            f"Expected K={K} clusters."
        )

    return labels


# ================================================================
# 11. REPRODUCTION PREFLIGHT AGAINST STORED FULL FIT
# ================================================================

TEST_ID = PUBLIC.iloc[0]["dataset_id"]

TEST_INPUT = pd.read_csv(
    DATA_DIR / f"{TEST_ID}.csv"
)

TEST_REF = pd.read_csv(
    FULL_DATA_DIR / TEST_ID / "labels.csv"
)

TEST_QUALITY = pd.read_csv(
    FULL_DATA_DIR / TEST_ID / "quality.csv"
).set_index("model_id")

assert np.array_equal(
    TEST_INPUT["observation_id"],
    TEST_REF["observation_id"]
)

TEST_CTX = create_fit_context(
    TEST_INPUT[FEATURES].to_numpy(dtype=float)
)

reproduction = []

for row in MODEL_ROWS:

    model_id = row["model_id"]

    fitted = fit_candidate(
        row, TEST_CTX
    )

    ref = TEST_REF[
        model_id
    ].to_numpy(dtype=int)

    ari = adjusted_rand_score(
        ref, fitted
    )

    current_sil = silhouette_score(
        TEST_CTX["D"],
        fitted,
        metric="precomputed"
    )

    ref_sil = float(
        TEST_QUALITY.loc[
            model_id,
            "silhouette_R0"
        ]
    )

    reproduction.append({
        "model_id": model_id,
        "ARI": float(ari),
        "sil_error": abs(
            current_sil - ref_sil
        )
    })

REPRODUCTION = pd.DataFrame(
    reproduction
)

assert len(REPRODUCTION) == 70

assert np.allclose(
    REPRODUCTION["ARI"],
    1.0,
    atol=1e-12,
    rtol=0
)

assert REPRODUCTION["sil_error"].max() < 1e-10

print("=" * 65)
print("A. FROZEN PROTOCOL / REFITTING AUDIT")
print("=" * 65)

print("Datasets              :", len(PUBLIC))
print("Candidate models      :", len(GRID))
print("Full models reproduced:", len(REPRODUCTION))
print("Minimum ARI           :", REPRODUCTION["ARI"].min())
print("Maximum Sil error     :", REPRODUCTION["sil_error"].max())
print("Software versions     :", CURRENT_VERSIONS)
print("Ground truth access   : NONE")
print("Audit status          : PASSED")


# ================================================================
# 12. DETERMINISTIC SUBSAMPLING PLAN
# ================================================================

def get_subsets(dataset_id):

    # Each dataset receives its own deterministic RNG.
    # No true-cluster labels or scenarios are needed.

    seed_id = int.from_bytes(
        hashlib.sha256(
            dataset_id.encode("utf-8")
        ).digest()[:4],
        byteorder="little"
    )

    child = np.random.SeedSequence(
        SEED_WORDS + [seed_id]
    )

    rng = np.random.default_rng(child)

    subsets = np.vstack([
        np.sort(
            rng.choice(
                85,
                size=SUBSET_SIZE,
                replace=False
            )
        )
        for _ in range(B)
    ]).astype(int)

    assert subsets.shape == (20, 68)

    assert np.all(
        (subsets >= 0) & (subsets < 85)
    )

    for indices in subsets:

        assert len(np.unique(indices)) == 68

    path = INDICES_DIR / f"{dataset_id}.csv"

    expected = pd.DataFrame(subsets)

    if path.exists():

        saved = pd.read_csv(
            path
        ).to_numpy(dtype=int)

        if not np.array_equal(saved, subsets):
            raise RuntimeError(
                f"Subset index mismatch: {dataset_id}"
            )

    else:

        atomic_csv(
            expected,
            path
        )

    return subsets


# ================================================================
# 13. PARTITION STABILITY METRICS
# ================================================================

def partition_stability(
    full_labels,
    subset_indices,
    refitted_labels
):
    """
    Restricted-reference ARI:
       ARI(full_labels[indices], refitted_labels)

    Cluster-wise best-match Jaccard:
       Compute on observations included in the subset.

    Absent reference clusters are counted separately.
    """

    y_full = np.asarray(
        full_labels, dtype=int
    )

    indices = np.asarray(
        subset_indices, dtype=int
    )

    y_subset = y_full[indices]

    y_refit = np.asarray(
        refitted_labels, dtype=int
    )

    assert len(y_subset) == len(y_refit)
    assert len(y_subset) == 68

    ari = adjusted_rand_score(
        y_subset,
        y_refit
    )

    ref_present = np.unique(y_subset)
    fitted_present = np.unique(y_refit)

    jaccards = []

    for reference_cluster in ref_present:

        A = y_subset == reference_cluster

        best_jaccard = 0.0

        for fitted_cluster in fitted_present:

            B_mask = y_refit == fitted_cluster

            intersection = np.sum(
                A & B_mask
            )

            union = np.sum(
                A | B_mask
            )

            j = intersection / union

            best_jaccard = max(
                best_jaccard,
                float(j)
            )

        jaccards.append(best_jaccard)

    n_full_clusters = len(
        np.unique(y_full)
    )

    n_present_clusters = len(
        ref_present
    )

    return {
        "ARI": float(ari),
        "jaccard_macro": float(
            np.mean(jaccards)
        ),
        "jaccard_min": float(
            np.min(jaccards)
        ),
        "reference_clusters_present": int(
            n_present_clusters
        ),
        "missing_reference_clusters": int(
            n_full_clusters - n_present_clusters
        )
    }


# ================================================================
# 14. CHECKPOINT SCHEMA
# ================================================================

SCORE_COLUMNS = [
    "dataset_id",
    "repeat",
    "model_id",
    "subset_sha256",
    "ARI",
    "jaccard_macro",
    "jaccard_min",
    "reference_clusters_present",
    "missing_reference_clusters"
]


def verify_partial_scores(
    dataset_id,
    subsets,
    score_path
):
    """
    A CSV is authoritative only when every completed
    repetition contains exactly the frozen 70 models.

    On interruption, the previous atomic CSV is retained.
    """

    if not score_path.exists():
        return pd.DataFrame(
            columns=SCORE_COLUMNS
        )

    saved = pd.read_csv(score_path)

    if list(saved.columns) != SCORE_COLUMNS:
        raise RuntimeError(
            f"Invalid score columns: {dataset_id}"
        )

    if len(saved) == 0:
        raise RuntimeError(
            f"Empty score checkpoint: {dataset_id}"
        )

    if len(saved) % 70 != 0:
        raise RuntimeError(
            f"Partial model results in {dataset_id}"
        )

    assert saved["dataset_id"].eq(
        dataset_id
    ).all()

    assert not saved.duplicated(
        ["repeat", "model_id"]
    ).any()

    completed_repeats = (
        sorted(saved["repeat"].unique().tolist())
    )

    assert completed_repeats == list(
        range(1, len(completed_repeats) + 1)
    )

    assert len(completed_repeats) <= B

    assert len(saved) == (
        len(completed_repeats) * 70
    )

    for b in completed_repeats:

        group = saved.loc[
            saved["repeat"] == b
        ]

        assert len(group) == 70

        assert set(group["model_id"]) == MODEL_IDS

        expected_hash = subset_hash(
            subsets[b - 1]
        )

        assert group["subset_sha256"].eq(
            expected_hash
        ).all()

    numeric = [
        "ARI",
        "jaccard_macro",
        "jaccard_min",
        "reference_clusters_present",
        "missing_reference_clusters"
    ]

    assert np.isfinite(
        saved[numeric].to_numpy(dtype=float)
    ).all()

    assert saved["ARI"].between(
        -1, 1
    ).all()

    assert saved["jaccard_macro"].between(
        0, 1
    ).all()

    assert saved["jaccard_min"].between(
        0, 1
    ).all()

    assert saved["missing_reference_clusters"].ge(
        0
    ).all()

    return saved


def verify_or_commit_final(
    dataset_id,
    scores,
    input_hash,
    subsets
):

    score_path = SCORES_DIR / f"{dataset_id}.csv"

    commit_path = (
        CHECKPOINT_DIR / f"{dataset_id}.json"
    )

    assert len(scores) == B * 70

    assert scores["repeat"].nunique() == B

    assert score_path.exists()

    expected_commit = {
        "dataset_id": dataset_id,
        "input_sha256": input_hash,
        "score_sha256": file_sha256(score_path),
        "indices_sha256": file_sha256(
            INDICES_DIR / f"{dataset_id}.csv"
        ),
        "amendment_sha256": AMENDMENT_HASH,
        "n_subsamples": B,
        "n_models": 70,
        "n_rows": B * 70,
        "status": "SUCCESS"
    }

    if commit_path.exists():

        if read_json(commit_path) != expected_commit:
            raise RuntimeError(
                f"Completion marker mismatch: {dataset_id}"
            )

    else:

        atomic_json(
            expected_commit,
            commit_path
        )

    return True


# ================================================================
# 15. INITIAL COVERAGE CHECK
# ================================================================

completed_datasets = []
incomplete_datasets = []

for record in PUBLIC.itertuples(index=False):

    score_path = (
        SCORES_DIR / f"{record.dataset_id}.csv"
    )

    if not score_path.exists():

        incomplete_datasets.append(
            record.dataset_id
        )

        continue

    subsets = get_subsets(
        record.dataset_id
    )

    saved = verify_partial_scores(
        record.dataset_id,
        subsets,
        score_path
    )

    if len(saved) == B * 70:

        verify_or_commit_final(
            record.dataset_id,
            saved,
            record.data_sha256,
            subsets
        )

        completed_datasets.append(
            record.dataset_id
        )

    else:

        incomplete_datasets.append(
            record.dataset_id
        )


print("\n" + "=" * 65)
print("B. INITIAL MONTE CARLO STABILITY COVERAGE")
print("=" * 65)

print(
    "Completed datasets   :",
    len(completed_datasets)
)

print(
    "Pending datasets     :",
    len(incomplete_datasets)
)

print(
    "Target subset fits   :",
    420 * B * 70
)


# ================================================================
# 16. AUTOMATIC SUBSAMPLING REFITTING
# ================================================================

# All remaining datasets are processed automatically.
# Completed subsets are preserved in atomic CSV checkpoints.

new_datasets_completed = 0
new_subsets_completed = 0

for position, record in enumerate(
    PUBLIC.itertuples(index=False),
    start=1
):

    dataset_id = record.dataset_id

    input_path = (
        DATA_DIR / f"{dataset_id}.csv"
    )

    assert input_path.exists()

    assert file_sha256(
        input_path
    ) == record.data_sha256

    subset_matrix = get_subsets(
        dataset_id
    )

    score_path = (
        SCORES_DIR / f"{dataset_id}.csv"
    )

    saved = verify_partial_scores(
        dataset_id,
        subset_matrix,
        score_path
    )

    completed_repeats = (
        int(saved["repeat"].max())
        if len(saved)
        else 0
    )

    # Already complete: no refitting.
    if completed_repeats == B:

        verify_or_commit_final(
            dataset_id,
            saved,
            record.data_sha256,
            subset_matrix
        )

        continue

    # --------------------------------------------------------
    # VERIFY FULL-DATA REFERENCE
    # --------------------------------------------------------

    full_dir = (
        FULL_DATA_DIR / dataset_id
    )

    full_commit = read_json(
        full_dir / "checkpoint.json"
    )

    full_labels_path = (
        full_dir / "labels.csv"
    )

    full_quality_path = (
        full_dir / "quality.csv"
    )

    assert full_commit["status"] == "SUCCESS"

    assert full_commit["data_sha256"] == (
        record.data_sha256
    )

    assert full_commit["grid_sha256"] == (
        FROZEN["candidate_grid_sha256"]
    )

    assert full_commit["model_seed"] == MODEL_SEED

    assert file_sha256(
        full_labels_path
    ) == full_commit["labels_sha256"]

    assert file_sha256(
        full_quality_path
    ) == full_commit["quality_sha256"]

    data = pd.read_csv(
        input_path
    )

    reference = pd.read_csv(
        full_labels_path
    )

    assert list(data.columns) == (
        ["observation_id"] + FEATURES
    )

    assert np.array_equal(
        data["observation_id"],
        np.arange(85)
    )

    assert np.array_equal(
        reference["observation_id"],
        data["observation_id"]
    )

    assert set(reference.columns) == (
        MODEL_IDS | {"observation_id"}
    )

    X = data[
        FEATURES
    ].to_numpy(dtype=float)

    assert X.shape == (85, 9)
    assert np.isfinite(X).all()

    REF_LABELS = {
        model_id: reference[
            model_id
        ].to_numpy(dtype=int)
        for model_id in MODEL_IDS
    }

    # --------------------------------------------------------
    # RESUME REPEAT b = completed_repeats + 1
    # --------------------------------------------------------

    for b in range(
        completed_repeats + 1,
        B + 1
    ):

        indices = subset_matrix[
            b - 1
        ]

        this_hash = subset_hash(
            indices
        )

        current_model_id = "<context>"

        try:

            # Refit preprocessing on 68 observations.
            subset_X = X[indices]

            ctx = create_fit_context(
                subset_X
            )

            batch_records = []

            for model_row in MODEL_ROWS:

                model_id = model_row["model_id"]

                current_model_id = model_id

                fitted = fit_candidate(
                    model_row,
                    ctx
                )

                metrics = partition_stability(
                    REF_LABELS[model_id],
                    indices,
                    fitted
                )

                batch_records.append({
                    "dataset_id": dataset_id,
                    "repeat": b,
                    "model_id": model_id,
                    "subset_sha256": this_hash,
                    **metrics
                })

            new_batch = pd.DataFrame(
                batch_records
            )[SCORE_COLUMNS]

            assert len(new_batch) == 70
            assert new_batch["model_id"].is_unique

            assert set(
                new_batch["model_id"]
            ) == MODEL_IDS

            assert np.isfinite(
                new_batch[
                    [
                        "ARI",
                        "jaccard_macro",
                        "jaccard_min"
                    ]
                ].to_numpy()
            ).all()

            # Append one COMPLETE subset to the
            # previous checkpoint atomically.
            updated = pd.concat(
                [saved, new_batch],
                ignore_index=True
            )[SCORE_COLUMNS]

            assert len(updated) == (
                b * 70
            )

            atomic_csv(
                updated,
                score_path,
                float_format="%.17g"
            )

            # Re-read the actual saved checkpoint.
            saved = verify_partial_scores(
                dataset_id,
                subset_matrix,
                score_path
            )

            assert len(saved) == b * 70

            new_subsets_completed += 1

        except Exception as exc:

            failure = {
                "dataset_id": dataset_id,
                "repeat": b,
                "model_id": current_model_id,
                "error_type": type(exc).__name__,
                "error": str(exc),
                "traceback": traceback.format_exc(),
                "completed_repeats_before_failure": b - 1
            }

            failure_file = (
                FAILURE_DIR
                / f"{dataset_id}_b{b:02d}.json"
            )

            atomic_json(
                failure,
                failure_file
            )

            print("\nSUBSAMPLING REFIT FAILURE")
            print("Dataset :", dataset_id)
            print("Repeat  :", b)
            print("Model   :", current_model_id)
            print("Error   :", repr(exc))
            print("Log     :", failure_file)
            print(
                "Previously completed subset checkpoints "
                "remain preserved."
            )

            raise

    # --------------------------------------------------------
    # COMPLETE DATASET COMMIT
    # --------------------------------------------------------

    assert len(saved) == B * 70

    verify_or_commit_final(
        dataset_id,
        saved,
        record.data_sha256,
        subset_matrix
    )

    new_datasets_completed += 1

    # Dataset-level progress.
    if (
        new_datasets_completed % 5 == 0
        or position == len(PUBLIC)
    ):

        committed = sum(
            (
                CHECKPOINT_DIR
                / f"{d}.json"
            ).exists()
            for d in PUBLIC["dataset_id"]
        )

        percentage = (
            100 * committed / len(PUBLIC)
        )

        print(
            f"[{committed:03d}/420] "
            f"{percentage:6.2f}% | "
            f"Completed refits: "
            f"{committed * B * 70:,} | "
            f"Remaining datasets: "
            f"{420 - committed}"
        )


# ================================================================
# 17. FINAL COMPUTATIONAL AUDIT
# ================================================================

all_summary_inputs = []

verified_datasets = 0
verified_subsets = 0
verified_refits = 0

for record in PUBLIC.itertuples(index=False):

    dataset_id = record.dataset_id

    indices = get_subsets(
        dataset_id
    )

    score_path = (
        SCORES_DIR / f"{dataset_id}.csv"
    )

    scores = verify_partial_scores(
        dataset_id,
        indices,
        score_path
    )

    if len(scores) != B * 70:

        raise RuntimeError(
            f"Incomplete dataset: {dataset_id}"
        )

    verify_or_commit_final(
        dataset_id,
        scores,
        record.data_sha256,
        indices
    )

    assert scores["repeat"].nunique() == B

    assert scores.groupby(
        "model_id"
    ).size().eq(B).all()

    verified_datasets += 1
    verified_subsets += B
    verified_refits += len(scores)

    # Summarize per candidate, not per ground-truth class.
    summary = (
        scores.groupby(
            "model_id",
            as_index=False
        )
        .agg(
            n_subsamples=("ARI", "count"),
            mean_ARI=("ARI", "mean"),
            median_ARI=("ARI", "median"),
            p10_ARI=(
                "ARI",
                lambda v: float(
                    np.quantile(v, 0.10)
                )
            ),
            min_ARI=("ARI", "min"),
            mean_jaccard_macro=(
                "jaccard_macro", "mean"
            ),
            mean_jaccard_min=(
                "jaccard_min", "mean"
            ),
            mean_missing_reference_clusters=(
                "missing_reference_clusters",
                "mean"
            )
        )
    )

    summary.insert(
        0,
        "dataset_id",
        dataset_id
    )

    all_summary_inputs.append(
        summary
    )


assert verified_datasets == 420
assert verified_subsets == 8400
assert verified_refits == 588000

STABILITY_ALL = pd.concat(
    all_summary_inputs,
    ignore_index=True
)

assert len(STABILITY_ALL) == 29400

assert not STABILITY_ALL.duplicated(
    ["dataset_id", "model_id"]
).any()

assert STABILITY_ALL[
    "n_subsamples"
].eq(B).all()

assert STABILITY_ALL[
    "mean_ARI"
].between(-1, 1).all()

assert np.isfinite(
    STABILITY_ALL[
        [
            "mean_ARI",
            "median_ARI",
            "p10_ARI",
            "mean_jaccard_macro",
            "mean_jaccard_min"
        ]
    ].to_numpy()
).all()


# ================================================================
# 18. MERGE WITH FULL-DATA QUALITY
# ================================================================

FULL_QUALITY = pd.read_csv(
    FULL_OUT / "quality_all_420.csv"
)

assert len(FULL_QUALITY) == 29400

assert not FULL_QUALITY.duplicated(
    ["dataset_id", "model_id"]
).any()

SELECTION_READY = FULL_QUALITY.merge(
    STABILITY_ALL,
    on=["dataset_id", "model_id"],
    how="inner",
    validate="one_to_one"
)

assert len(SELECTION_READY) == 29400

assert SELECTION_READY.groupby(
    "dataset_id"
).size().eq(70).all()

# At this stage there is NO policy selection.
# No ground truth or true scenario labels are accessed.

atomic_csv(
    STABILITY_ALL,
    STAB_OUT / "stability_all_candidates.csv",
    float_format="%.17g"
)

atomic_csv(
    SELECTION_READY,
    STAB_OUT / "selection_ready_blinded.csv",
    float_format="%.17g"
)


# ================================================================
# 19. FINAL SUMMARY
# ================================================================

coverage = pd.DataFrame([
    {
        "datasets_expected": 420,
        "datasets_verified": verified_datasets,
        "subsets_expected": 8400,
        "subsets_verified": verified_subsets,
        "refits_expected": 588000,
        "refits_verified": verified_refits,
        "candidates_per_dataset": 70,
        "subsets_per_dataset": 20,
        "ground_truth_access": False,
        "status": "SUCCESS"
    }
])

atomic_csv(
    coverage,
    STAB_OUT / "computational_coverage.csv"
)

print("\n" + "=" * 65)
print("C. FINAL INDEPENDENT MONTE CARLO STABILITY AUDIT")
print("=" * 65)

display(coverage)

print("\nSTABILITY DISTRIBUTION BY MODEL FAMILY")

family_summary = STABILITY_ALL.merge(
    GRID[["model_id", "method"]],
    on="model_id",
    validate="many_to_one"
).groupby(
    "method",
    as_index=False
).agg(
    n_configurations=("model_id", "count"),
    mean_subsample_ARI=("mean_ARI", "mean"),
    median_subsample_ARI=("mean_ARI", "median"),
    minimum_observed_ARI=("min_ARI", "min"),
    mean_jaccard=("mean_jaccard_macro", "mean")
)

display(
    family_summary.round(4)
)

atomic_csv(
    family_summary,
    STAB_OUT / "family_stability_summary.csv"
)

print("\n" + "=" * 65)
print("INDEPENDENT MONTE CARLO SUBSAMPLING COMPLETED")
print("=" * 65)

print("Datasets          : 420/420")
print("Subsamples        : 8,400/8,400")
print("Clustering refits : 588,000/588,000")
print("Model seed        :", MODEL_SEED)
print("Ground truth read : NO")
print("All checkpoints   : VERIFIED")
print("Selection policy  : NOT YET APPLIED")

print("\nSaved:")
print(STAB_OUT / "stability_all_candidates.csv")
print(STAB_OUT / "selection_ready_blinded.csv")
print(STAB_OUT / "computational_coverage.csv")
print(STAB_OUT / "family_stability_summary.csv")

print("\nNext stage:")
print("Frozen quality-only vs stability-aware selection.")
print("Then independent ground-truth evaluation.")
print("=" * 65)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
A. FROZEN PROTOCOL / REFITTING AUDIT
Datasets              : 420
Candidate models      : 70
Full models reproduced: 70
Minimum ARI           : 1.0
Maximum Sil error     : 9.71445146547012e-17
Software versions     : {'numpy': '2.1.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'kmedoids': '0.5.5'}
Ground truth access   : NONE
Audit status          : PASSED

B. INITIAL MONTE CARLO STABILITY COVERAGE
Completed datasets   : 0
Pending datasets     : 420
Target subset fits   : 588000


/tmp/ipykernel_1426/2755532212.py:1467: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated = pd.concat(
/tmp/ipykernel_1426/2755532212.py:1467: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated = pd.concat(
/tmp/ipykernel_1426/2755532212.py:1467: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude th

[005/420]   1.19% | Completed refits: 7,000 | Remaining datasets: 415


/tmp/ipykernel_1426/2755532212.py:1467: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated = pd.concat(
/tmp/ipykernel_1426/2755532212.py:1467: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated = pd.concat(
/tmp/ipykernel_1426/2755532212.py:1467: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude th

[010/420]   2.38% | Completed refits: 14,000 | Remaining datasets: 410


...

[420/420] 100.00% | Completed refits: 588,000 | Remaining datasets: 0

C. FINAL INDEPENDENT MONTE CARLO STABILITY AUDIT


,datasets_expected,datasets_verified,subsets_expected,subsets_verified,refits_expected,refits_verified,candidates_per_dataset,subsets_per_dataset,ground_truth_access,status
0,420,420,8400,8400,588000,588000,70,20,False,SUCCESS



STABILITY DISTRIBUTION BY MODEL FAMILY


,method,n_configurations,mean_subsample_ARI,median_subsample_ARI,minimum_observed_ARI,mean_jaccard
0,Hierarchical_Average,2100,0.5129,0.5153,-0.1339,0.6935
1,Hierarchical_Complete,2100,0.3423,0.3401,-0.1458,0.5147
2,Hierarchical_Ward,2100,0.4906,0.4906,-0.1145,0.6182
3,KMedoids_PAM,2100,0.5742,0.5732,-0.1055,0.6750
4,Spectral_Early,4200,0.6448,0.6334,-0.1130,0.7379
5,Spectral_MultiView,16800,0.6179,0.6146,-0.1145,0.7210



INDEPENDENT MONTE CARLO SUBSAMPLING COMPLETED
Datasets          : 420/420
Subsamples        : 8,400/8,400
Clustering refits : 588,000/588,000
Model seed        : 20261018
Ground truth read : NO
All checkpoints   : VERIFIED
Selection policy  : NOT YET APPLIED

Saved:
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/subsampling_stability_all420_v1/stability_all_candidates.csv
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/subsampling_stability_all420_v1/selection_ready_blinded.csv
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/subsampling_stability_all420_v1/computational_coverage.csv
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/subsampling_stability_all420_v1/family_stability_summary.csv

Next stage:
Frozen

# Frozen blind model selection

In [29]:

# ============================================================
# CELL 22 — FROZEN BLIND MODEL SELECTION
#
# Inputs:
#   29,400 full-data quality and stability records
#
# Primary comparison:
#   Quality-only vs Stability-aware
#   Minimum cluster size = 5
#   Silhouette tolerance = 0.05
#
# Secondary sensitivity:
#   Minimum size = 1, 5, 8
#   Delta = 0.02, 0.03, 0.05, 0.07, 0.10
#   Stability-only
#
# NO fitting, NO ground-truth access, NO parameter tuning.
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from IPython.display import display

import hashlib
import json
import os

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# A. INPUT PATHS
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

MC_OUT = (
    ROOT / "03_outputs"
    / "09_independent_monte_carlo"
    / "simplex_v2_frozen_v1"
)

STAB_OUT = (
    MC_OUT / "subsampling_stability_all420_v1"
)

SELECT_OUT = (
    MC_OUT / "frozen_blind_selection_v1"
)

SELECT_OUT.mkdir(parents=True, exist_ok=True)

SOURCE_FILE = (
    STAB_OUT / "selection_ready_blinded.csv"
)

MANIFEST_FILE = (
    MC_OUT / "blind_inputs"
    / "dataset_manifest.csv"
)

GRID_FILE = (
    ROOT / "03_outputs"
    / "08_monte_carlo_simulation"
    / "recovery_pilot_v1"
    / "candidate_grid.csv"
)

FROZEN_FILE = MC_OUT / "protocol.json"


def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def save_once_csv(df, path):
    """
    Refuses to overwrite a different frozen result.
    """

    payload = df.to_csv(
        index=False,
        float_format="%.17g",
        lineterminator="\n"
    ).encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Frozen output changed: {path}"
        )
    else:
        temp = path.with_suffix(".tmp")
        temp.write_bytes(payload)
        os.replace(temp, path)


def save_once_json(obj, path):

    payload = json.dumps(
        obj, indent=2,
        ensure_ascii=False
    ).encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Frozen protocol changed: {path}"
        )
    else:
        temp = path.with_suffix(".tmp")
        temp.write_bytes(payload)
        os.replace(temp, path)


# ------------------------------------------------------------
# B. VERIFY THE ORIGINAL PROTOCOL
# ------------------------------------------------------------

FROZEN = json.loads(
    FROZEN_FILE.read_text(encoding="utf-8")
)

PUBLIC = pd.read_csv(MANIFEST_FILE)
GRID = pd.read_csv(GRID_FILE)
S = pd.read_csv(SOURCE_FILE)

assert len(PUBLIC) == 420
assert PUBLIC["dataset_id"].is_unique

assert len(GRID) == 70
assert GRID["model_id"].is_unique

assert len(S) == 29400
assert S["dataset_id"].nunique() == 420

assert not S.duplicated(
    ["dataset_id", "model_id"]
).any()

assert set(S["dataset_id"]) == set(
    PUBLIC["dataset_id"]
)

assert S.groupby("dataset_id").size().eq(70).all()

assert set(S["model_id"]) == set(
    GRID["model_id"]
)

assert S["n_subsamples"].eq(20).all()

required_numeric = [
    "K",
    "silhouette_R0",
    "smallest_cluster",
    "mean_ARI",
    "p10_ARI"
]

assert np.isfinite(
    S[required_numeric].to_numpy(dtype=float)
).all()

assert S["mean_ARI"].between(-1, 1).all()
assert S["smallest_cluster"].ge(1).all()

assert sha256_file(GRID_FILE) == (
    FROZEN["candidate_grid_sha256"]
)

MINIMUM_RULES = list(
    FROZEN["secondary_analyses"][
        "minimum_cluster_sizes"
    ]
)

DELTA_VALUES = list(
    FROZEN["secondary_analyses"][
        "silhouette_tolerances"
    ]
)

PRIMARY_MIN = FROZEN[
    "primary_policy"
]["minimum_cluster_size"]

PRIMARY_DELTA = FROZEN[
    "primary_policy"
]["silhouette_tolerance"]

assert MINIMUM_RULES == [1, 5, 8]
assert np.allclose(
    DELTA_VALUES,
    [0.02, 0.03, 0.05, 0.07, 0.10]
)

assert PRIMARY_MIN == 5
assert np.isclose(PRIMARY_DELTA, 0.05)


# ------------------------------------------------------------
# C. FREEZE SELECTION SPECIFICATION
# ------------------------------------------------------------

SELECTION_SPEC = {
    "stage": "frozen_blind_selection_v1",
    "statistical_status": (
        "out_of_development_evaluation"
    ),
    "frozen_protocol_sha256": sha256_file(
        FROZEN_FILE
    ),
    "selection_input_sha256": sha256_file(
        SOURCE_FILE
    ),
    "public_manifest_sha256": sha256_file(
        MANIFEST_FILE
    ),
    "candidate_grid_sha256": sha256_file(
        GRID_FILE
    ),
    "datasets": 420,
    "candidates_per_dataset": 70,
    "ground_truth_access": False,
    "primary": {
        "minimum_cluster_size": PRIMARY_MIN,
        "delta": PRIMARY_DELTA,
        "baseline": "quality_only",
        "comparison": "SA_delta_0.05"
    },
    "minimum_size_sensitivity": MINIMUM_RULES,
    "delta_sensitivity": DELTA_VALUES,
    "quality_rule": (
        "Maximum silhouette_R0 among eligible models; "
        "ties: lower K, then lexicographic model_id"
    ),
    "stability_rule": (
        "Maximum mean_ARI; ties: higher p10_ARI, "
        "higher silhouette_R0, lower K, "
        "then lexicographic model_id"
    ),
    "SA_quality_constraint": (
        "silhouette_R0 >= eligible maximum "
        "silhouette_R0 - delta - 1e-12"
    ),
    "null_rejection": False,
    "selection_must_precede_truth": True
}

SPEC_FILE = SELECT_OUT / "selection_spec.json"

save_once_json(SELECTION_SPEC, SPEC_FILE)


# ------------------------------------------------------------
# D. DETERMINISTIC SELECTION FUNCTIONS
# ------------------------------------------------------------

def select_quality(candidates):

    return candidates.sort_values(
        ["silhouette_R0", "K", "model_id"],
        ascending=[False, True, True],
        kind="stable"
    ).iloc[0]


def select_stability(candidates):

    return candidates.sort_values(
        [
            "mean_ARI",
            "p10_ARI",
            "silhouette_R0",
            "K",
            "model_id"
        ],
        ascending=[
            False, False, False, True, True
        ],
        kind="stable"
    ).iloc[0]


def selection_record(
    dataset_id, minimum, policy,
    selected, eligible_count,
    allowed_count, delta
):

    return {
        "dataset_id": dataset_id,
        "min_cluster_rule": minimum,
        "policy": policy,
        "delta": delta,
        "n_eligible": eligible_count,
        "n_quality_allowed": allowed_count,
        "selected_model": selected["model_id"],
        "selected_method": selected["method"],
        "selected_K": int(selected["K"]),
        "selected_silhouette": float(
            selected["silhouette_R0"]
        ),
        "selected_mean_ARI": float(
            selected["mean_ARI"]
        ),
        "selected_p10_ARI": float(
            selected["p10_ARI"]
        ),
        "selected_min_cluster": int(
            selected["smallest_cluster"]
        )
    }


# ------------------------------------------------------------
# E. SELECT ALL POLICIES WITHOUT GROUND TRUTH
# ------------------------------------------------------------

records = []

dataset_groups = {
    dataset_id: group
    for dataset_id, group in S.groupby(
        "dataset_id", sort=False
    )
}

for dataset_id in PUBLIC["dataset_id"]:

    candidate = dataset_groups[dataset_id]

    assert len(candidate) == 70

    for minimum in MINIMUM_RULES:

        eligible = candidate.loc[
            candidate["smallest_cluster"] >= minimum
        ].copy()

        assert len(eligible) > 0

        quality_best = select_quality(eligible)
        stability_best = select_stability(eligible)

        qmax = float(
            eligible["silhouette_R0"].max()
        )

        records.append(
            selection_record(
                dataset_id, minimum,
                "quality_only",
                quality_best,
                len(eligible),
                len(eligible),
                np.nan
            )
        )

        records.append(
            selection_record(
                dataset_id, minimum,
                "stability_only",
                stability_best,
                len(eligible),
                len(eligible),
                np.nan
            )
        )

        for delta in DELTA_VALUES:

            allowed = eligible.loc[
                eligible["silhouette_R0"]
                >= qmax - delta - 1e-12
            ]

            assert len(allowed) >= 1

            sa_best = select_stability(allowed)

            assert (
                sa_best["silhouette_R0"]
                >= qmax - delta - 1e-12
            )

            assert (
                sa_best["mean_ARI"]
                >= quality_best["mean_ARI"] - 1e-12
            )

            policy = f"SA_delta_{delta:.2f}"

            records.append(
                selection_record(
                    dataset_id, minimum,
                    policy,
                    sa_best,
                    len(eligible),
                    len(allowed),
                    delta
                )
            )


ALL_SELECTIONS = pd.DataFrame(records)

assert len(ALL_SELECTIONS) == (
    420 * 3 * 7
)

assert not ALL_SELECTIONS.duplicated(
    [
        "dataset_id",
        "min_cluster_rule",
        "policy"
    ]
).any()

assert ALL_SELECTIONS[
    "selected_min_cluster"
].ge(
    ALL_SELECTIONS["min_cluster_rule"]
).all()


# ------------------------------------------------------------
# F. EXTRACT FROZEN PRIMARY COMPARISON
# ------------------------------------------------------------

PRIMARY = ALL_SELECTIONS.loc[
    (
        ALL_SELECTIONS["min_cluster_rule"]
        == PRIMARY_MIN
    )
    & (
        ALL_SELECTIONS["policy"].isin([
            "quality_only",
            "SA_delta_0.05"
        ])
    )
].copy()

assert len(PRIMARY) == 840

assert PRIMARY.groupby(
    "dataset_id"
).size().eq(2).all()

PRIMARY = PRIMARY.sort_values(
    ["dataset_id", "policy"]
).reset_index(drop=True)

ALL_SELECTIONS = ALL_SELECTIONS.sort_values(
    [
        "dataset_id",
        "min_cluster_rule",
        "policy"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# G. BLIND DIAGNOSTIC COMPARISON
# ------------------------------------------------------------

wide = PRIMARY.pivot(
    index="dataset_id",
    columns="policy",
    values=[
        "selected_model",
        "selected_K",
        "selected_silhouette",
        "selected_mean_ARI"
    ]
)

wide.columns = [
    f"{metric}_{policy}"
    for metric, policy in wide.columns
]

wide = wide.reset_index()

wide["changed_model"] = (
    wide["selected_model_quality_only"]
    != wide["selected_model_SA_delta_0.05"]
)

wide["changed_K"] = (
    wide["selected_K_quality_only"]
    != wide["selected_K_SA_delta_0.05"]
)

wide["gain_selected_stability"] = (
    wide["selected_mean_ARI_SA_delta_0.05"]
    - wide["selected_mean_ARI_quality_only"]
)

wide["loss_selected_silhouette"] = (
    wide["selected_silhouette_quality_only"]
    - wide["selected_silhouette_SA_delta_0.05"]
)

assert len(wide) == 420

assert wide["gain_selected_stability"].ge(
    -1e-12
).all()

assert wide["loss_selected_silhouette"].between(
    -1e-12,
    PRIMARY_DELTA + 1e-12
).all()


# ------------------------------------------------------------
# H. FREEZE OUTPUT FILES
# ------------------------------------------------------------

ALL_FILE = (
    SELECT_OUT / "all_blind_policy_selections.csv"
)

PRIMARY_FILE = (
    SELECT_OUT / "primary_blind_selections.csv"
)

COMPARE_FILE = (
    SELECT_OUT / "primary_blind_comparison.csv"
)

save_once_csv(
    ALL_SELECTIONS, ALL_FILE
)

save_once_csv(
    PRIMARY, PRIMARY_FILE
)

save_once_csv(
    wide, COMPARE_FILE
)

SEAL = {
    "selection_spec_sha256": sha256_file(
        SPEC_FILE
    ),
    "all_selections_sha256": sha256_file(
        ALL_FILE
    ),
    "primary_selections_sha256": sha256_file(
        PRIMARY_FILE
    ),
    "primary_comparison_sha256": sha256_file(
        COMPARE_FILE
    ),
    "n_datasets": 420,
    "n_primary_rows": 840,
    "n_all_policy_rows": 8820,
    "ground_truth_access": False,
    "status": "FROZEN"
}

save_once_json(
    SEAL,
    SELECT_OUT / "selection_seal.json"
)


# ------------------------------------------------------------
# I. REPORT
# ------------------------------------------------------------

print("=" * 65)
print("FROZEN BLIND SELECTION COMPLETED")
print("=" * 65)

print("Datasets              :", len(wide))
print("Primary selections    :", len(PRIMARY))
print("Sensitivity selections:", len(ALL_SELECTIONS))
print("Minimum cluster       :", PRIMARY_MIN)
print("Primary delta         :", PRIMARY_DELTA)
print("Ground truth access   : NONE")
print("Selection seal        : VERIFIED")

print("\nA. PRIMARY POLICY CHANGES")

print(
    "Different model:",
    int(wide["changed_model"].sum()),
    "/420"
)

print(
    "Different K:",
    int(wide["changed_K"].sum()),
    "/420"
)

print(
    "Mean stability gain:",
    round(
        float(wide["gain_selected_stability"].mean()),
        6
    )
)

print(
    "Mean Silhouette loss:",
    round(
        float(wide["loss_selected_silhouette"].mean()),
        6
    )
)

print("\nB. PRIMARY MODEL FAMILY DISTRIBUTION")

display(
    PRIMARY.groupby(
        ["policy", "selected_method"]
    ).size().rename(
        "n_selections"
    ).reset_index()
)

print("\nC. OUTPUT DIRECTORY")
print(SELECT_OUT)

print("\nSTATUS: READY FOR GROUND-TRUTH EVALUATION")
print("=" * 65)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FROZEN BLIND SELECTION COMPLETED
Datasets              : 420
Primary selections    : 840
Sensitivity selections: 8820
Minimum cluster       : 5
Primary delta         : 0.05
Ground truth access   : NONE
Selection seal        : VERIFIED

A. PRIMARY POLICY CHANGES
Different model: 333 /420
Different K: 93 /420
Mean stability gain: 0.12558
Mean Silhouette loss: 0.017982

B. PRIMARY MODEL FAMILY DISTRIBUTION


,policy,selected_method,n_selections
0,SA_delta_0.05,Hierarchical_Average,10
1,SA_delta_0.05,Hierarchical_Complete,6
2,SA_delta_0.05,Hierarchical_Ward,1
3,SA_delta_0.05,KMedoids_PAM,10
4,SA_delta_0.05,Spectral_Early,154
5,SA_delta_0.05,Spectral_MultiView,239
6,quality_only,Hierarchical_Average,31
7,quality_only,Hierarchical_Complete,35
8,quality_only,Hierarchical_Ward,20
9,quality_only,KMedoids_PAM,9



C. OUTPUT DIRECTORY
/content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/frozen_blind_selection_v1

STATUS: READY FOR GROUND-TRUTH EVALUATION


# Independent Ground-Truth Evaluation

In [30]:

# ================================================================
# CELL 23 — INDEPENDENT GROUND-TRUTH EVALUATION
#
# PRIMARY:
# Quality-only vs Stability-aware (delta=0.05, min_size=5)
#
# 360 non-null datasets: 6 scenarios x 60 replicates
# 60 Gaussian null datasets: descriptive diagnostics only
#
# No clustering refits.
# No re-selection of models.
# Dataset-level paired bootstrap, 10,000 repetitions.
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from IPython.display import display

import os
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.metrics import adjusted_rand_score


# ------------------------------------------------------------
# 1. PATHS AND IMMUTABLE OUTPUT HELPERS
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

MC = (
    ROOT / "03_outputs"
    / "09_independent_monte_carlo"
    / "simplex_v2_frozen_v1"
)

SEL = MC / "frozen_blind_selection_v1"
FULL = MC / "full_data_fits"
EVAL = MC / "evaluation_only"
BLIND = MC / "blind_inputs"

OUT = MC / "ground_truth_evaluation_v1"
OUT.mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def load_json(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )


def save_once_json(obj, path):
    payload = json.dumps(
        obj, indent=2, ensure_ascii=False
    ).encode("utf-8")

    if path.exists():
        if path.read_bytes() != payload:
            raise RuntimeError(
                f"Frozen JSON differs: {path}"
            )
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_bytes(payload)
        os.replace(tmp, path)


def save_once_csv(df, path):
    payload = df.to_csv(
        index=False,
        float_format="%.17g",
        lineterminator="\n"
    ).encode("utf-8")

    if path.exists():
        if path.read_bytes() != payload:
            raise RuntimeError(
                f"Frozen CSV differs: {path}"
            )
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_bytes(payload)
        os.replace(tmp, path)


# ------------------------------------------------------------
# 2. VERIFY ALL BLIND SELECTIONS FIRST
# ------------------------------------------------------------

seal_path = SEL / "selection_seal.json"
spec_path = SEL / "selection_spec.json"

all_path = SEL / "all_blind_policy_selections.csv"
primary_path = SEL / "primary_blind_selections.csv"
comparison_path = SEL / "primary_blind_comparison.csv"

frozen_path = MC / "protocol.json"
source_path = (
    MC / "subsampling_stability_all420_v1"
    / "selection_ready_blinded.csv"
)

seal = load_json(seal_path)
select_spec = load_json(spec_path)
frozen = load_json(frozen_path)

assert seal["status"] == "FROZEN"
assert seal["ground_truth_access"] is False

assert sha256(spec_path) == seal[
    "selection_spec_sha256"
]

assert sha256(all_path) == seal[
    "all_selections_sha256"
]

assert sha256(primary_path) == seal[
    "primary_selections_sha256"
]

assert sha256(comparison_path) == seal[
    "primary_comparison_sha256"
]

assert sha256(source_path) == select_spec[
    "selection_input_sha256"
]

assert sha256(frozen_path) == select_spec[
    "frozen_protocol_sha256"
]

ALL = pd.read_csv(all_path)
PRIMARY = pd.read_csv(primary_path)
SOURCE = pd.read_csv(source_path)
PUBLIC = pd.read_csv(
    BLIND / "dataset_manifest.csv"
)

assert len(ALL) == 8820
assert len(PRIMARY) == 840
assert len(SOURCE) == 29400
assert len(PUBLIC) == 420

assert not ALL.duplicated(
    ["dataset_id", "min_cluster_rule", "policy"]
).any()

assert not SOURCE.duplicated(
    ["dataset_id", "model_id"]
).any()

assert ALL.groupby("dataset_id").size().eq(21).all()

assert set(ALL["dataset_id"]) == set(
    PUBLIC["dataset_id"]
)

assert set(PRIMARY["policy"]) == {
    "quality_only", "SA_delta_0.05"
}

assert PRIMARY["min_cluster_rule"].eq(5).all()

# Verify that selected models are precisely those stored
# in the frozen candidate-level results.
check = ALL.merge(
    SOURCE[
        [
            "dataset_id", "model_id", "method", "K",
            "silhouette_R0", "mean_ARI",
            "smallest_cluster"
        ]
    ],
    left_on=["dataset_id", "selected_model"],
    right_on=["dataset_id", "model_id"],
    validate="many_to_one",
    how="left"
)

assert len(check) == len(ALL)
assert check["model_id"].notna().all()

assert (
    check["selected_method"] == check["method"]
).all()

assert (
    check["selected_K"] == check["K"]
).all()

assert (
    check["selected_min_cluster"]
    == check["smallest_cluster"]
).all()

assert np.allclose(
    check["selected_silhouette"],
    check["silhouette_R0"],
    atol=1e-12, rtol=0
)

assert np.allclose(
    check["selected_mean_ARI"],
    check["mean_ARI"],
    atol=1e-12, rtol=0
)

# Primary selection must exactly match the frozen all-policy
# selection file, not an independent recalculation.
expected_primary = ALL.loc[
    (ALL["min_cluster_rule"] == 5)
    & ALL["policy"].isin([
        "quality_only", "SA_delta_0.05"
    ])
].sort_values(
    ["dataset_id", "policy"]
).reset_index(drop=True)

actual_primary = PRIMARY.sort_values(
    ["dataset_id", "policy"]
).reset_index(drop=True)

pd.testing.assert_frame_equal(
    expected_primary[actual_primary.columns],
    actual_primary,
    check_exact=False,
    rtol=0,
    atol=1e-12
)

print("A. BLIND SELECTION SEAL: VERIFIED")


# ------------------------------------------------------------
# 3. FREEZE EVALUATION ANALYSIS BEFORE READING TRUTH
# ------------------------------------------------------------

BOOT_B = 10000
BOOT_SEED = [20261019, 2301]

ANALYSIS_SPEC = {
    "stage": "ground_truth_evaluation_v1",
    "selection_seal_sha256": sha256(seal_path),
    "all_selections_sha256": sha256(all_path),
    "full_protocol_sha256": sha256(frozen_path),
    "primary_baseline": "quality_only",
    "primary_comparison": "SA_delta_0.05",
    "primary_min_cluster": 5,
    "primary_delta": 0.05,
    "primary_outcome": "adjusted_rand_index_to_truth",
    "primary_estimand": (
        "Equal-weighted mean of six scenario-specific "
        "paired mean ARI differences"
    ),
    "primary_scenarios": [
        "aligned_K2",
        "aligned_K3",
        "aligned_K4",
        "conflict_K3",
        "imbalanced_K3",
        "weak_K3"
    ],
    "replicates_per_scenario": 60,
    "bootstrap_unit": "independent_dataset",
    "bootstrap_type": "stratified_paired_percentile",
    "bootstrap_repetitions": BOOT_B,
    "bootstrap_seed_words": BOOT_SEED,
    "confidence_level": 0.95,
    "secondary": [
        "correct_K",
        "scenario_specific_paired_ARI",
        "minimum_size_sensitivity",
        "delta_sensitivity",
        "selected_method_distribution",
        "climate_nuisance_agreement_in_conflict"
    ],
    "null_datasets": (
        "Descriptive forced-partition diagnostics only"
    ),
    "multiple_comparisons": (
        "Scenario-specific intervals are descriptive "
        "and not simultaneous familywise intervals"
    ),
    "selection_recalculated_after_truth": False,
    "status": "LOCKED_BEFORE_UNBLINDING"
}

save_once_json(
    ANALYSIS_SPEC,
    OUT / "analysis_spec.json"
)


# ------------------------------------------------------------
# 4. OPEN AND VALIDATE PRIVATE GENERATOR TRUTH
# ------------------------------------------------------------

PRIVATE = pd.read_csv(
    EVAL / "generator_truth_manifest.csv"
)

assert len(PRIVATE) == 420
assert PRIVATE["dataset_id"].is_unique

assert set(PRIVATE["dataset_id"]) == set(
    PUBLIC["dataset_id"]
)

expected_scenarios = (
    ANALYSIS_SPEC["primary_scenarios"]
    + ["null_gaussian"]
)

assert set(PRIVATE["scenario"]) == set(
    expected_scenarios
)

assert PRIVATE.groupby(
    "scenario"
).size().eq(60).all()

assert PRIVATE.loc[
    PRIVATE["scenario"] == "null_gaussian",
    "K_true"
].eq(-1).all()

assert PRIVATE.loc[
    PRIVATE["scenario"] != "null_gaussian",
    "K_true"
].ge(2).all()

assert PRIVATE["generator_replicate"].between(
    1, 60
).all()

assert not PRIVATE.duplicated(
    ["scenario", "generator_replicate"]
).any()

META = PRIVATE.set_index("dataset_id")
ALL_GROUPS = {
    d: g.copy()
    for d, g in ALL.groupby("dataset_id")
}

EVALUATIONS = []

for record in PUBLIC.itertuples(index=False):

    did = record.dataset_id
    info = META.loc[did]

    # Verify both the original feature hash and truth hash.
    feature_path = (
        BLIND / "datasets" / f"{did}.csv"
    )

    truth_path = (
        EVAL / "ground_truth" / f"{did}.csv"
    )

    labels_path = (
        FULL / "datasets" / did / "labels.csv"
    )

    checkpoint_path = (
        FULL / "datasets" / did / "checkpoint.json"
    )

    assert sha256(feature_path) == record.data_sha256
    assert info["data_sha256"] == record.data_sha256
    assert sha256(truth_path) == info["truth_sha256"]

    full_checkpoint = load_json(checkpoint_path)

    assert full_checkpoint["status"] == "SUCCESS"
    assert sha256(labels_path) == full_checkpoint[
        "labels_sha256"
    ]

    truth = pd.read_csv(truth_path)
    labels = pd.read_csv(labels_path)

    assert len(truth) == len(labels) == 85

    assert np.array_equal(
        truth["observation_id"].to_numpy(),
        labels["observation_id"].to_numpy()
    )

    y_true = truth["true_cluster"].to_numpy(
        dtype=int
    )

    nuisance = truth[
        "nuisance_climate_group"
    ].to_numpy(dtype=int)

    is_null = info["scenario"] == "null_gaussian"

    if is_null:
        assert np.all(y_true == -1)
    else:
        assert len(np.unique(y_true)) == int(
            info["K_true"]
        )

    if info["scenario"] == "conflict_K3":
        assert len(np.unique(nuisance)) == 3

    # Evaluate unique selected models only once per dataset.
    chosen_models = ALL_GROUPS[did][
        "selected_model"
    ].unique()

    metric_cache = {}

    for model_id in chosen_models:

        y_pred = labels[model_id].to_numpy(
            dtype=int
        )

        clusters, cluster_sizes = np.unique(
            y_pred, return_counts=True
        )

        metric_cache[model_id] = {
            "K_selected_verified": len(clusters),
            "minimum_size_verified": int(
                cluster_sizes.min()
            ),
            "true_ARI": (
                np.nan
                if is_null
                else float(
                    adjusted_rand_score(y_true, y_pred)
                )
            ),
            "nuisance_ARI": (
                float(adjusted_rand_score(
                    nuisance, y_pred
                ))
                if info["scenario"] == "conflict_K3"
                else np.nan
            )
        }

    for row in ALL_GROUPS[did].itertuples(index=False):

        values = metric_cache[row.selected_model]

        assert values["K_selected_verified"] == (
            row.selected_K
        )

        assert values["minimum_size_verified"] == (
            row.selected_min_cluster
        )

        EVALUATIONS.append({
            "dataset_id": did,
            "scenario": info["scenario"],
            "generator_replicate": int(
                info["generator_replicate"]
            ),
            "K_true": int(info["K_true"]),
            "min_cluster_rule": int(
                row.min_cluster_rule
            ),
            "policy": row.policy,
            "delta": row.delta,
            "selected_model": row.selected_model,
            "selected_method": row.selected_method,
            "selected_K": int(row.selected_K),
            "selected_silhouette": float(
                row.selected_silhouette
            ),
            "selected_stability": float(
                row.selected_mean_ARI
            ),
            "selected_min_cluster": int(
                row.selected_min_cluster
            ),
            "true_ARI": values["true_ARI"],
            "correct_K": (
                np.nan if is_null
                else int(
                    row.selected_K == info["K_true"]
                )
            ),
            "climate_nuisance_ARI": values[
                "nuisance_ARI"
            ]
        })


RESULT = pd.DataFrame(EVALUATIONS).sort_values(
    ["dataset_id", "min_cluster_rule", "policy"]
).reset_index(drop=True)

assert len(RESULT) == 8820

assert not RESULT.duplicated(
    ["dataset_id", "min_cluster_rule", "policy"]
).any()

NONNULL = RESULT.loc[
    RESULT["scenario"] != "null_gaussian"
].copy()

NULL = RESULT.loc[
    RESULT["scenario"] == "null_gaussian"
].copy()

assert NONNULL["dataset_id"].nunique() == 360
assert NULL["dataset_id"].nunique() == 60

assert NONNULL["true_ARI"].notna().all()
assert NULL["true_ARI"].isna().all()

assert NONNULL["true_ARI"].between(-1, 1).all()

print("B. GROUND TRUTH VALIDATED")
print("Non-null datasets:", 360)
print("Gaussian null datasets:", 60)
print("Policy evaluations:", len(RESULT))


# ------------------------------------------------------------
# 5. FROZEN PRIMARY PAIRED EVALUATION
# ------------------------------------------------------------

P = NONNULL.loc[
    (NONNULL["min_cluster_rule"] == 5)
    & NONNULL["policy"].isin([
        "quality_only", "SA_delta_0.05"
    ])
].copy()

assert len(P) == 720

idx = [
    "dataset_id", "scenario",
    "generator_replicate", "K_true"
]

W = P.pivot(
    index=idx,
    columns="policy",
    values=[
        "true_ARI",
        "correct_K",
        "selected_K",
        "selected_model",
        "selected_method",
        "selected_silhouette",
        "selected_stability",
        "climate_nuisance_ARI"
    ]
)

W.columns = [
    f"{field}_{policy}"
    for field, policy in W.columns
]

W = W.reset_index()

assert len(W) == 360

# Pivot with mixed object/numeric fields may yield
# object dtype. Convert relevant quantities explicitly.
numeric_metrics = [
    "true_ARI",
    "correct_K",
    "selected_K",
    "selected_silhouette",
    "selected_stability",
    "climate_nuisance_ARI"
]

for metric in numeric_metrics:
    for policy in [
        "quality_only", "SA_delta_0.05"
    ]:
        col = f"{metric}_{policy}"
        W[col] = pd.to_numeric(
            W[col], errors="coerce"
        )

W["delta_true_ARI"] = (
    W["true_ARI_SA_delta_0.05"]
    - W["true_ARI_quality_only"]
)

W["delta_correct_K"] = (
    W["correct_K_SA_delta_0.05"]
    - W["correct_K_quality_only"]
)

W["changed_model"] = (
    W["selected_model_SA_delta_0.05"]
    != W["selected_model_quality_only"]
)

W["changed_K"] = (
    W["selected_K_SA_delta_0.05"]
    != W["selected_K_quality_only"]
)

W["delta_climate_nuisance_ARI"] = (
    W["climate_nuisance_ARI_SA_delta_0.05"]
    - W["climate_nuisance_ARI_quality_only"]
)

assert W["delta_true_ARI"].notna().all()
assert W["delta_correct_K"].notna().all()

assert W["delta_correct_K"].isin(
    [-1, 0, 1]
).all()


# ------------------------------------------------------------
# 6. STRATIFIED PAIRED BOOTSTRAP
# ------------------------------------------------------------

scenario_order = ANALYSIS_SPEC[
    "primary_scenarios"
]

rng = np.random.default_rng(
    np.random.SeedSequence(BOOT_SEED)
)

boot_overall_ARI = np.zeros(BOOT_B)
boot_overall_K = np.zeros(BOOT_B)

scenario_rows = []


def percentile_interval(values):
    return np.quantile(
        values, [0.025, 0.975]
    )


for scenario in scenario_order:

    d = W.loc[
        W["scenario"] == scenario
    ].sort_values(
        "generator_replicate"
    ).copy()

    assert len(d) == 60

    diffs_ari = d["delta_true_ARI"].to_numpy(
        dtype=float
    )

    diffs_k = d["delta_correct_K"].to_numpy(
        dtype=float
    )

    # Independent resampling of complete datasets.
    # Quality-only and SA stay paired within dataset.
    indices = rng.integers(
        0, 60,
        size=(BOOT_B, 60)
    )

    boot_ari = diffs_ari[
        indices
    ].mean(axis=1)

    boot_k = diffs_k[
        indices
    ].mean(axis=1)

    boot_overall_ARI += boot_ari / 6
    boot_overall_K += boot_k / 6

    ci_ari = percentile_interval(boot_ari)
    ci_k = percentile_interval(boot_k)

    scenario_rows.append({
        "scenario": scenario,
        "n_datasets": 60,
        "mean_ARI_quality": float(
            d["true_ARI_quality_only"].mean()
        ),
        "mean_ARI_SA": float(
            d["true_ARI_SA_delta_0.05"].mean()
        ),
        "mean_delta_ARI": float(
            diffs_ari.mean()
        ),
        "CI95_delta_ARI_low": float(ci_ari[0]),
        "CI95_delta_ARI_high": float(ci_ari[1]),
        "median_delta_ARI": float(
            np.median(diffs_ari)
        ),
        "SA_better": int((diffs_ari > 1e-12).sum()),
        "SA_worse": int((diffs_ari < -1e-12).sum()),
        "equal_ARI": int(
            np.isclose(
                diffs_ari, 0,
                atol=1e-12, rtol=0
            ).sum()
        ),
        "K_accuracy_quality": float(
            d["correct_K_quality_only"].mean()
        ),
        "K_accuracy_SA": float(
            d["correct_K_SA_delta_0.05"].mean()
        ),
        "delta_K_accuracy": float(diffs_k.mean()),
        "CI95_delta_K_low": float(ci_k[0]),
        "CI95_delta_K_high": float(ci_k[1]),
        "changed_model_fraction": float(
            d["changed_model"].mean()
        ),
        "changed_K_fraction": float(
            d["changed_K"].mean()
        )
    })


SCENARIO_SUMMARY = pd.DataFrame(
    scenario_rows
)

overall_ari_ci = percentile_interval(
    boot_overall_ARI
)

overall_k_ci = percentile_interval(
    boot_overall_K
)

OVERALL = pd.DataFrame([{
    "n_nonnull_datasets": 360,
    "n_scenarios": 6,
    "mean_ARI_quality": float(
        W["true_ARI_quality_only"].mean()
    ),
    "mean_ARI_SA": float(
        W["true_ARI_SA_delta_0.05"].mean()
    ),
    "mean_delta_ARI": float(
        W["delta_true_ARI"].mean()
    ),
    "CI95_delta_ARI_low": float(
        overall_ari_ci[0]
    ),
    "CI95_delta_ARI_high": float(
        overall_ari_ci[1]
    ),
    "K_accuracy_quality": float(
        W["correct_K_quality_only"].mean()
    ),
    "K_accuracy_SA": float(
        W["correct_K_SA_delta_0.05"].mean()
    ),
    "delta_K_accuracy": float(
        W["delta_correct_K"].mean()
    ),
    "CI95_delta_K_low": float(
        overall_k_ci[0]
    ),
    "CI95_delta_K_high": float(
        overall_k_ci[1]
    ),
    "fraction_changed_model": float(
        W["changed_model"].mean()
    ),
    "fraction_changed_K": float(
        W["changed_K"].mean()
    ),
    "bootstrap_B": BOOT_B
}])

# Because every scenario has n=60,
# the dataset average equals the equally weighted
# average of the six scenario means.
assert np.isclose(
    OVERALL.loc[0, "mean_delta_ARI"],
    SCENARIO_SUMMARY["mean_delta_ARI"].mean(),
    atol=1e-12
)


# ------------------------------------------------------------
# 7. SENSITIVITY ANALYSIS — FROZEN SECONDARY POLICIES
# ------------------------------------------------------------

QUALITY_BASE = NONNULL.loc[
    NONNULL["policy"] == "quality_only",
    [
        "dataset_id",
        "min_cluster_rule",
        "true_ARI",
        "correct_K"
    ]
].rename(columns={
    "true_ARI": "baseline_ARI",
    "correct_K": "baseline_correct_K"
})

SENS = NONNULL.merge(
    QUALITY_BASE,
    on=["dataset_id", "min_cluster_rule"],
    validate="many_to_one"
)

SENS["delta_vs_quality"] = (
    SENS["true_ARI"] - SENS["baseline_ARI"]
)

SENS["delta_correct_K_vs_quality"] = (
    SENS["correct_K"]
    - SENS["baseline_correct_K"]
)

SENS_SUMMARY = (
    SENS.groupby(
        ["min_cluster_rule", "policy"],
        as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "nunique"),
        mean_true_ARI=("true_ARI", "mean"),
        mean_delta_vs_quality=(
            "delta_vs_quality", "mean"
        ),
        median_delta_vs_quality=(
            "delta_vs_quality", "median"
        ),
        K_accuracy=("correct_K", "mean"),
        mean_selected_stability=(
            "selected_stability", "mean"
        ),
        mean_selected_silhouette=(
            "selected_silhouette", "mean"
        )
    )
)

assert SENS_SUMMARY.shape[0] == 21

SENS_BY_SCENARIO = (
    SENS.groupby(
        ["scenario", "min_cluster_rule", "policy"],
        as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "nunique"),
        mean_true_ARI=("true_ARI", "mean"),
        mean_delta_vs_quality=(
            "delta_vs_quality", "mean"
        ),
        K_accuracy=("correct_K", "mean")
    )
)

assert SENS_BY_SCENARIO.shape[0] == (
    6 * 3 * 7
)


# ------------------------------------------------------------
# 8. CONFLICT-VIEW MECHANISM DIAGNOSTIC
# ------------------------------------------------------------

CONFLICT = W.loc[
    W["scenario"] == "conflict_K3"
].copy()

assert len(CONFLICT) == 60

CONFLICT_SUMMARY = pd.DataFrame([{
    "n_datasets": len(CONFLICT),
    "target_ARI_quality": float(
        CONFLICT["true_ARI_quality_only"].mean()
    ),
    "target_ARI_SA": float(
        CONFLICT["true_ARI_SA_delta_0.05"].mean()
    ),
    "nuisance_ARI_quality": float(
        CONFLICT[
            "climate_nuisance_ARI_quality_only"
        ].mean()
    ),
    "nuisance_ARI_SA": float(
        CONFLICT[
            "climate_nuisance_ARI_SA_delta_0.05"
        ].mean()
    ),
    "delta_target_ARI": float(
        CONFLICT["delta_true_ARI"].mean()
    ),
    "delta_nuisance_ARI": float(
        CONFLICT[
            "delta_climate_nuisance_ARI"
        ].mean()
    ),
    "SA_more_aligned_with_nuisance_count": int(
        (
            CONFLICT["delta_climate_nuisance_ARI"]
            > 1e-12
        ).sum()
    )
}])


# ------------------------------------------------------------
# 9. NULL SCENARIO — NO ARTIFICIAL GROUND-TRUTH ARI
# ------------------------------------------------------------

NULL_PRIMARY = NULL.loc[
    (NULL["min_cluster_rule"] == 5)
    & NULL["policy"].isin([
        "quality_only", "SA_delta_0.05"
    ])
].copy()

assert len(NULL_PRIMARY) == 120

NULL_SUMMARY = (
    NULL_PRIMARY.groupby(
        "policy", as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "nunique"),
        mean_selected_K=("selected_K", "mean"),
        mean_silhouette=(
            "selected_silhouette", "mean"
        ),
        mean_subsampling_ARI=(
            "selected_stability", "mean"
        ),
        mean_minimum_cluster=(
            "selected_min_cluster", "mean"
        )
    )
)

NULL_METHODS = (
    NULL_PRIMARY.groupby(
        ["policy", "selected_method"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "n_selections"})
)

NULL_K = (
    NULL_PRIMARY.groupby(
        ["policy", "selected_K"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "n_selections"})
)

assert NULL_PRIMARY["true_ARI"].isna().all()


# ------------------------------------------------------------
# 10. SELECTED MODEL FAMILIES BY SCENARIO
# ------------------------------------------------------------

PRIMARY_METHODS = (
    P.groupby(
        ["scenario", "policy", "selected_method"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "n_selections"})
)

assert PRIMARY_METHODS.groupby(
    ["scenario", "policy"]
)["n_selections"].sum().eq(60).all()


# ------------------------------------------------------------
# 11. SAVE IMMUTABLE RESULTS
# ------------------------------------------------------------

output_frames = {
    "all_ground_truth_policy_evaluations.csv":
        RESULT,
    "primary_paired_dataset_results.csv":
        W,
    "primary_overall_summary.csv":
        OVERALL,
    "primary_scenario_summary.csv":
        SCENARIO_SUMMARY,
    "sensitivity_overall.csv":
        SENS_SUMMARY,
    "sensitivity_by_scenario.csv":
        SENS_BY_SCENARIO,
    "primary_method_by_scenario.csv":
        PRIMARY_METHODS,
    "conflict_view_diagnostic.csv":
        CONFLICT_SUMMARY,
    "null_primary_summary.csv":
        NULL_SUMMARY,
    "null_method_distribution.csv":
        NULL_METHODS,
    "null_K_distribution.csv":
        NULL_K
}

for filename, frame in output_frames.items():
    save_once_csv(
        frame,
        OUT / filename
    )

EVALUATION_SEAL = {
    "analysis_spec_sha256": sha256(
        OUT / "analysis_spec.json"
    ),
    "selection_seal_sha256": sha256(seal_path),
    "results_sha256": {
        name: sha256(OUT / name)
        for name in output_frames
    },
    "n_datasets": 420,
    "n_nonnull": 360,
    "n_null": 60,
    "n_policy_evaluations": 8820,
    "n_primary_pairs": 360,
    "model_fits_performed": 0,
    "selection_changed_after_truth": False,
    "status": "EVALUATION_COMPLETED"
}

save_once_json(
    EVALUATION_SEAL,
    OUT / "evaluation_seal.json"
)


# ------------------------------------------------------------
# 12. SCIENTIFIC REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INDEPENDENT GROUND-TRUTH EVALUATION COMPLETED")
print("=" * 70)

print("\nA. PRIMARY OVERALL COMPARISON")
display(OVERALL.round(4))

print("\nB. SCENARIO-SPECIFIC PAIRED COMPARISON")
display(
    SCENARIO_SUMMARY[
        [
            "scenario",
            "mean_ARI_quality",
            "mean_ARI_SA",
            "mean_delta_ARI",
            "CI95_delta_ARI_low",
            "CI95_delta_ARI_high",
            "SA_better",
            "SA_worse",
            "equal_ARI"
        ]
    ].round(4)
)

print("\nC. NUMBER-OF-CLUSTERS RECOVERY")
display(
    SCENARIO_SUMMARY[
        [
            "scenario",
            "K_accuracy_quality",
            "K_accuracy_SA",
            "delta_K_accuracy",
            "changed_K_fraction"
        ]
    ].round(4)
)

print("\nD. FROZEN POLICY SENSITIVITY")
display(
    SENS_SUMMARY.loc[
        SENS_SUMMARY["min_cluster_rule"] == 5
    ].round(4)
)

print("\nE. CLIMATE CONFLICT DIAGNOSTIC")
display(CONFLICT_SUMMARY.round(4))

print("\nF. GAUSSIAN NULL — DESCRIPTIVE ONLY")
display(NULL_SUMMARY.round(4))

print("\nG. NULL METHOD DISTRIBUTION")
display(NULL_METHODS)

print("\nH. PRIMARY MODEL FAMILY BY SCENARIO")
display(PRIMARY_METHODS)

print("\n" + "=" * 70)
print("FINAL INTEGRITY")
print("=" * 70)
print("Frozen selections     : VERIFIED")
print("Non-null datasets     : 360")
print("Null datasets         : 60")
print("Primary paired units  : 360")
print("Bootstrap repetitions :", BOOT_B)
print("Additional model fits : 0")
print("Analysis seal         : VERIFIED")
print("Output:", OUT)
print("=" * 70)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
A. BLIND SELECTION SEAL: VERIFIED
B. GROUND TRUTH VALIDATED
Non-null datasets: 360
Gaussian null datasets: 60
Policy evaluations: 8820

INDEPENDENT GROUND-TRUTH EVALUATION COMPLETED

A. PRIMARY OVERALL COMPARISON


,n_nonnull_datasets,n_scenarios,mean_ARI_quality,mean_ARI_SA,mean_delta_ARI,CI95_delta_ARI_low,CI95_delta_ARI_high,K_accuracy_quality,K_accuracy_SA,delta_K_accuracy,CI95_delta_K_low,CI95_delta_K_high,fraction_changed_model,fraction_changed_K,bootstrap_B
0,360,6,0.4428,0.4151,-0.0277,-0.0447,-0.0105,0.4444,0.5083,0.0639,0.0222,0.1056,0.8028,0.2472,10000



B. SCENARIO-SPECIFIC PAIRED COMPARISON


,scenario,mean_ARI_quality,mean_ARI_SA,mean_delta_ARI,CI95_delta_ARI_low,CI95_delta_ARI_high,SA_better,SA_worse,equal_ARI
0,aligned_K2,0.7439,0.6979,-0.0459,-0.0760,-0.0187,10,25,25
1,aligned_K3,0.6176,0.5635,-0.0541,-0.0996,-0.0040,12,33,15
2,aligned_K4,0.5750,0.5213,-0.0537,-0.1089,0.0016,16,32,12
3,conflict_K3,0.2146,0.2384,0.0239,-0.0204,0.0669,31,23,6
4,imbalanced_K3,0.4921,0.4495,-0.0426,-0.0879,0.0057,17,38,5
5,weak_K3,0.0137,0.0200,0.0063,-0.0014,0.0139,28,19,13



C. NUMBER-OF-CLUSTERS RECOVERY


...


G. NULL METHOD DISTRIBUTION


,policy,selected_method,n_selections
0,SA_delta_0.05,Hierarchical_Average,4
1,SA_delta_0.05,Hierarchical_Complete,1
2,SA_delta_0.05,Hierarchical_Ward,1
3,SA_delta_0.05,Spectral_Early,21
4,SA_delta_0.05,Spectral_MultiView,33
5,quality_only,Hierarchical_Average,8
6,quality_only,Hierarchical_Complete,7
7,quality_only,Hierarchical_Ward,5
8,quality_only,KMedoids_PAM,1
9,quality_only,Spectral_Early,36



H. PRIMARY MODEL FAMILY BY SCENARIO


,scenario,policy,selected_method,n_selections
0,aligned_K2,SA_delta_0.05,Spectral_Early,32
1,aligned_K2,SA_delta_0.05,Spectral_MultiView,28
2,aligned_K2,quality_only,KMedoids_PAM,1
3,aligned_K2,quality_only,Spectral_Early,57
4,aligned_K2,quality_only,Spectral_MultiView,2
5,aligned_K3,SA_delta_0.05,Hierarchical_Complete,1
6,aligned_K3,SA_delta_0.05,KMedoids_PAM,1
7,aligned_K3,SA_delta_0.05,Spectral_Early,26
8,aligned_K3,SA_delta_0.05,Spectral_MultiView,32
9,aligned_K3,quality_only,Hierarchical_Average,3



FINAL INTEGRITY
Frozen selections     : VERIFIED
Non-null datasets     : 360
Null datasets         : 60
Primary paired units  : 360
Bootstrap repetitions : 10000
Additional model fits : 0
Analysis seal         : VERIFIED
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/09_independent_monte_carlo/simplex_v2_frozen_v1/ground_truth_evaluation_v1


# Audit mekanisme ketepatan K dan perubahan keluarga model

In [31]:

# ============================================================
# CELL 24 — POST-EVALUATION MECHANISM AUDIT
#
# READ-ONLY DIAGNOSTIC ANALYSIS
#
# No clustering.
# No model re-selection.
# No new output files.
# No confirmatory hypothesis changes.
# ============================================================

from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import json
import hashlib

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

MC = (
    ROOT / "03_outputs"
    / "09_independent_monte_carlo"
    / "simplex_v2_frozen_v1"
)

EVAL = MC / "ground_truth_evaluation_v1"

def file_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

seal = json.loads(
    (EVAL / "evaluation_seal.json").read_text()
)

assert seal["status"] == "EVALUATION_COMPLETED"

source = (
    EVAL / "primary_paired_dataset_results.csv"
)

assert file_hash(source) == seal[
    "results_sha256"
]["primary_paired_dataset_results.csv"]

D = pd.read_csv(source)

assert len(D) == 360
assert D["dataset_id"].is_unique
assert D["scenario"].nunique() == 6
assert D.groupby("scenario").size().eq(60).all()

Q = "quality_only"
SA = "SA_delta_0.05"

for policy in [Q, SA]:

    for field in [
        "true_ARI",
        "correct_K",
        "selected_K"
    ]:

        column = f"{field}_{policy}"

        D[column] = pd.to_numeric(
            D[column],
            errors="raise"
        )

D["delta_ARI"] = (
    D[f"true_ARI_{SA}"]
    - D[f"true_ARI_{Q}"]
)

D["Q_correct"] = (
    D[f"correct_K_{Q}"] == 1
)

D["SA_correct"] = (
    D[f"correct_K_{SA}"] == 1
)

D["model_changed"] = (
    D[f"selected_model_{SA}"]
    != D[f"selected_model_{Q}"]
)

D["K_changed"] = (
    D[f"selected_K_{SA}"]
    != D[f"selected_K_{Q}"]
)

# ------------------------------------------------------------
# A. DIRECTION OF ARI CHANGES
# ------------------------------------------------------------

eps = 1e-12

D["ARI_direction"] = np.select(
    [
        D["delta_ARI"] > eps,
        D["delta_ARI"] < -eps
    ],
    [
        "SA_better",
        "SA_worse"
    ],
    default="equal"
)

direction = pd.crosstab(
    D["scenario"],
    D["ARI_direction"]
)

for col in ["SA_better", "SA_worse", "equal"]:
    if col not in direction.columns:
        direction[col] = 0

direction = direction[
    ["SA_better", "SA_worse", "equal"]
]

direction.loc["TOTAL"] = direction.sum()

print("A. ARI IMPROVEMENT / DETERIORATION COUNTS")
display(direction)

assert direction.loc["TOTAL"].sum() == 360


# ------------------------------------------------------------
# B. CORRECT-K TRANSITIONS
# ------------------------------------------------------------

D["K_recovery_status"] = np.select(
    [
        D["Q_correct"] & D["SA_correct"],
        D["Q_correct"] & ~D["SA_correct"],
        ~D["Q_correct"] & D["SA_correct"],
        ~D["Q_correct"] & ~D["SA_correct"]
    ],
    [
        "both_correct",
        "quality_only_correct",
        "SA_only_correct",
        "neither_correct"
    ],
    default="invalid"
)

assert not D[
    "K_recovery_status"
].eq("invalid").any()

K_mechanism = (
    D.groupby(
        ["scenario", "K_recovery_status"],
        as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "count"),
        mean_delta_ARI=("delta_ARI", "mean"),
        median_delta_ARI=("delta_ARI", "median")
    )
)

print("\nB. CORRECT-K TRANSITION MECHANISM")
display(K_mechanism.round(4))


# ------------------------------------------------------------
# C. IS LOWER ARI PRESENT EVEN WHEN BOTH GET K RIGHT?
# ------------------------------------------------------------

both_correct = D.loc[
    D["Q_correct"] & D["SA_correct"]
]

print("\nC. WHEN BOTH POLICIES SELECT THE CORRECT K")

both_correct_summary = (
    both_correct.groupby(
        "scenario", as_index=False
    )
    .agg(
        n_datasets=("dataset_id", "count"),
        mean_delta_ARI=("delta_ARI", "mean"),
        fraction_SA_better=(
            "delta_ARI",
            lambda x: float(np.mean(x > eps))
        ),
        fraction_SA_worse=(
            "delta_ARI",
            lambda x: float(np.mean(x < -eps))
        )
    )
)

display(both_correct_summary.round(4))

print(
    "Pooled mean difference when both K correct:",
    round(float(both_correct["delta_ARI"].mean()), 4)
)

# This is conditional descriptive analysis,
# not a new confirmatory causal estimand.


# ------------------------------------------------------------
# D. MODEL FAMILY TRANSITIONS
# ------------------------------------------------------------

family_switch = pd.crosstab(
    D[f"selected_method_{Q}"],
    D[f"selected_method_{SA}"]
)

print("\nD. QUALITY-ONLY → STABILITY-AWARE METHOD SWITCH")
display(family_switch)


# ------------------------------------------------------------
# E. SELECTED K BY SCENARIO
# ------------------------------------------------------------

k_rows = []

for policy in [Q, SA]:

    counts = (
        D.groupby(
            ["scenario", f"selected_K_{policy}"]
        )
        .size()
        .reset_index(name="n_datasets")
    )

    counts.columns = [
        "scenario", "selected_K", "n_datasets"
    ]

    counts["policy"] = policy

    k_rows.append(counts)

K_DISTRIBUTION = pd.concat(
    k_rows,
    ignore_index=True
)

print("\nE. K DISTRIBUTION BY SCENARIO")
display(
    K_DISTRIBUTION.sort_values(
        ["scenario", "policy", "selected_K"]
    )
)


# ------------------------------------------------------------
# F. COMPACT SCIENTIFIC DIAGNOSTIC
# ------------------------------------------------------------

print("\nF. OVERALL SUMMARY")

print(
    "Mean paired ΔARI:",
    round(float(D["delta_ARI"].mean()), 6)
)

print(
    "K accuracy Quality:",
    round(float(D["Q_correct"].mean()), 6)
)

print(
    "K accuracy Stability-aware:",
    round(float(D["SA_correct"].mean()), 6)
)

print(
    "Model changed:",
    int(D["model_changed"].sum()), "/360"
)

print(
    "K changed:",
    int(D["K_changed"].sum()), "/360"
)

print(
    "Both policies chose true K:",
    len(both_correct), "/360"
)

print("\nCELL 24 COMPLETED")
print("Read-only post-evaluation diagnostic.")
print("Frozen results remain unchanged.")


A. ARI IMPROVEMENT / DETERIORATION COUNTS


ARI_direction,SA_better,SA_worse,equal
scenario,,,
aligned_K2,10,25,25
aligned_K3,12,33,15
aligned_K4,16,32,12
conflict_K3,31,23,6
imbalanced_K3,17,38,5
weak_K3,28,19,13
TOTAL,114,170,76



B. CORRECT-K TRANSITION MECHANISM


,scenario,K_recovery_status,n_datasets,mean_delta_ARI,median_delta_ARI
0,aligned_K2,both_correct,58,-0.0441,0.0000
1,aligned_K2,neither_correct,1,0.0000,0.0000
2,aligned_K2,quality_only_correct,1,-0.1959,-0.1959
3,aligned_K3,SA_only_correct,6,0.2630,0.1331
4,aligned_K3,both_correct,38,-0.0665,-0.0439
5,aligned_K3,neither_correct,8,-0.0213,0.0000
6,aligned_K3,quality_only_correct,8,-0.2655,-0.2763
7,aligned_K4,SA_only_correct,8,0.1443,0.1687
8,aligned_K4,both_correct,35,-0.0776,-0.0128
9,aligned_K4,neither_correct,13,-0.0427,-0.0078



C. WHEN BOTH POLICIES SELECT THE CORRECT K


...

Pooled mean difference when both K correct: -0.0595

D. QUALITY-ONLY → STABILITY-AWARE METHOD SWITCH


selected_method_SA_delta_0.05,Hierarchical_Average,Hierarchical_Complete,KMedoids_PAM,Spectral_Early,Spectral_MultiView
selected_method_quality_only,,,,,
Hierarchical_Average,6,0,0,9,8
Hierarchical_Complete,0,4,0,11,13
Hierarchical_Ward,0,1,1,4,9
KMedoids_PAM,0,0,1,3,4
Spectral_Early,0,0,7,102,163
Spectral_MultiView,0,0,1,4,9



E. K DISTRIBUTION BY SCENARIO


,scenario,selected_K,n_datasets,policy
24,aligned_K2,2,58,SA_delta_0.05
25,aligned_K2,3,2,SA_delta_0.05
0,aligned_K2,2,59,quality_only
1,aligned_K2,3,1,quality_only
26,aligned_K3,2,15,SA_delta_0.05
27,aligned_K3,3,44,SA_delta_0.05
28,aligned_K3,4,1,SA_delta_0.05
2,aligned_K3,2,8,quality_only
3,aligned_K3,3,46,quality_only
4,aligned_K3,4,5,quality_only



F. OVERALL SUMMARY
Mean paired ΔARI: -0.027687
K accuracy Quality: 0.444444
K accuracy Stability-aware: 0.508333
Model changed: 289 /360
K changed: 89 /360
Both policies chose true K: 138 /360

CELL 24 COMPLETED
Read-only post-evaluation diagnostic.
Frozen results remain unchanged.


# Bagian Baru

# Implikasi metodologis yang perlu kita pertahankan
Stabilitas subsampling dapat meningkatkan ketepatan jumlah cluster tanpa meningkatkan ketepatan keanggotaan cluster. Pada simulasi kita, kondisi ini bahkan muncul ketika kedua kebijakan memilih K yang benar.

Temuan tersebut mempunyai tiga konsekuensi untuk artikel.

Pertama, ARI subsampling tidak boleh digunakan sebagai pengganti ukuran validitas partisi. Kedua, stabilitas sebaiknya diposisikan sebagai dimensi diagnostik tambahan, bukan jaminan bahwa struktur yang dipilih benar. Ketiga, pergeseran dari Early Fusion menuju Multi-View perlu dilaporkan sebagai perilaku kebijakan seleksi, bukan bukti keunggulan atau kelemahan intrinsik salah satu algoritma.

Ada satu batasan penting: nilai ARI −0,0595 pada kelompok both correct merupakan hasil analisis kondisional. Kelompok tersebut terbentuk berdasarkan keputusan kedua kebijakan. Karena itu, hasilnya menjelaskan pola perbedaan, tetapi tidak membuktikan bahwa perubahan algoritma menyebabkan penurunan ARI.

## Identifikasi hasil empiris yang sudah tersedia

In [32]:

# ============================================================
# CELL 25 — EXISTING EMPIRICAL RESULTS INVENTORY
# Read-only; no fitting, no new files.
# ============================================================

from pathlib import Path
from IPython.display import display
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

BASE = ROOT / "03_outputs"
assert BASE.exists()

print("A. OUTPUT DIRECTORIES")

for folder in sorted(BASE.iterdir()):
    if folder.is_dir():
        print(folder.name)

# Exclude completed Monte Carlo simulation experiments.
EXCLUDED = {
    "08_monte_carlo_simulation",
    "09_independent_monte_carlo"
}

records = []

for folder in sorted(BASE.iterdir()):

    if not folder.is_dir():
        continue

    if folder.name in EXCLUDED:
        continue

    for path in folder.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".csv", ".parquet", ".json", ".npz"
        }:
            continue

        records.append({
            "folder": folder.name,
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_MB": round(
                path.stat().st_size / 1048576, 3
            ),
            "relative_path": str(
                path.relative_to(ROOT)
            )
        })

inventory = pd.DataFrame(records)

print("\nB. EXISTING NON-SIMULATION OUTPUT FILES")
print("Total:", len(inventory))

if not inventory.empty:

    summary = (
        inventory.groupby(
            ["folder", "extension"],
            as_index=False
        )
        .agg(
            n_files=("filename", "count"),
            size_MB=("size_MB", "sum")
        )
    )

    display(summary.round(3))

    keywords = (
        "cluster",
        "stability",
        "pareto",
        "candidate",
        "quality",
        "label",
        "empirical",
        "selection"
    )

    relevant = inventory.loc[
        inventory["relative_path"].str.lower()
        .str.contains(
            "|".join(keywords),
            regex=True
        )
    ].sort_values("relative_path")

    print("\nC. POTENTIALLY RELEVANT FILES")
    display(
        relevant.head(70)
    )

    print("\nD. CSV STRUCTURE PREVIEW")

    for path_str in relevant.loc[
        relevant["extension"] == ".csv",
        "relative_path"
    ].head(15):

        try:
            path = ROOT / path_str
            preview = pd.read_csv(path, nrows=3)

            print("\n", path_str)
            print("Columns:", preview.columns.tolist())

        except Exception as exc:
            print("Could not inspect:", path_str)
            print(type(exc).__name__, str(exc))

else:
    print("No matching output files found.")

print("\nEMPIRICAL INVENTORY COMPLETED")
print("No existing outputs were modified.")


A. OUTPUT DIRECTORIES
01_audit
02_representation
03_gridsearch_pilot
04_stability_pilot
05_extended_gridsearch_R0
06_stability_selection
07_initialization_sensitivity
08_monte_carlo_simulation
09_independent_monte_carlo

B. EXISTING NON-SIMULATION OUTPUT FILES
Total: 204


,folder,extension,n_files,size_MB
0,01_audit,.csv,10,0.015
1,02_representation,.csv,6,0.081
2,02_representation,.json,2,0.006
3,02_representation,.npz,4,0.632
4,03_gridsearch_pilot,.csv,18,0.178
5,03_gridsearch_pilot,.json,2,0.002
6,04_stability_pilot,.csv,13,0.185
7,04_stability_pilot,.json,2,0.002
8,05_extended_gridsearch_R0,.csv,11,0.302
9,05_extended_gridsearch_R0,.json,1,0.001



C. POTENTIALLY RELEVANT FILES


,folder,filename,extension,size_MB,relative_path
28,03_gridsearch_pilot,candidate_ARI_matrix.csv,.csv,0.001,03_outputs/03_gridsearch_pilot/20261018_175911...
26,03_gridsearch_pilot,candidate_cluster_memberships.csv,.csv,0.051,03_outputs/03_gridsearch_pilot/20261018_175911...
25,03_gridsearch_pilot,candidate_cluster_profiles.csv,.csv,0.005,03_outputs/03_gridsearch_pilot/20261018_175911...
29,03_gridsearch_pilot,candidate_pairwise_ARI.csv,.csv,0.002,03_outputs/03_gridsearch_pilot/20261018_175911...
31,03_gridsearch_pilot,candidate_province_composition.csv,.csv,0.005,03_outputs/03_gridsearch_pilot/20261018_175911...
...,...,...,...,...,...
112,06_stability_selection,scores_035_040.csv,.csv,0.166,03_outputs/06_stability_selection/R0_all395_se...
114,06_stability_selection,scores_040_045.csv,.csv,0.167,03_outputs/06_stability_selection/R0_all395_se...
116,06_stability_selection,scores_045_050.csv,.csv,0.166,03_outputs/06_stability_selection/R0_all395_se...
118,06_stability_selection,scores_050_055.csv,.csv,0.166,03_outputs/06_stability_selection/R0_all395_se...



D. CSV STRUCTURE PREVIEW

 03_outputs/03_gridsearch_pilot/20261018_175911_UTC/candidate_ARI_matrix.csv
Columns: ['Unnamed: 0', 'Hierarchical_Average_K2', 'Hierarchical_Complete_K2', 'Spectral_MultiView_K2_KG24', 'Spectral_Early_K3_KG16', 'KMedoids_PAM_K3', 'Hierarchical_Ward_K2', 'Hierarchical_Average_K3', 'Spectral_MultiView_K3_KG24']

 03_outputs/03_gridsearch_pilot/20261018_175911_UTC/candidate_cluster_memberships.csv
Columns: ['model_id', 'cluster', 'cluster_size', 'kode_bps', 'provinsi', 'kabupaten_kota', 'silhouette_reference']

 03_outputs/03_gridsearch_pilot/20261018_175911_UTC/candidate_cluster_profiles.csv
Columns: ['model_id', 'method', 'K', 'cluster', 'n', 'mean_silhouette', 'n_negative_silhouette', 'zmean_F1_ikp_ketersediaan', 'zmean_F2_ikp_keterjangkauan', 'zmean_F3_ikp_pemanfaatan', 'zmean_C1_anomali_hujan_mm', 'zmean_C2_cdd_hari', 'zmean_C3_hari_hujan_ekstrem', 'zmean_A1_persen_tanpa_angkutan', 'zmean_A2_persen_jalan_tanah_lainnya', 'zmean_A3_persen_tanpa_pertokoan_pas

## Audit sumber hasil empiris R0

In [33]:

# ============================================================
# CELL 26 — EMPIRICAL R0 SOURCE AUDIT
#
# Read-only investigation of:
# 05_extended_gridsearch_R0
# 06_stability_selection
# 07_initialization_sensitivity
#
# No model fitting.
# No file creation.
# No existing result modification.
# ============================================================

from pathlib import Path
from IPython.display import display
import pandas as pd
import numpy as np
import json
import re

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

BASE = ROOT / "03_outputs"

FOLDERS = {
    "R0_GRID": BASE / "05_extended_gridsearch_R0",
    "STABILITY": BASE / "06_stability_selection",
    "SEED_AUDIT": BASE / "07_initialization_sensitivity"
}

for key, folder in FOLDERS.items():
    assert folder.is_dir(), f"Missing {key}: {folder}"


# ============================================================
# A. FIND ACTUAL CSV AND JSON FILES
# ============================================================

records = []

for stage, folder in FOLDERS.items():

    for path in sorted(folder.rglob("*")):

        if not path.is_file():
            continue

        if path.suffix.lower() not in {".csv", ".json"}:
            continue

        is_score_chunk = bool(
            re.fullmatch(
                r"scores_\d+_\d+",
                path.stem
            )
        )

        records.append({
            "stage": stage,
            "filename": path.name,
            "extension": path.suffix.lower(),
            "is_score_chunk": is_score_chunk,
            "size_MB": (
                path.stat().st_size / (1024 ** 2)
            ),
            "relative_path": str(
                path.relative_to(ROOT)
            ),
            "absolute_path": str(path)
        })

FILES = pd.DataFrame(records)

assert not FILES.empty

print("=" * 70)
print("A. SOURCE FILE COVERAGE")
print("=" * 70)

display(
    FILES.groupby(
        ["stage", "extension"],
        as_index=False
    ).agg(
        n_files=("filename", "count"),
        total_MB=("size_MB", "sum")
    ).round(4)
)

print("\nSubsampling score chunks:",
      int(FILES["is_score_chunk"].sum()))


# ============================================================
# B. LIST ALL R0 FULL-DATA RESULT FILES
# ============================================================

print("\n" + "=" * 70)
print("B. R0 FULL-DATA GRIDSEARCH FILES")
print("=" * 70)

R0_FILES = FILES.loc[
    FILES["stage"] == "R0_GRID"
].sort_values("relative_path")

for r in R0_FILES.itertuples(index=False):
    print(f"{r.filename} | {r.relative_path}")


# ============================================================
# C. LIST NON-CHUNK STABILITY OUTPUTS
# ============================================================

print("\n" + "=" * 70)
print("C. STABILITY SUMMARY / PROTOCOL FILES")
print("=" * 70)

STABILITY_MAIN = FILES.loc[
    (FILES["stage"] == "STABILITY")
    & (~FILES["is_score_chunk"])
].sort_values("relative_path")

for r in STABILITY_MAIN.itertuples(index=False):
    print(f"{r.filename} | {r.relative_path}")


# ============================================================
# D. INSPECT FILE SCHEMAS
# ============================================================

def inspect_csv(path):

    sample = pd.read_csv(
        path,
        nrows=5
    )

    columns = sample.columns.tolist()

    # Most non-chunk files are small.
    # Read them once to determine row count and model IDs.
    df = pd.read_csv(path)

    output = {
        "rows": len(df),
        "columns": len(columns),
        "column_names": columns,
        "model_ids": None,
        "duplicate_model_ids": None,
        "unique_districts": None
    }

    if "model_id" in df.columns:

        output["model_ids"] = (
            df["model_id"]
            .dropna()
            .astype(str)
            .nunique()
        )

        output["duplicate_model_ids"] = int(
            df["model_id"].duplicated().sum()
        )

    if "kode_bps" in df.columns:

        output["unique_districts"] = (
            df["kode_bps"].nunique()
        )

    return output


SCHEMA_RECORDS = []
MODEL_ID_SETS = {}

inspect_files = FILES.loc[
    (FILES["extension"] == ".csv")
    & (
        (FILES["stage"] == "R0_GRID")
        | (
            (FILES["stage"] == "STABILITY")
            & (~FILES["is_score_chunk"])
        )
    )
].sort_values(["stage", "filename"])

print("\n" + "=" * 70)
print("D. STRUCTURE OF R0 AND STABILITY SUMMARY TABLES")
print("=" * 70)

for r in inspect_files.itertuples(index=False):

    path = Path(r.absolute_path)

    try:

        info = inspect_csv(path)

        SCHEMA_RECORDS.append({
            "stage": r.stage,
            "filename": r.filename,
            "rows": info["rows"],
            "columns": info["columns"],
            "unique_model_ids": info["model_ids"],
            "duplicate_model_ids": (
                info["duplicate_model_ids"]
            ),
            "unique_districts": (
                info["unique_districts"]
            ),
            "path": r.relative_path
        })

        print("\nFILE:", r.relative_path)
        print("Shape:", (
            info["rows"], info["columns"]
        ))

        print(
            "Columns (first 20):",
            info["column_names"][:20]
        )

        if len(info["column_names"]) > 20:
            print(
                "Columns (last 5):",
                info["column_names"][-5:]
            )

        if info["model_ids"] is not None:
            print(
                "Unique model IDs:",
                info["model_ids"]
            )

        if info["unique_districts"] is not None:
            print(
                "Unique districts:",
                info["unique_districts"]
            )

        if (
            info["rows"] > 0
            and info["columns"] <= 35
        ):

            display(
                pd.read_csv(
                    path,
                    nrows=2
                )
            )

        # Store ID sets in memory for cross-source audit.
        if "model_id" in info["column_names"]:

            ids = set(
                pd.read_csv(
                    path,
                    usecols=["model_id"]
                )["model_id"]
                .dropna()
                .astype(str)
            )

            MODEL_ID_SETS[
                r.relative_path
            ] = ids

    except Exception as exc:

        print(
            "SCHEMA INSPECTION FAILED:",
            r.relative_path
        )

        print(
            type(exc).__name__,
            str(exc)
        )

if SCHEMA_RECORDS:

    SCHEMAS = pd.DataFrame(
        SCHEMA_RECORDS
    )

    print("\nE. CONSOLIDATED SOURCE SCHEMAS")

    display(
        SCHEMAS[
            [
                "stage",
                "filename",
                "rows",
                "columns",
                "unique_model_ids",
                "duplicate_model_ids",
                "unique_districts"
            ]
        ]
    )


# ============================================================
# F. VERIFY 395-CANDIDATE SOURCES
# ============================================================

print("\n" + "=" * 70)
print("F. CANDIDATE IDENTITY AUDIT")
print("=" * 70)

CANDIDATE_SETS = {
    path: ids
    for path, ids in MODEL_ID_SETS.items()
    if len(ids) == 395
}

if CANDIDATE_SETS:

    print(
        "Tables containing 395 unique model IDs:",
        len(CANDIDATE_SETS)
    )

    for path in CANDIDATE_SETS:
        print(" -", path)

    paths = list(CANDIDATE_SETS)

    reference_ids = CANDIDATE_SETS[
        paths[0]
    ]

    for path in paths[1:]:

        other_ids = CANDIDATE_SETS[path]

        print("\nComparison:", path)

        print(
            "Matching 395-ID set:",
            reference_ids == other_ids
        )

        print(
            "Missing from comparison:",
            len(reference_ids - other_ids)
        )

        print(
            "Additional IDs:",
            len(other_ids - reference_ids)
        )

else:
    print(
        "No non-chunk table with exactly "
        "395 unique model IDs was identified."
    )

    print(
        "Check full-data label columns or "
        "chunk-level stability schema before proceeding."
    )


# ============================================================
# G. INSPECT ONE STORED SUBSAMPLING CHUNK
# ============================================================

CHUNKS = FILES.loc[
    (FILES["stage"] == "STABILITY")
    & FILES["is_score_chunk"]
].sort_values("filename")

print("\n" + "=" * 70)
print("G. STORED SUBSAMPLING SCORE FORMAT")
print("=" * 70)

print("Number of score chunks:", len(CHUNKS))

if len(CHUNKS) > 0:

    sample_chunk_path = Path(
        CHUNKS.iloc[0]["absolute_path"]
    )

    chunk = pd.read_csv(
        sample_chunk_path,
        nrows=10
    )

    print("Sample chunk:")
    print(
        CHUNKS.iloc[0]["relative_path"]
    )

    print("Columns:")
    print(chunk.columns.tolist())

    display(chunk.head(5))

else:

    print(
        "No scores_<start>_<end>.csv "
        "chunks detected."
    )


# ============================================================
# H. AUDIT JSON PROTOCOLS
# ============================================================

print("\n" + "=" * 70)
print("H. EXPERIMENT PROTOCOL METADATA")
print("=" * 70)

json_files = FILES.loc[
    FILES["extension"] == ".json"
].sort_values(
    ["stage", "filename"]
)

for r in json_files.itertuples(index=False):

    print("\nJSON:", r.relative_path)

    try:

        metadata = json.loads(
            Path(r.absolute_path).read_text(
                encoding="utf-8"
            )
        )

        if isinstance(metadata, dict):

            print(
                "Top-level keys:",
                list(metadata.keys())
            )

            for key, value in metadata.items():

                # Avoid printing huge nested arrays.
                if isinstance(
                    value,
                    (str, int, float, bool)
                ):

                    shown = str(value)

                    if len(shown) > 180:
                        shown = shown[:180] + "..."

                    print(
                        f"  {key}: {shown}"
                    )

        else:
            print(
                "JSON root type:",
                type(metadata).__name__
            )

    except Exception as exc:

        print(
            "Metadata inspection failed:",
            type(exc).__name__,
            str(exc)
        )


print("\n" + "=" * 70)
print("EMPIRICAL R0 SOURCE AUDIT COMPLETED")
print("=" * 70)

print("Read-only audit.")
print("No model fitting performed.")
print("No output files created.")
print("No empirical model selected yet.")


A. SOURCE FILE COVERAGE


,stage,extension,n_files,total_MB
0,R0_GRID,.csv,11,0.3018
1,R0_GRID,.json,1,0.0011
2,SEED_AUDIT,.csv,26,0.9516
3,SEED_AUDIT,.json,1,0.0009
4,STABILITY,.csv,106,30.0586
5,STABILITY,.json,2,0.0016



Subsampling score chunks: 40

B. R0 FULL-DATA GRIDSEARCH FILES
failed_configurations.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/failed_configurations.csv
failures_checkpoint.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/failures_checkpoint.csv
grid_plan.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/grid_plan.csv
labels_all.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/labels_all.csv
partition_group_summary.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/partition_group_summary.csv
partition_uniqueness_audit.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/partition_uniqueness_audit.csv
partition_uniqueness_by_method.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/partition_uniqueness_by_method.csv
pilot_reproducibility_check.csv | 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/pilot_reproducibility_check.csv
protocol.json | 03_outputs/05_extended_gridsearch_R0/2

,model_id,method,K,k_graph,k_scale,weight_id
0,KMedoids_PAM_K2,KMedoids_PAM,2,NaN,NaN,na
1,KMedoids_PAM_K3,KMedoids_PAM,3,NaN,NaN,na



FILE: 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/labels_all.csv
Shape: (85, 396)
Columns (first 20): ['kode_bps', 'KMedoids_PAM_K2', 'KMedoids_PAM_K3', 'KMedoids_PAM_K4', 'KMedoids_PAM_K5', 'KMedoids_PAM_K6', 'Hierarchical_Ward_K2', 'Hierarchical_Ward_K3', 'Hierarchical_Ward_K4', 'Hierarchical_Ward_K5', 'Hierarchical_Ward_K6', 'Hierarchical_Average_K2', 'Hierarchical_Average_K3', 'Hierarchical_Average_K4', 'Hierarchical_Average_K5', 'Hierarchical_Average_K6', 'Hierarchical_Complete_K2', 'Hierarchical_Complete_K3', 'Hierarchical_Complete_K4', 'Hierarchical_Complete_K5']
Columns (last 5): ['Multi_K2_G24_S10_access50', 'Multi_K3_G24_S10_access50', 'Multi_K4_G24_S10_access50', 'Multi_K5_G24_S10_access50', 'Multi_K6_G24_S10_access50']
Unique districts: 85

FILE: 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/partition_group_summary.csv
Shape: (303, 7)
Columns (first 20): ['partition_id', 'n_configurations', 'K', 'n_methods', 'silhouette_min', 'silhouette_max', 'silh

...


E. CONSOLIDATED SOURCE SCHEMAS


,stage,filename,rows,columns,unique_model_ids,duplicate_model_ids,unique_districts
0,R0_GRID,failed_configurations.csv,0,8,0.0,0.0,NaN
1,R0_GRID,grid_plan.csv,395,6,395.0,0.0,NaN
2,R0_GRID,labels_all.csv,85,396,NaN,NaN,85.0
3,R0_GRID,partition_group_summary.csv,303,7,NaN,NaN,NaN
4,R0_GRID,partition_uniqueness_audit.csv,395,8,395.0,0.0,NaN
...,...,...,...,...,...,...,...
71,STABILITY,selection_clusterwise_diagnostics.csv,23,6,9.0,14.0,NaN
72,STABILITY,stability_selection_all395.csv,395,28,395.0,0.0,NaN
73,STABILITY,stability_selection_all395.csv,395,28,395.0,0.0,NaN
74,STABILITY,stability_selection_repetitions.csv,39500,9,395.0,39105.0,NaN



F. CANDIDATE IDENTITY AUDIT
Tables containing 395 unique model IDs: 54
 - 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/grid_plan.csv
 - 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/partition_uniqueness_audit.csv
 - 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/quality_grid_all.csv
 - 03_outputs/06_stability_selection/R0_all395_selection_seed_aligned_v2/all395_pareto_flags.csv
 - 03_outputs/06_stability_selection/R0_all395_selection_v1/all395_pareto_flags.csv
 - 03_outputs/06_stability_selection/R0_all395_selection_seed_aligned_v2/batches/clusters_000_005.csv
 - 03_outputs/06_stability_selection/R0_all395_selection_v1/batches/clusters_000_005.csv
 - 03_outputs/06_stability_selection/R0_all395_selection_seed_aligned_v2/batches/clusters_005_010.csv
 - 03_outputs/06_stability_selection/R0_all395_selection_v1/batches/clusters_005_010.csv
 - 03_outputs/06_stability_selection/R0_all395_selection_seed_aligned_v2/batches/clusters_010_015.csv
 - 03_outputs/06_

,repeat,model_id,status,ARI,jaccard_macro,jaccard_min,missing_clusters,error_type,error_message
0,1,KMedoids_PAM_K2,SUCCESS,0.153260,0.544513,0.523810,0,NaN,NaN
1,1,KMedoids_PAM_K3,SUCCESS,0.897489,0.953803,0.925926,0,NaN,NaN
2,1,KMedoids_PAM_K4,SUCCESS,0.599289,0.633150,0.300000,0,NaN,NaN
3,1,KMedoids_PAM_K5,SUCCESS,0.616757,0.660102,0.312500,0,NaN,NaN
4,1,KMedoids_PAM_K6,SUCCESS,0.606207,0.671245,0.384615,0,NaN,NaN



H. EXPERIMENT PROTOCOL METADATA

JSON: 03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC/protocol.json
Top-level keys: ['stage', 'run_id', 'input_sha256', 'representation', 'reference_distance', 'K_grid', 'k_graph_grid', 'k_scale_grid', 'weight_grid', 'random_seed', 'pam_init', 'spectral_assign_labels', 'spectral_n_init', 'graph_type', 'affinity', 'fusion_normalization', 'status', 'versions']
  stage: Extended quality grid search R0
  run_id: 20261019_014117_UTC
  input_sha256: ce932d322eb49cbdb6ea3b0a8768ca89c2e9e7f0722232883541d891089ca6fa
  representation: R0_raw_zscore
  reference_distance: R0_fused_euclidean
  random_seed: 20261018
  pam_init: build
  spectral_assign_labels: kmeans
  spectral_n_init: 20
  graph_type: symmetric_union_kNN
  affinity: local_scaling_gaussian
  fusion_normalization: frobenius
  status: QUALITY_GRID_NOT_FINAL_SELECTION

JSON: 03_outputs/07_initialization_sensitivity/R0_pareto9_10seeds_20subsets_v1/manifest.json
Top-level keys: ['stage', 'source'